# Global Automotive Investment Database — Block 9

## AI-Assisted Quality Control

This notebook is the controlled AI enrichment and validation layer for the regional fundamentals produced by Blocks 3–8.

It performs six linked tasks:

1. loads standardised fundamentals, review queues, document text and entity-resolution exceptions from the regional blocks;
2. removes duplicate exceptions and groups equivalent records into representative issue signatures;
3. applies deterministic quality-control tests before any AI request;
4. sends only unresolved representative signatures to Azure OpenAI for enrichment;
5. applies deterministic acceptance gates, synonym learning and conservative cluster propagation;
6. publishes accepted, rejected and unresolved outcomes together with complete decision, provider and audit trails.

### Governing principles

- AI never overwrites a deterministic source value.
- Only unresolved signatures are sent to Azure OpenAI.
- Every AI proposal must satisfy deterministic publication gates.
- Low-confidence or contradictory proposals remain unresolved.
- Completed signatures and cached responses are reused so interrupted runs can resume efficiently.
- Block 10 consumes the quality-controlled outputs of this block.

Create Colab secrets named `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_DEPLOYMENT` and `AZURE_OPENAI_ENDPOINT` before running.

In [1]:
# ============================================================
# 1. IMPORTS
# ============================================================

# Python package installation
!pip -q install --upgrade pandas numpy pyarrow tqdm pydantic openai azure-identity google-genai

# Standard-library imports
import os
import re
import gc
import json
import time
import hashlib
import unicodedata
import warnings
import sqlite3

from abc import ABC, abstractmethod
from dataclasses import dataclass, asdict
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Optional, Literal, Type

# Third-party imports
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from pydantic import BaseModel, Field, ValidationError

warnings.filterwarnings("ignore", category=FutureWarning)

# Colab integration
try:
    from google.colab import drive, userdata
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    userdata = None

if IN_COLAB:
    drive.mount("/content/drive")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/

In [2]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

PROJECT_ROOT_CANDIDATES = [
    (
        Path(
            os.environ[
                "CFA_AUTOMOTIVE_PROJECT_ROOT"
            ]
        )
        if os.environ.get(
            "CFA_AUTOMOTIVE_PROJECT_ROOT"
        )
        else None
    ),
    Path(
        "/content/drive/MyDrive/Colab Notebooks/"
        "00 A1 Auto Factor Strategy"
    ),
    Path(
        "/content/drive/MyDrive/CFA Automotive Project/"
        "Global Automotive Investment Database"
    ),
]

PROJECT_ROOT_CANDIDATES = [
    path
    for path in PROJECT_ROOT_CANDIDATES
    if path is not None
]

PROJECT_ROOT = next(
    (
        path
        for path in PROJECT_ROOT_CANDIDATES
        if path.exists()
    ),
    PROJECT_ROOT_CANDIDATES[0],
)

DATA_ROOT = PROJECT_ROOT / "data"
INTERIM_ROOT = DATA_ROOT / "interim"
EXTERNAL_ROOT = DATA_ROOT / "external"

BLOCK_OUTPUT_DIRS = {
    block_number:
        INTERIM_ROOT / f"block_{block_number}"
    for block_number in range(3, 9)
}

BLOCK_9_OUTPUT_DIR = (
    INTERIM_ROOT / "block_9"
)

BLOCK_9_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

BLOCK_9_MANIFEST_PATH = (
    BLOCK_9_OUTPUT_DIR
    / "block_9_manifest.json"
)


# ------------------------------------------------------------
# AI cache locations
# ------------------------------------------------------------

# Google Drive holds only the persistent snapshot.
PERSISTENT_AI_CACHE_PATH = (
    BLOCK_9_OUTPUT_DIR
    / "block_9_ai_cache.sqlite"
)

# SQLite reads and writes occur on the local Colab runtime.
LOCAL_AI_CACHE_PATH = Path(
    "/content/block_9_ai_cache.sqlite"
)

AI_CACHE_PATH = LOCAL_AI_CACHE_PATH

COMPLETION_REGISTRY_PATH = (
    BLOCK_9_OUTPUT_DIR
    / "completed_ai_signatures.parquet"
)

BATCH_REGISTRY_PATH = (
    BLOCK_9_OUTPUT_DIR
    / "ai_batch_registry.parquet"
)

ACCEPTED_SYNONYM_REGISTRY_PATH = (
    BLOCK_9_OUTPUT_DIR
    / "accepted_synonym_registry.parquet"
)


# ------------------------------------------------------------
# Azure execution
# ------------------------------------------------------------

DETERMINISTIC_DRY_RUN = False
FINAL_PRODUCTION_MODE = True
AI_ENABLED = not DETERMINISTIC_DRY_RUN

ENABLE_CACHE = True
ENABLE_PROPAGATION = True
ENABLE_SIMILARITY_PROPAGATION = True
SKIP_COMPLETED_SIGNATURES = True

# This means fresh Azure requests per task, not selected rows
# containing old cached requests.
MAX_AI_RECORDS_PER_TASK = 1000

# Increment whenever prompts, schemas, candidate retrieval,
# clustering or acceptance logic materially changes.
AI_PROCESSING_GENERATION = (
    "block9_azure_final_production_v1"
)

# Acceptance-policy changes do not invalidate completed Azure calls.
ACCEPTANCE_POLICY_VERSION = "block9_acceptance_policy_v2"
PRE_AI_FRAGMENT_POLICY_VERSION = "block9_pre_ai_fragment_policy_v1"

AI_SLEEP_SECONDS = 0.5
MAX_RETRIES_PER_RECORD = 3
MAX_CONSECUTIVE_QUOTA_ERRORS = 3
RETRY_BASE_SECONDS = 2.0


# ------------------------------------------------------------
# Historical cache republication
# ------------------------------------------------------------

REPUBLISH_ALL_COMPLETED_CACHE_RESPONSES = True
REPUBLICATION_POLICY_VERSION = ACCEPTANCE_POLICY_VERSION

# Historical cache responses are reconstructed and re-evaluated locally.
# No Azure request is made by the republication stage.
HISTORICAL_CACHE_REPUBLICATION_SOURCE = "SQLITE_AI_CACHE"


# ------------------------------------------------------------
# Pre-Azure fragment exclusion
# ------------------------------------------------------------

ENABLE_PRE_AI_FRAGMENT_EXCLUSION = True
PRE_AI_FRAGMENT_MAX_WORDS = 12

PRE_AI_TERMINAL_FRAGMENT_WORDS = {
    "of", "in", "for", "from", "to", "with", "without",
    "during", "as", "at", "was", "were", "is", "are",
    "and", "or", "the", "a", "an",
}

PRE_AI_NARRATIVE_STARTERS = {
    "during the year",
    "the company",
    "the group",
    "management believes",
    "as discussed",
    "as disclosed",
    "the following",
}

PRE_AI_LAYOUT_LABELS = {
    "note", "notes", "continued", "continuation",
    "table of contents", "contents",
}


# ------------------------------------------------------------
# Candidate retrieval
# ------------------------------------------------------------

MAX_CANONICAL_CANDIDATES_PER_PROMPT = 15
MIN_CANDIDATE_SIMILARITY = 0.20


# ------------------------------------------------------------
# Similarity propagation
# ------------------------------------------------------------

DETERMINISTIC_CLUSTER_MIN_SIZE = 2
SEMANTIC_CLUSTER_MIN_SIZE = 3
SEMANTIC_PROPAGATION_THRESHOLD = 0.97

LARGE_CLUSTER_MEMBER_THRESHOLD = 100
LARGE_CLUSTER_REQUIRED_PROTOTYPES = 3
SMALL_CLUSTER_REQUIRED_PROTOTYPES = 1

CLUSTER_CONSENSUS_REQUIRED = 1.00
MAX_CLUSTER_PROTOTYPES_PER_CLUSTER = 3


# ------------------------------------------------------------
# Evidence and prompt controls
# ------------------------------------------------------------

MIN_EVIDENCE_CHARACTERS = 20
MAX_SOURCE_TEXT_CHARACTERS = 8000
MAX_PROMPT_CHARACTERS = 18000


# ------------------------------------------------------------
# Rule-specific acceptance thresholds
# ------------------------------------------------------------

IDENTITY_MAPPING_THRESHOLD = 0.80
EXACT_LABEL_MAPPING_THRESHOLD = 0.85
KNOWN_SYNONYM_MAPPING_THRESHOLD = 0.85
MODEL_INFERRED_MAPPING_THRESHOLD = 0.90
HISTORICAL_SYNONYM_THRESHOLD = 0.80
HIGH_CERTAINTY_PHRASE_THRESHOLD = 0.80
KEEP_SOURCE_THRESHOLD = 0.85
CORRECTION_THRESHOLD = 0.98

# Historical synonym support
MIN_ACCEPTED_SYNONYM_OBSERVATIONS = 2
MIN_ACCEPTED_SYNONYM_ISSUERS = 1
MIN_ACCEPTED_SYNONYM_FILINGS = 2
MIN_TRUSTED_RAW_OBSERVATIONS = 2
MIN_TRUSTED_DEDUPLICATED_ROWS = 2

# Fragment-quality gate
MIN_COMPLETE_LABEL_CHARACTERS = 4
FRAGMENT_TRAILING_TERMS = {
    "of",
    "was",
    "were",
    "during",
    "for",
    "from",
    "with",
    "in",
    "to",
}

REPORTING_PERIOD_SUFFIX_PATTERNS = [
    r"\s+as\s+at(?:\s+.+)?$",
    r"\s+as\s+of(?:\s+.+)?$",
    r"\s+at\s+\d{1,2}\s+"
    r"(?:january|february|march|april|may|june|"
    r"july|august|september|october|november|december)"
    r"(?:\s+\d{4})?$",
    r"\s+for\s+the\s+(?:year|period)\s+ended(?:\s+.+)?$",
    r"\s+during\s+the\s+(?:year|period)\s+ended(?:\s+.+)?$",
]

ROMAN_NOTE_TOKEN_PATTERN = (
    r"(?<![a-z])"
    r"(?:i|ii|iii|iv|v|vi|vii|viii|ix|x|"
    r"xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xix|xx|"
    r"xxi|xxii|xxiii|xxiv|xxv|xxvi|xxvii|xxviii|"
    r"xxix|xxx|xxxi|xxxii|xxxiii|xxxiv|xxxv|xxxvi|"
    r"xxxvii|xxxviii|xxxix|xl|xli|xlii|xliii|xliv|"
    r"xlv|xlvi|xlvii|xlviii|xlix|l|li|lii|liii|liv|"
    r"lv|lvi|lvii|lviii|lix|lx|lxi|lxii|lxiii|lxiv|"
    r"lxv|lxvi|lxvii|lxviii|lxix|lxx)"
    r"(?![a-z])"
)


# ------------------------------------------------------------
# Source and persistence controls
# ------------------------------------------------------------

CANONICAL_US_FACT_SOURCE = (
    "sec_fundamentals_standardised_df"
)

DUPLICATE_US_FACT_SOURCE = (
    "usa_fundamentals_standardised_df"
)

SAVE_PARQUET = True
SAVE_CSV_SUMMARIES = True


# ------------------------------------------------------------
# Azure token-price assumptions
# ------------------------------------------------------------

AZURE_INPUT_COST_PER_MILLION_USD = float(
    os.environ.get(
        "AZURE_INPUT_COST_PER_MILLION_USD",
        "0.25",
    )
)

AZURE_OUTPUT_COST_PER_MILLION_USD = float(
    os.environ.get(
        "AZURE_OUTPUT_COST_PER_MILLION_USD",
        "2.00",
    )
)

MAX_ESTIMATED_RUN_COST_USD = float(
    os.environ.get(
        "MAX_ESTIMATED_RUN_COST_USD",
        "25.00",
    )
)


# ------------------------------------------------------------
# Secrets
# ------------------------------------------------------------

def get_secret(name: str) -> Optional[str]:
    """Read a credential from Colab Secrets or the environment."""
    if IN_COLAB and userdata is not None:
        try:
            value = userdata.get(name)
            if value:
                return str(value).strip()
        except Exception:
            pass

    value = os.environ.get(name)

    return (
        str(value).strip()
        if value
        else None
    )

AZURE_OPENAI_API_KEY = get_secret(
    "AZURE_OPENAI_API_KEY"
)

AZURE_OPENAI_DEPLOYMENT = get_secret(
    "AZURE_OPENAI_DEPLOYMENT"
)

AZURE_OPENAI_ENDPOINT = get_secret(
    "AZURE_OPENAI_ENDPOINT"
)


print("Project root:", PROJECT_ROOT)
print("Interim root:", INTERIM_ROOT)
print("Block 9 output:", BLOCK_9_OUTPUT_DIR)
print("Azure deployment:", AZURE_OPENAI_DEPLOYMENT)
print(
    "Deterministic dry run:",
    DETERMINISTIC_DRY_RUN,
)
print("Cache enabled:", ENABLE_CACHE)
print(
    "Fresh Azure requests per task:",
    MAX_AI_RECORDS_PER_TASK,
)


Project root: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy
Interim root: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/interim
Block 9 output: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/interim/block_9
Azure deployment: gpt-5-mini
Deterministic dry run: False
Cache enabled: True
Fresh Azure requests per task: 1


In [3]:
# ============================================================
# 3. RECOVER AND BOOTSTRAP THE LOCAL SQLITE AI CACHE
# ============================================================

import os
import shlex
import shutil
import sqlite3
import subprocess

from datetime import datetime, timezone
from pathlib import Path


CACHE_RECOVERY_TIMESTAMP = (
    datetime.now(timezone.utc)
    .strftime("%Y%m%dT%H%M%SZ")
)

PERSISTENT_AI_CACHE_PATH = Path(
    PERSISTENT_AI_CACHE_PATH
)

LOCAL_AI_CACHE_PATH = Path(
    LOCAL_AI_CACHE_PATH
)

LOCAL_RECOVERY_SOURCE_PATH = Path(
    "/content/block_9_ai_cache_recovery_source.sqlite"
)

LOCAL_RECOVERED_CACHE_PATH = Path(
    "/content/block_9_ai_cache_recovered.sqlite"
)

LOCAL_SNAPSHOT_PATH = Path(
    "/content/block_9_ai_cache_snapshot.sqlite"
)

CORRUPT_BACKUP_PATH = (
    PERSISTENT_AI_CACHE_PATH.parent
    / (
        PERSISTENT_AI_CACHE_PATH.stem
        + "_CORRUPT_"
        + CACHE_RECOVERY_TIMESTAMP
        + PERSISTENT_AI_CACHE_PATH.suffix
    )
)


def remove_sqlite_files(
    database_path: Path,
) -> None:
    for path in [
        database_path,
        Path(str(database_path) + "-wal"),
        Path(str(database_path) + "-shm"),
    ]:
        if path.exists():
            path.unlink()


def copy_sqlite_bundle(
    source_path: Path,
    destination_path: Path,
) -> None:
    """
    Copy the SQLite database together with any WAL and SHM sidecars.
    """
    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    remove_sqlite_files(
        destination_path
    )

    if source_path.exists():
        shutil.copy2(
            source_path,
            destination_path,
        )

    for suffix in [
        "-wal",
        "-shm",
    ]:
        source_sidecar = Path(
            str(source_path) + suffix
        )

        destination_sidecar = Path(
            str(destination_path) + suffix
        )

        if source_sidecar.exists():
            shutil.copy2(
                source_sidecar,
                destination_sidecar,
            )


def sqlite_integrity_check(
    database_path: Path,
) -> str:
    if not database_path.exists():
        return "FILE_MISSING"

    try:
        with sqlite3.connect(
            database_path
        ) as connection:
            result = connection.execute(
                "PRAGMA integrity_check;"
            ).fetchone()

        if result is None:
            return "NO_RESULT"

        return str(result[0])

    except Exception as exc:
        return (
            "FAILED: "
            + repr(exc)
        )


def sqlite_cache_row_count(
    database_path: Path,
) -> int:
    if not database_path.exists():
        return 0

    try:
        with sqlite3.connect(
            database_path
        ) as connection:
            table_exists = connection.execute(
                """
                SELECT 1
                FROM sqlite_master
                WHERE type = 'table'
                  AND name = 'ai_cache'
                LIMIT 1
                """
            ).fetchone()

            if table_exists is None:
                return 0

            result = connection.execute(
                """
                SELECT COUNT(*)
                FROM ai_cache
                """
            ).fetchone()

        return int(
            result[0]
            if result
            else 0
        )

    except Exception:
        return 0


def ensure_sqlite_cli() -> None:
    if shutil.which("sqlite3") is not None:
        return

    print(
        "Installing the SQLite command-line utility..."
    )

    subprocess.run(
        [
            "apt-get",
            "-qq",
            "update",
        ],
        check=True,
    )

    subprocess.run(
        [
            "apt-get",
            "-qq",
            "install",
            "-y",
            "sqlite3",
        ],
        check=True,
    )


def recover_sqlite_database(
    damaged_path: Path,
    recovered_path: Path,
) -> None:
    ensure_sqlite_cli()

    remove_sqlite_files(
        recovered_path
    )

    damaged_quoted = shlex.quote(
        str(damaged_path)
    )

    recovered_quoted = shlex.quote(
        str(recovered_path)
    )

    command = (
        f"sqlite3 {damaged_quoted} '.recover' "
        f"| sqlite3 {recovered_quoted}"
    )

    result = subprocess.run(
        [
            "bash",
            "-lc",
            command,
        ],
        capture_output=True,
        text=True,
    )

    print(
        "SQLite recovery return code:",
        result.returncode,
    )

    if result.stderr:
        print(
            "SQLite recovery messages:"
        )
        print(
            result.stderr[-4000:]
        )


def sync_local_ai_cache_to_drive() -> None:
    """
    Create a transactionally consistent local snapshot and replace
    the Google Drive cache atomically.
    """
    local_path = Path(
        AI_CACHE_PATH
    )

    persistent_path = Path(
        PERSISTENT_AI_CACHE_PATH
    )

    if not local_path.exists():
        raise FileNotFoundError(
            "The local AI cache does not exist: "
            + str(local_path)
        )

    local_integrity = sqlite_integrity_check(
        local_path
    )

    if local_integrity != "ok":
        raise RuntimeError(
            "The local AI cache failed its integrity check: "
            + local_integrity
        )

    remove_sqlite_files(
        LOCAL_SNAPSHOT_PATH
    )

    with sqlite3.connect(
        local_path
    ) as source_connection:
        try:
            source_connection.execute(
                "PRAGMA wal_checkpoint(FULL);"
            )
        except Exception:
            pass

        with sqlite3.connect(
            LOCAL_SNAPSHOT_PATH
        ) as destination_connection:
            source_connection.backup(
                destination_connection
            )

    snapshot_integrity = sqlite_integrity_check(
        LOCAL_SNAPSHOT_PATH
    )

    if snapshot_integrity != "ok":
        raise RuntimeError(
            "The local AI-cache snapshot failed its "
            "integrity check: "
            + snapshot_integrity
        )

    persistent_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_drive_path = Path(
        str(persistent_path) + ".tmp"
    )

    if temporary_drive_path.exists():
        temporary_drive_path.unlink()

    shutil.copy2(
        LOCAL_SNAPSHOT_PATH,
        temporary_drive_path,
    )

    os.replace(
        temporary_drive_path,
        persistent_path,
    )

    # A clean snapshot must not retain stale Drive sidecars.
    for suffix in [
        "-wal",
        "-shm",
    ]:
        sidecar = Path(
            str(persistent_path) + suffix
        )

        if sidecar.exists():
            sidecar.unlink()

    print(
        "Local AI cache synchronised safely to Google Drive."
    )

    print(
        "Persistent rows:",
        f"{sqlite_cache_row_count(persistent_path):,}",
    )


# ------------------------------------------------------------
# Preserve the damaged persistent database
# ------------------------------------------------------------

print(
    "Persistent cache:",
    PERSISTENT_AI_CACHE_PATH,
)

print(
    "Persistent-cache integrity:",
    sqlite_integrity_check(
        PERSISTENT_AI_CACHE_PATH
    ),
)

if PERSISTENT_AI_CACHE_PATH.exists():
    copy_sqlite_bundle(
        PERSISTENT_AI_CACHE_PATH,
        CORRUPT_BACKUP_PATH,
    )

    print(
        "Safety backup created:",
        CORRUPT_BACKUP_PATH,
    )


# ------------------------------------------------------------
# Copy the Drive cache to the local runtime
# ------------------------------------------------------------

if PERSISTENT_AI_CACHE_PATH.exists():
    copy_sqlite_bundle(
        PERSISTENT_AI_CACHE_PATH,
        LOCAL_RECOVERY_SOURCE_PATH,
    )

    recovery_source_integrity = (
        sqlite_integrity_check(
            LOCAL_RECOVERY_SOURCE_PATH
        )
    )

    print(
        "Local copy integrity:",
        recovery_source_integrity,
    )

    if recovery_source_integrity == "ok":
        copy_sqlite_bundle(
            LOCAL_RECOVERY_SOURCE_PATH,
            LOCAL_AI_CACHE_PATH,
        )

    else:
        print(
            "The copied database is malformed. "
            "Attempting SQLite .recover..."
        )

        recover_sqlite_database(
            LOCAL_RECOVERY_SOURCE_PATH,
            LOCAL_RECOVERED_CACHE_PATH,
        )

        recovered_integrity = (
            sqlite_integrity_check(
                LOCAL_RECOVERED_CACHE_PATH
            )
        )

        print(
            "Recovered-cache integrity:",
            recovered_integrity,
        )

        if recovered_integrity != "ok":
            raise RuntimeError(
                "SQLite recovery did not produce a valid cache. "
                "Do not continue to Step 12. The damaged cache "
                "has been preserved at: "
                + str(CORRUPT_BACKUP_PATH)
            )

        copy_sqlite_bundle(
            LOCAL_RECOVERED_CACHE_PATH,
            LOCAL_AI_CACHE_PATH,
        )

else:
    print(
        "No persistent cache exists. "
        "A new local cache will be created."
    )

    remove_sqlite_files(
        LOCAL_AI_CACHE_PATH
    )


# ------------------------------------------------------------
# Final local-cache validation
# ------------------------------------------------------------

AI_CACHE_PATH = LOCAL_AI_CACHE_PATH

local_final_integrity = sqlite_integrity_check(
    AI_CACHE_PATH
)

if (
    AI_CACHE_PATH.exists()
    and local_final_integrity != "ok"
):
    raise RuntimeError(
        "The local cache remains invalid: "
        + local_final_integrity
    )

print(
    "Live SQLite path:",
    AI_CACHE_PATH,
)

print(
    "Live cache integrity:",
    local_final_integrity,
)

print(
    "Recovered cache rows:",
    f"{sqlite_cache_row_count(AI_CACHE_PATH):,}",
)

print(
    "CACHE BOOTSTRAP COMPLETE"
)

Persistent cache: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/interim/block_9/block_9_ai_cache.sqlite
Persistent-cache integrity: ok
Safety backup created: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/interim/block_9/block_9_ai_cache_CORRUPT_20260801T163756Z.sqlite
Local copy integrity: ok
Live SQLite path: /content/block_9_ai_cache.sqlite
Live cache integrity: ok
Recovered cache rows: 58,962
CACHE BOOTSTRAP COMPLETE


In [4]:
# ============================================================
# 4. UTILITIES
# ============================================================
NULL_TEXT_VALUES = {"", "none", "null", "nan", "na", "n/a", "<na>"}

def safe_text(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    return None if text.lower() in NULL_TEXT_VALUES else text

def stable_hash(value):
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()

def safe_numeric(value):
    return pd.to_numeric(value, errors="coerce")

def normalise_identifier(value):
    text = safe_text(value)
    if text is None:
        return None
    text = unicodedata.normalize("NFKC", text).lower()
    return re.sub(r"\s+", " ", text).strip()

def normalise_account_label(value):
    text = safe_text(value)
    if text is None:
        return None
    text = unicodedata.normalize("NFKC", text).lower()
    text = re.sub(r"[\n\r\t]+", " ", text)
    text = re.sub(r"[^\w\s%/\-]", " ", text)
    text = re.sub(r"\b\d+(?:[.,]\d+)*\b", "<num>", text)
    return re.sub(r"\s+", " ", text).strip()

def coalesce_columns(frame, candidates, default=pd.NA):
    existing = [c for c in candidates if c in frame.columns]
    if not existing:
        return pd.Series(default, index=frame.index, dtype="object")
    out = frame[existing[0]].copy()
    for col in existing[1:]:
        out = out.where(out.notna(), frame[col])
    return out

def ensure_columns(frame, columns):
    out = frame.copy()
    for col in columns:
        if col not in out.columns:
            out[col] = pd.NA
    return out

def infer_region(table_name):
    name = table_name.lower()
    if "australia" in name: return "Australia"
    if "europe" in name: return "Europe"
    if "japan" in name: return "Japan"
    if "korea" in name: return "Korea"
    if "hong_kong" in name or "china" in name: return "Hong Kong and China"
    if "sec_" in name or "usa_" in name: return "USA"
    return "Unknown"

def infer_block(region):
    return {
        "USA": 3, "Europe": 4, "Japan": 5, "Korea": 6,
        "Hong Kong and China": 7, "Australia": 8,
    }.get(region)

def sign_bucket(value):
    x = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
    if pd.isna(x): return "MISSING"
    if x > 0: return "POSITIVE"
    if x < 0: return "NEGATIVE"
    return "ZERO"

def magnitude_bucket(value):
    x = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
    if pd.isna(x): return "MISSING"
    x = abs(float(x))
    if x == 0: return "ZERO"
    if x < 1e3: return "LT_1K"
    if x < 1e6: return "1K_TO_1M"
    if x < 1e9: return "1M_TO_1B"
    if x < 1e12: return "1B_TO_1T"
    return "GT_1T"

def relative_bucket(value):
    x = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
    if pd.isna(x): return "MISSING"
    x = abs(float(x))
    if x == 0: return "ZERO"
    if x < 0.01: return "LT_1_PERCENT"
    if x < 0.10: return "1_TO_10_PERCENT"
    if x < 0.50: return "10_TO_50_PERCENT"
    if x < 1.00: return "50_TO_100_PERCENT"
    return "GT_100_PERCENT"

def json_value(value):
    if value is None or pd.isna(value):
        return None
    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()
    if isinstance(value, np.generic):
        return value.item()
    return value

def canonical_text(value: Any) -> Optional[str]:
    """Canonicalise contextual text without discarding accounting meaning."""
    text = safe_text(value)
    if text is None:
        return None
    text = unicodedata.normalize("NFKC", text).casefold()
    text = re.sub(r"[‐-―]", "-", text)
    text = re.sub(r"[^\w\s%/\-]", " ", text)
    text = re.sub(r"(?:20)?\d{2}", "<year>", text)
    text = re.sub(r"\d+(?:[.,]\d+)*", "<num>", text)
    return re.sub(r"\s+", " ", text).strip() or None


def canonical_unit(value: Any, scale: Any = None) -> Optional[str]:
    unit = canonical_text(value)
    numeric_scale = pd.to_numeric(pd.Series([scale]), errors="coerce").iloc[0]
    scale_text = None if pd.isna(numeric_scale) else f"{float(numeric_scale):g}"
    return "|".join(x for x in [unit, scale_text] if x) or None


def safe_json_dumps(value: Any) -> str:
    return json.dumps(value, sort_keys=True, ensure_ascii=False, default=str, separators=(",", ":"))


def utc_now() -> datetime:
    return datetime.now(timezone.utc)


In [5]:
# ============================================================
# 5. AI RESPONSE SCHEMAS
# ============================================================

class AccountMappingResponse(BaseModel):
    decision: Literal[
        "MAP",
        "KEEP_SOURCE",
        "REJECT",
        "UNRESOLVED",
    ]
    candidate_index: Optional[int] = Field(
        default=None,
        ge=0,
    )
    confidence: float = Field(ge=0, le=1)
    evidence_text: str
    reasoning_summary: str
    statement_type: Optional[str] = None
    reporting_scope: Optional[str] = None


class FactQCResponse(BaseModel):
    decision: Literal[
        "KEEP_SOURCE",
        "PROPOSE_CORRECTION",
        "REJECT_FACT",
        "UNRESOLVED",
    ]
    confidence: float = Field(ge=0, le=1)
    evidence_text: str
    reasoning_summary: str
    proposed_value: Optional[float] = None
    proposed_currency: Optional[str] = None
    proposed_unit_scale: Optional[float] = None
    proposed_sign_multiplier: Optional[float] = None
    statement_type: Optional[str] = None
    reporting_scope: Optional[str] = None


class ValueAnomalyResponse(FactQCResponse):
    pass


RESPONSE_SCHEMA_BY_TASK = {
    "ACCOUNT_MAPPING": AccountMappingResponse,
    "FACT_QC": FactQCResponse,
    "VALUE_ANOMALY": ValueAnomalyResponse,
}


In [ ]:
# ============================================================
# 6. MANIFEST-DRIVEN LOADING AND STANDARDISATION OF BLOCKS 3–8
# ============================================================
# Block 9 discovers each upstream block's manifest and loads its persisted tables.

REGION_BY_BLOCK = {
    3: "USA",
    4: "Europe",
    5: "Japan",
    6: "Korea",
    7: "Hong Kong and China",
    8: "Australia",
}

STANDARDISED_FACT_PATTERNS = (
    "fundamentals_standardised_df",
    "fundamentals_standardized_df",
    "standardized_fundamentals_df",
    "standardised_facts_df",
    "standardized_facts_df",
)

REVIEW_TABLE_PATTERNS = (
    "review_queue",
    "unmapped_account",
    "low_quality",
    "unresolved",
    "duplicate_primary",
    "duplicate_pdf",
    "quality",
)

ENTITY_REVIEW_PATTERNS = (
    "entity_bridge_unresolved",
    "entity_resolution",
    "unmatched_filings",
    "relationship_quality",
)


# Review tables identify the notebook-level queue that generated an exception.
# Where the originating production namespace is known, retain it separately.
# A production row number is never inferred from the review-table index.
PRODUCTION_LINEAGE_BY_REVIEW_TABLE = {
    # Block 7 — mainland China
    "china_mapping_review_queue_df": {
        "region": "MAINLAND_CHINA",
        "table": "china_fundamentals_standardised_df",
    },
    "china_unmapped_account_inventory_df": {
        "region": "MAINLAND_CHINA",
        "table": "china_fundamentals_standardised_df",
    },

    # Block 7 — Hong Kong
    "hong_kong_mapping_review_queue_df": {
        "region": "HONG_KONG",
        "table": "hong_kong_fundamentals_standardised_df",
    },
    "hong_kong_unmapped_account_inventory_df": {
        "region": "HONG_KONG",
        "table": "hong_kong_fundamentals_standardised_df",
    },
    "hong_kong_fuzzy_label_quality_df": {
        "region": "HONG_KONG",
        "table": "hong_kong_fundamentals_standardised_df",
    },

    # Block 8 — Australia
    "australia_mapping_review_queue_df": {
        "region": "AUSTRALIA",
        "table": "australia_fundamentals_standardised_df",
    },
    "australia_unmapped_account_inventory_df": {
        "region": "AUSTRALIA",
        "table": "australia_fundamentals_standardised_df",
    },
}

AGGREGATED_REVIEW_PATTERNS = (
    "unmapped_account_inventory",
    "account_inventory",
    "label_inventory",
    "coverage_report",
    "quality_report",
    "mapping_quality",
    "fuzzy_label_quality",
)

CANONICAL_SOURCE_COLUMNS = [
    # Queue provenance retained for audit and cache reproducibility.
    "source_table",
    "region",
    "queue_source_table",
    "queue_row_number",
    "upstream_block",

    # Classification and downstream publication contract.
    "source_record_granularity",
    "production_source_region",
    "production_source_table",
    "production_source_row_number",
    "fact_publishable",

    # Issuer, filing and document context.
    "global_issuer_id",
    "filing_id",
    "global_source_observation_id",
    "source_row_number",
    "document_id",
    "document_path",

    # Fundamental or review payload.
    "source_account_label",
    "standard_concept",
    "reported_value",
    "reported_value_numeric",
    "currency",
    "unit_scale",
    "statement_type",
    "reporting_scope",
    "period_end",
    "source_text",

    # QC metadata.
    "exception_code",
    "qc_rule_id",
    "qc_failure_type",
    "anomaly_type",
    "relative_difference",

    # Block 9 internal queue-row identifier only.
    "_original_row_index",
]


def read_json(path: Path) -> dict[str, Any]:
    with Path(path).open("r", encoding="utf-8") as file:
        return json.load(file)


def candidate_manifest_paths(block_number: int) -> list[Path]:
    output_dir = BLOCK_OUTPUT_DIRS[block_number]
    candidates = [
        output_dir / f"block_{block_number}_manifest.json",
        output_dir / "manifest.json",
    ]

    # The pipeline temporarily persisted Australia's Block 8 artefacts
    # beneath block_9. Retain that compatibility, but verify the manifest identity.
    if block_number == 8:
        candidates.extend([
            INTERIM_ROOT / "block_9" / "block_9_manifest.json",
            INTERIM_ROOT / "block_9" / "manifest.json",
        ])
    return candidates


def discover_manifest(block_number: int) -> tuple[Path, dict[str, Any]]:
    checked = candidate_manifest_paths(block_number)
    for path in checked:
        if not path.exists():
            continue
        manifest = read_json(path)
        block_name = str(manifest.get("block_name", "")).lower()
        if block_number == 8 and path.parent.name == "block_9":
            if "australia" not in block_name:
                continue
        return path, manifest

    raise FileNotFoundError(
        f"No usable manifest found for Block {block_number}. "
        f"Checked: {[str(path) for path in checked]}"
    )


def manifest_table_index(manifest: dict[str, Any]) -> dict[str, dict[str, Any]]:
    return {
        record["table_name"]: record
        for record in manifest.get("tables", [])
        if record.get("table_name")
    }


def resolve_table_path(record: dict[str, Any], manifest_path: Path) -> Path:
    raw_path = record.get("path")
    table_name = record["table_name"]

    candidates: list[Path] = []
    if raw_path:
        path = Path(raw_path)
        candidates.extend([
            path,
            manifest_path.parent / path.name,
        ])

    candidates.extend([
        manifest_path.parent / f"{table_name}.parquet",
        manifest_path.parent / f"{table_name}.csv",
    ])

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        f"Could not resolve persisted table '{table_name}' from {manifest_path}. "
        f"Checked: {[str(path) for path in candidates]}"
    )


def load_manifest_table(
    manifest_path: Path,
    manifest: dict[str, Any],
    table_name: str,
    *,
    required: bool = False,
) -> pd.DataFrame:
    records = manifest_table_index(manifest)
    if table_name not in records:
        if required:
            raise KeyError(f"{table_name} is absent from {manifest_path}")
        return pd.DataFrame()

    path = resolve_table_path(records[table_name], manifest_path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, low_memory=False)
    raise ValueError(f"Unsupported persisted table format: {path}")


def table_name_matches(table_name: str, patterns: tuple[str, ...]) -> bool:
    lowered = table_name.lower()
    return any(pattern in lowered for pattern in patterns)


# Fail early with a complete manifest diagnostic rather than looking for
# in-memory DataFrames that do not exist in a fresh notebook runtime.
upstream_manifests: dict[int, dict[str, Any]] = {}
manifest_errors: list[str] = []

for block_number in range(3, 9):
    try:
        manifest_path, manifest = discover_manifest(block_number)
        upstream_manifests[block_number] = {
            "path": manifest_path,
            "manifest": manifest,
            "tables": manifest_table_index(manifest),
        }
        print(
            f"Block {block_number}: {manifest_path} "
            f"({len(upstream_manifests[block_number]['tables']):,} tables)"
        )
    except Exception as exc:
        manifest_errors.append(f"Block {block_number}: {type(exc).__name__}: {exc}")

if manifest_errors:
    raise RuntimeError(
        "Block 9 could not load the complete persisted Block 3–8 input set.\n"
        "Confirm that the upstream notebooks have saved their manifests and Parquet "
        "outputs beneath data/interim/block_3 through block_8.\n\n"
        + "\n".join(manifest_errors)
    )


inventory_rows: list[dict[str, Any]] = []
standardised_frames: list[pd.DataFrame] = []
load_errors: list[str] = []

for block_number, information in upstream_manifests.items():
    region = REGION_BY_BLOCK[block_number]

    for table_name, record in information["tables"].items():
        is_fact = table_name_matches(table_name, STANDARDISED_FACT_PATTERNS)
        is_review = table_name_matches(
            table_name, REVIEW_TABLE_PATTERNS + ENTITY_REVIEW_PATTERNS
        )
        if not (is_fact or is_review):
            continue

        categories = []
        if is_fact:
            categories.append("standardised_facts")
        if is_review:
            categories.append("review_or_quality")

        inventory_rows.append({
            "block_number": block_number,
            "region": region,
            "table_name": table_name,
            "categories": "|".join(categories),
            "manifest_row_count": record.get("row_count"),
            "manifest_column_count": record.get("column_count"),
            "path": record.get("path"),
        })

        try:
            frame = load_manifest_table(
                information["path"],
                information["manifest"],
                table_name,
            )
        except Exception as exc:
            load_errors.append(
                f"Block {block_number} / {table_name}: {type(exc).__name__}: {exc}"
            )
            continue

        if frame.empty:
            continue

        out = pd.DataFrame(index=frame.index)

        # Queue provenance: these fields identify the persisted input table
        # that Block 9 loaded. They are not assumed to identify a production fact.
        out["source_table"] = table_name
        out["region"] = region
        out["queue_source_table"] = table_name
        out["queue_row_number"] = frame.index.astype("string")
        out["upstream_block"] = block_number

        table_name_lower = table_name.casefold()
        is_aggregated_inventory = any(
            pattern in table_name_lower
            for pattern in AGGREGATED_REVIEW_PATTERNS
        )

        if is_fact:
            record_granularity = "PRODUCTION_FACT"
        elif is_aggregated_inventory:
            record_granularity = "AGGREGATED_LABEL"
        else:
            record_granularity = "ROW_LEVEL_REVIEW"

        out["source_record_granularity"] = record_granularity

        production_lineage = PRODUCTION_LINEAGE_BY_REVIEW_TABLE.get(
            table_name,
            {},
        )

        if is_fact:
            production_region = region.upper().replace(" ", "_")
            production_table = table_name
        else:
            production_region = production_lineage.get("region", pd.NA)
            production_table = production_lineage.get("table", pd.NA)

        out["production_source_region"] = production_region
        out["production_source_table"] = production_table

        # Only preserve a genuine upstream production-row identifier.
        # Never substitute the review-table row index.
        out["production_source_row_number"] = safe_numeric(
            coalesce_columns(
                frame,
                [
                    "production_source_row_number",
                    "source_fact_row_number",
                    "standardised_fact_row_number",
                    "standardized_fact_row_number",
                    "regional_fact_row_number",
                    "production_row_number",
                ],
            )
        )
        out["global_issuer_id"] = coalesce_columns(
            frame,
            ["global_issuer_id", "issuer_id", "economic_issuer_id",
             "canonical_issuer_id", "entity_id", "company_id", "cik", "lei"],
        )
        out["filing_id"] = coalesce_columns(
            frame,
            ["filing_id", "document_id", "announcement_id",
             "accession_number", "report_id"],
        )

        # ------------------------------------------------------------
        # Preserve permanent source lineage for Block 10
        # ------------------------------------------------------------

        out["global_source_observation_id"] = coalesce_columns(
            frame,
            [
                "global_source_observation_id",
                "source_observation_id",
                "global_fact_observation_id",
                "observation_id",
            ],
        )

        out["source_row_number"] = safe_numeric(
            coalesce_columns(
                frame,
                [
                    "source_row_number",
                    "source_row_index",
                    "row_number",
                    "row_index",
                    "original_row_number",
                ],
            )
        )

        out["document_id"] = coalesce_columns(
            frame,
            [
                "document_id",
                "source_document_id",
                "filing_document_id",
                "announcement_id",
                "accession_number",
                "report_id",
            ],
        )

        out["document_path"] = coalesce_columns(
            frame,
            ["document_path", "local_path", "pdf_path",
             "filing_path", "source_path", "file_path", "path", "url"],
        )
        out["source_account_label"] = coalesce_columns(
            frame,
            ["source_account_label", "account_label", "source_label",
             "reported_label", "raw_label", "label", "fact_label",
             "account_text", "account_name"],
        )
        out["standard_concept"] = coalesce_columns(
            frame,
            ["standard_concept", "proposed_standard_concept", "canonical_concept",
             "candidate_concept", "mapped_concept", "concept", "concept_name",
             "metric", "target_concept"],
        )
        out["reported_value"] = coalesce_columns(
            frame,
            ["reported_value", "value", "numeric_value",
             "fact_value", "amount"],
        )
        out["reported_value_numeric"] = safe_numeric(
            coalesce_columns(
                frame,
                ["reported_value_numeric", "numeric_value",
                 "reported_value", "value", "fact_value", "amount"],
            )
        )
        out["currency"] = coalesce_columns(
            frame, ["currency", "currency_code", "reported_currency"]
        )
        out["unit_scale"] = safe_numeric(
            coalesce_columns(
                frame,
                ["unit_scale", "scale", "scale_multiplier", "reported_unit_scale"],
                default=1.0,
            )
        ).fillna(1.0)
        out["statement_type"] = coalesce_columns(
            frame,
            ["statement_type", "financial_statement", "statement", "statement_name"],
        )
        out["reporting_scope"] = coalesce_columns(
            frame, ["reporting_scope", "scope", "consolidation_scope"]
        )
        out["period_end"] = pd.to_datetime(
            coalesce_columns(
                frame,
                ["period_end", "end_date", "reporting_period_end",
                 "fiscal_period_end"],
            ),
            errors="coerce",
            utc=True,
        )
        out["source_text"] = coalesce_columns(
            frame,
            ["source_text", "context", "raw_text", "text",
             "evidence_text", "context_text", "filing_text", "description"],
        )
        out["exception_code"] = coalesce_columns(
            frame,
            ["exception_code", "quality_flag", "qc_flag", "review_reason",
             "reason", "quality_issue", "status", "resolution_status", "issue_type"],
        )
        out["qc_rule_id"] = coalesce_columns(
            frame, ["qc_rule_id", "rule_id", "validation_rule"]
        )
        out["qc_failure_type"] = coalesce_columns(
            frame, ["qc_failure_type", "failure_type", "validation_failure"]
        )
        out["anomaly_type"] = coalesce_columns(
            frame, ["anomaly_type", "value_anomaly_type", "outlier_type"]
        )
        out["relative_difference"] = safe_numeric(
            coalesce_columns(
                frame,
                ["relative_difference", "relative_error",
                 "percentage_difference", "deviation_ratio"],
            )
        )
        out["_original_row_index"] = frame.index.astype("string")

        has_existing_production_lineage = (
            out["production_source_region"].notna()
            & out["production_source_table"].notna()
            & out["production_source_row_number"].notna()
        )

        has_complete_new_fact_payload = (
            out["global_issuer_id"].notna()
            & out["reported_value_numeric"].notna()
            & out["period_end"].notna()
        )

        # Existing rows are publishable only with genuine production lineage.
        # Row-level review records may instead become new-fact candidates when
        # they carry a complete factual payload. Aggregated labels are registry-only.
        out["fact_publishable"] = (
            has_existing_production_lineage
            | (
                out["source_record_granularity"].eq("ROW_LEVEL_REVIEW")
                & has_complete_new_fact_payload
            )
        )

        standardised_frames.append(out.reset_index(drop=True))

        del frame, out
        gc.collect()

if load_errors:
    raise RuntimeError(
        "One or more manifest-listed Block 3–8 input tables could not be loaded:\n"
        + "\n".join(load_errors)
    )

upstream_table_inventory_df = pd.DataFrame(inventory_rows)
upstream_relevant_table_inventory_df = upstream_table_inventory_df.copy()

regional_source_records_df = (
    pd.concat(standardised_frames, ignore_index=True, sort=False, copy=False)
    if standardised_frames
    else pd.DataFrame(columns=CANONICAL_SOURCE_COLUMNS)
)

for column in CANONICAL_SOURCE_COLUMNS:
    if column not in regional_source_records_df.columns:
        regional_source_records_df[column] = pd.NA
regional_source_records_df = regional_source_records_df[CANONICAL_SOURCE_COLUMNS]

print("Relevant persisted upstream tables:", len(upstream_table_inventory_df))
display(upstream_table_inventory_df)
print("Standardised source rows:", f"{len(regional_source_records_df):,}")

if upstream_table_inventory_df.empty:
    raise RuntimeError(
        "The Block 3–8 manifests were found, but no standardised-fact or review/QC "
        "tables matched the expected naming patterns."
    )

if regional_source_records_df.empty:
    raise RuntimeError(
        "Relevant Block 3–8 tables were identified, but all loaded tables contain "
        "zero rows. Inspect upstream_table_inventory_df and the upstream manifests."
    )


In [7]:
# ============================================================
# 7. BUILD RAW EXCEPTION QUEUE
# ============================================================
def classify_exception(source_table: object, row: pd.Series) -> str | None:
    name = (safe_text(source_table) or "").lower()

    if (
        "mapping_review_queue" in name
        or "unmapped_account_inventory" in name
        or "mapping_quality" in name
        or "review_queue" in name
        or "account_inventory" in name
    ):
        return "ACCOUNT_MAPPING"

    # Column-driven fallback supports regional/versioned upstream table names.
    has_account_issue = any(
        safe_text(row.get(column)) is not None
        for column in ["source_account_label", "exception_code", "qc_failure_type"]
    )
    concept_missing = safe_text(row.get("standard_concept")) is None
    if has_account_issue and concept_missing:
        return "ACCOUNT_MAPPING"

    if "fundamentals" in name or pd.notna(row.get("reported_value_numeric")):
        if safe_text(row.get("anomaly_type")) is not None:
            return "VALUE_ANOMALY"

        rel = row.get("relative_difference")
        if pd.notna(rel) and abs(float(rel)) >= 0.10:
            return "VALUE_ANOMALY"

        if any(
            safe_text(row.get(column)) is not None
            for column in ["exception_code", "qc_rule_id", "qc_failure_type"]
        ):
            return "FACT_QC"

    return None

required_queue_columns = set(CANONICAL_SOURCE_COLUMNS)
missing_queue_columns = required_queue_columns.difference(regional_source_records_df.columns)
if missing_queue_columns:
    raise RuntimeError(
        "Internal Block 9 schema error. Missing standardised columns: "
        + ", ".join(sorted(missing_queue_columns))
    )

queue = regional_source_records_df.copy()
queue["exception_type"] = [
    classify_exception(row.get("source_table"), row)
    for _, row in queue.iterrows()
]
queue = queue.loc[queue["exception_type"].notna()].reset_index(drop=True)

if queue.empty:
    source_counts = (
        regional_source_records_df.groupby("source_table", dropna=False)
        .size().reset_index(name="rows")
    )
    display(source_counts)
    raise RuntimeError(
        "Block 9 found upstream records but none qualified as AI/QC exceptions. "
        "Confirm that the mapping-review, QC-flag or anomaly columns from Blocks 3–8 "
        "are present and populated."
    )

queue["normalised_source_account_label"] = (
    queue["source_account_label"].map(normalise_account_label)
)
queue["normalised_standard_concept"] = (
    queue["standard_concept"].map(normalise_identifier)
)
queue["value_sign_bucket"] = queue["reported_value_numeric"].map(sign_bucket)
queue["value_magnitude_bucket"] = queue["reported_value_numeric"].map(magnitude_bucket)
queue["relative_difference_bucket"] = queue["relative_difference"].map(relative_bucket)

queue["review_record_id"] = [
    stable_hash("||".join([
        safe_text(row.get("source_table")) or "",
        safe_text(row.get("_original_row_index")) or "",
        safe_text(row.get("global_issuer_id")) or "",
        safe_text(row.get("filing_id")) or "",
        safe_text(row.get("source_account_label")) or "",
        safe_text(row.get("standard_concept")) or "",
        safe_text(row.get("period_end")) or "",
        safe_text(row.get("reported_value_numeric")) or "",
        safe_text(row.get("exception_type")) or "",
    ]))
    for _, row in queue.iterrows()
]

ai_exception_queue_raw_df = queue
del queue
gc.collect()

print("Raw exception rows:", f"{len(ai_exception_queue_raw_df):,}")
display(
    ai_exception_queue_raw_df.groupby(
        ["exception_type", "region", "source_table"], dropna=False
    ).size().reset_index(name="raw_rows")
    .sort_values("raw_rows", ascending=False).head(30)
)


Raw exception rows: 95,376


,exception_type,region,source_table,raw_rows
5,ACCOUNT_MAPPING,Hong Kong and China,china_unmapped_account_inventory_df,43211
1,ACCOUNT_MAPPING,Australia,australia_unmapped_account_inventory_df,15354
6,ACCOUNT_MAPPING,Hong Kong and China,hong_kong_fuzzy_label_quality_df,15093
9,ACCOUNT_MAPPING,Hong Kong and China,hong_kong_unmapped_account_inventory_df,14556
4,ACCOUNT_MAPPING,Hong Kong and China,china_mapping_review_queue_df,4515
12,ACCOUNT_MAPPING,Korea,korea_unmapped_account_inventory_df,1893
8,ACCOUNT_MAPPING,Hong Kong and China,hong_kong_mapping_review_queue_df,684
7,ACCOUNT_MAPPING,Hong Kong and China,hong_kong_mapping_quality_df,14
3,ACCOUNT_MAPPING,Hong Kong and China,china_mapping_quality_df,12
2,ACCOUNT_MAPPING,Europe,europe_mapping_quality_df,9


In [8]:
# ============================================================
# 8. EXACT DEDUPLICATION AND DUPLICATE SOURCE-FEED CONTROL
# ============================================================
fact_key_columns = [
    col for col in [
        "region", "global_issuer_id", "filing_id", "standard_concept",
        "period_end", "reported_value_numeric", "currency", "unit_scale",
        "statement_type", "reporting_scope"
    ]
    if col in ai_exception_queue_raw_df.columns
]

work = ai_exception_queue_raw_df.copy()

work["_source_preference"] = np.select(
    [
        work["source_table"].eq(CANONICAL_US_FACT_SOURCE),
        work["source_table"].eq(DUPLICATE_US_FACT_SOURCE),
    ],
    [0, 1],
    default=0,
)

def exact_key(row):
    payload = {
        col: json_value(row.get(col))
        for col in fact_key_columns
    }
    payload["exception_type"] = row.get("exception_type")
    payload["source_account_label"] = (
        row.get("normalised_source_account_label")
        if row.get("exception_type") == "ACCOUNT_MAPPING"
        else None
    )
    return stable_hash(json.dumps(payload, sort_keys=True, default=str))

work["_exact_dedupe_key"] = work.apply(exact_key, axis=1)

dedupe_stats_df = (
    work.groupby("_exact_dedupe_key", dropna=False)
    .agg(
        exact_duplicate_raw_rows=("review_record_id", "size"),
        exact_duplicate_source_tables=("source_table", "nunique"),
    )
    .reset_index()
)

ai_exception_queue_deduplicated_df = (
    work.sort_values(
        ["_source_preference", "source_table", "review_record_id"],
        kind="mergesort",
    )
    .drop_duplicates("_exact_dedupe_key", keep="first")
    .merge(dedupe_stats_df, on="_exact_dedupe_key", how="left")
    .drop(columns=["_source_preference"], errors="ignore")
    .reset_index(drop=True)
)

duplicate_source_neutralisation_summary_df = pd.DataFrame([
    {"metric": "raw_exception_rows", "value": len(ai_exception_queue_raw_df)},
    {"metric": "rows_after_exact_deduplication",
     "value": len(ai_exception_queue_deduplicated_df)},
    {"metric": "rows_removed",
     "value": len(ai_exception_queue_raw_df) - len(ai_exception_queue_deduplicated_df)},
    {"metric": "multi_source_duplicate_keys",
     "value": int(dedupe_stats_df["exact_duplicate_source_tables"].gt(1).sum())},
])

display(duplicate_source_neutralisation_summary_df)
del work
gc.collect()

,metric,value
0,raw_exception_rows,95376
1,rows_after_exact_deduplication,74153
2,rows_removed,21223
3,multi_source_duplicate_keys,14560


0

In [9]:
# ============================================================
# 9. DETERMINISTIC ISSUE SIGNATURES AND COLLAPSE
# ============================================================
SIGNATURE_CONTEXT_COLUMNS = [
    "exception_type", "statement_type", "standard_concept",
    "source_account_label", "currency", "unit_scale", "global_issuer_id",
    "region", "reporting_scope", "exception_code", "qc_rule_id",
    "qc_failure_type", "anomaly_type", "relative_difference_bucket",
    "value_sign_bucket", "value_magnitude_bucket",
]


def make_issue_signature_payload(row: pd.Series) -> dict[str, Any]:
    """Build a stable semantic fingerprint for one accounting issue.

    Filing IDs, dates and raw numeric amounts are deliberately excluded where they
    would prevent the same recurring issue from collapsing. Issuer and contextual
    metadata remain included to avoid unsafe cross-issuer propagation.
    """
    exception_type = safe_text(row.get("exception_type"))
    payload = {
        "signature_version": 2,
        "exception_type": exception_type,
        "statement": normalise_identifier(row.get("statement_type")),
        "concept": normalise_identifier(row.get("standard_concept")),
        "account_text": canonical_text(row.get("source_account_label")),
        "units": canonical_unit(row.get("currency"), row.get("unit_scale")),
        "issuer": normalise_identifier(row.get("global_issuer_id")),
        "region": normalise_identifier(row.get("region")),
        "scope": normalise_identifier(row.get("reporting_scope")),
        "exception_code": normalise_identifier(row.get("exception_code")),
        "qc_rule": normalise_identifier(row.get("qc_rule_id")),
        "failure_type": normalise_identifier(row.get("qc_failure_type")),
        "anomaly_type": normalise_identifier(row.get("anomaly_type")),
    }
    if exception_type == "VALUE_ANOMALY":
        payload.update({
            "relative_bucket": safe_text(row.get("relative_difference_bucket")),
            "sign_bucket": safe_text(row.get("value_sign_bucket")),
            "magnitude_bucket": safe_text(row.get("value_magnitude_bucket")),
        })
    elif exception_type == "FACT_QC":
        payload.update({
            "sign_bucket": safe_text(row.get("value_sign_bucket")),
            "magnitude_bucket": safe_text(row.get("value_magnitude_bucket")),
        })
    return {k: v for k, v in payload.items() if v is not None}


def make_issue_signature(row: pd.Series) -> str:
    return stable_hash(safe_json_dumps(make_issue_signature_payload(row)))


signature_work = ai_exception_queue_deduplicated_df.copy()
signature_work["issue_signature"] = signature_work.apply(make_issue_signature, axis=1)
signature_work["signature_payload"] = signature_work.apply(
    lambda row: safe_json_dumps(make_issue_signature_payload(row)), axis=1
)

signature_stats_df = (
    signature_work.groupby("issue_signature", dropna=False)
    .agg(
        raw_row_count=("review_record_id", "size"),
        unique_issuer_count=("global_issuer_id", "nunique"),
        unique_filing_count=("filing_id", "nunique"),
        unique_source_table_count=("source_table", "nunique"),
    )
    .reset_index()
)

representative_sort_columns = [
    "issue_signature", "source_text", "global_issuer_id", "filing_id", "review_record_id"
]
representatives_df = (
    signature_work.assign(
        _evidence_length=signature_work["source_text"].astype("string").str.len().fillna(0),
        _issuer_present=signature_work["global_issuer_id"].notna().astype(int),
        _filing_present=signature_work["filing_id"].notna().astype(int),
    )
    .sort_values(
        ["issue_signature", "_evidence_length", "_issuer_present", "_filing_present", "review_record_id"],
        ascending=[True, False, False, False, True], kind="mergesort"
    )
    .drop_duplicates("issue_signature", keep="first")
    .drop(columns=["_evidence_length", "_issuer_present", "_filing_present"])
)

ai_exception_signature_queue_df = (
    representatives_df.merge(signature_stats_df, on="issue_signature", how="left")
    .reset_index(drop=True)
)
ai_exception_signature_queue_df["has_source_evidence"] = (
    ai_exception_signature_queue_df["source_text"].map(safe_text).notna()
)

collapse_summary_df = pd.DataFrame([
    {"metric": "deduplicated_exception_rows", "value": len(signature_work)},
    {"metric": "representative_signatures", "value": len(ai_exception_signature_queue_df)},
    {"metric": "rows_collapsed", "value": len(signature_work) - len(ai_exception_signature_queue_df)},
    {"metric": "duplicate_collapse_ratio", "value": (
        1 - len(ai_exception_signature_queue_df) / len(signature_work)
        if len(signature_work) else 0.0
    )},
])

display(collapse_summary_df)
print("Representative signatures:", f"{len(ai_exception_signature_queue_df):,}")


,metric,value
0,deduplicated_exception_rows,74153.000000
1,representative_signatures,69873.000000
2,rows_collapsed,4280.000000
3,duplicate_collapse_ratio,0.057719


Representative signatures: 69,873


In [10]:
# ============================================================
# 10. ELIGIBILITY, COMPLETION REGISTRY, CLUSTERS AND PRIORITY
# ============================================================

def pre_ai_fragment_reason(
    value: Any,
) -> Optional[str]:
    """Identify obvious layout, narrative and incomplete-label fragments."""
    text = safe_text(value)

    if text is None:
        return "MISSING_SOURCE_LABEL"

    normalised = unicodedata.normalize(
        "NFKC",
        text,
    ).casefold().strip()

    normalised = re.sub(
        r"\s+",
        " ",
        normalised,
    )

    if normalised in PRE_AI_LAYOUT_LABELS:
        return "DOCUMENT_LAYOUT_LABEL"

    if re.fullmatch(
        r"(page|p\.?)\s*\d+(\s*(of|/)\s*\d+)?",
        normalised,
    ):
        return "PAGE_NUMBER"

    if re.fullmatch(
        r"(19|20)\d{2}([\-/.年](0?[1-9]|1[0-2]))?"
        r"([\-/.月](0?[1-9]|[12]\d|3[01])日?)?",
        normalised,
    ):
        return "DATE_LABEL"

    if re.fullmatch(
        r"(rmb|cny|usd|eur|jpy|krw|aud|hkd|gbp|"
        r"in\s+(thousands?|millions?|billions?)|"
        r"单位[:：]?\s*(元|千元|百万元))",
        normalised,
    ):
        return "UNIT_LABEL"

    if not re.search(
        r"[a-z\u3400-\u9fff\u3040-\u30ff\uac00-\ud7af]",
        normalised,
    ):
        return "NON_TEXT_LABEL"

    words = re.findall(
        r"[a-z]+",
        normalised,
    )

    if (
        words
        and len(words) <= PRE_AI_FRAGMENT_MAX_WORDS
        and words[-1] in PRE_AI_TERMINAL_FRAGMENT_WORDS
    ):
        return "TRAILING_FRAGMENT"

    # Longer narrative fragments frequently contain a valid accounting
    # phrase but terminate in an incomplete connective or verb.
    if (
        words
        and words[-1] in PRE_AI_TERMINAL_FRAGMENT_WORDS
        and any(
            phrase in normalised
            for phrase in [
                "cash generated from",
                "cash used in",
                "activities",
                "receivables",
                "payables",
                "plant and equipment",
                "as of december",
                "as at",
            ]
        )
    ):
        return "TRAILING_NARRATIVE_FRAGMENT"

    if (
        words
        and len(words) <= 2
        and words[0] in PRE_AI_TERMINAL_FRAGMENT_WORDS
    ):
        return "LEADING_FRAGMENT"

    if any(
        normalised.startswith(starter)
        for starter in PRE_AI_NARRATIVE_STARTERS
    ) and len(words) >= 6:
        return "NARRATIVE_FRAGMENT"

    # Stand-alone section numbers and visibly corrupted OCR strings are not
    # useful model inputs.
    if re.fullmatch(
        r"[一二三四五六七八九十百零〇]+",
        normalised,
    ):
        return "SECTION_NUMBER_ONLY"

    readable_letters = re.findall(
        r"[a-z\u3400-\u9fff\u3040-\u30ff\uac00-\ud7af]",
        normalised,
    )

    if (
        len(normalised) >= 3
        and len(readable_letters) / max(len(normalised), 1) < 0.35
    ):
        return "UNREADABLE_OCR_FRAGMENT"

    if normalised in {
        "activities of",
        "liabilities of",
        "capital share of",
        "payable in",
        "receivable from",
        "during the year",
        "as at",
        "was",
        "were",
        "of",
        "and",
    }:
        return "KNOWN_EXTRACTION_FRAGMENT"

    return None


def signature_eligible(
    row: pd.Series,
) -> bool:
    exception_type = safe_text(
        row.get("exception_type")
    )

    if exception_type == "ACCOUNT_MAPPING":
        source_label = (
            safe_text(
                row.get(
                    "normalised_source_account_label"
                )
            )
            or normalise_account_label(
                row.get("source_account_label")
            )
        )

        if source_label is None:
            return False

        if (
            ENABLE_PRE_AI_FRAGMENT_EXCLUSION
            and pre_ai_fragment_reason(
                row.get("source_account_label")
            )
            is not None
        ):
            return False

        return True

    if exception_type in {
        "FACT_QC",
        "VALUE_ANOMALY",
    }:
        return any(
            safe_text(row.get(column)) is not None
            for column in [
                "standard_concept",
                "source_account_label",
                "filing_id",
                "source_text",
            ]
        )

    return False


def current_processing_key(
    exception_type: str,
) -> str:
    """Key for one Azure processing generation."""
    deployment_label = (
        globals().get(
            "AZURE_OPENAI_DEPLOYMENT"
        )
        or "missing_deployment"
    )

    schema_name = {
        "ACCOUNT_MAPPING":
            "AccountMappingResponse",
        "FACT_QC":
            "FactQCResponse",
        "VALUE_ANOMALY":
            "ValueAnomalyResponse",
    }.get(
        exception_type,
        "UnknownSchema",
    )

    return stable_hash(
        "||".join(
            [
                AI_PROCESSING_GENERATION,
                "Azure OpenAI",
                safe_text(deployment_label) or "",
                schema_name,
                exception_type,
            ]
        )
    )


def strip_reporting_period_suffix(
    value: Any,
) -> Optional[str]:
    text = safe_text(value)

    if text is None:
        return None

    result = unicodedata.normalize(
        "NFKC",
        text,
    ).strip()

    for pattern in REPORTING_PERIOD_SUFFIX_PATTERNS:
        result = re.sub(
            pattern,
            "",
            result,
            flags=re.IGNORECASE,
        ).strip()

    return result or None


def remove_roman_note_tokens(
    value: Any,
) -> Optional[str]:
    text = safe_text(value)

    if text is None:
        return None

    result = re.sub(
        ROMAN_NOTE_TOKEN_PATTERN,
        " ",
        unicodedata.normalize(
            "NFKC",
            text,
        ).casefold(),
        flags=re.IGNORECASE,
    )

    return re.sub(
        r"\s+",
        " ",
        result,
    ).strip() or None


def canonicalise_source_label(
    value: Any,
) -> Optional[str]:
    text = strip_reporting_period_suffix(value)

    if text is None:
        return None

    text = remove_roman_note_tokens(text)

    if text is None:
        return None

    text = re.sub(
        r"(^|\s)[一二三四五六七八九十]+(?=\s|$)",
        " ",
        text,
    )

    text = re.sub(
        r"\b(rmb|cny|usd|eur|jpy|krw|hkd|aud|gbp|"
        r"yuan|share|unit)\b",
        " ",
        text,
    )

    text = re.sub(r"\b\d+\b", " ", text)
    text = re.sub(
        r"[^0-9a-z\u3400-\u9fff]+",
        " ",
        text,
    )
    text = re.sub(r"\s+", " ", text).strip()

    return text or None


def normalised_cluster_label(
    row: pd.Series,
) -> Optional[str]:
    value = (
        safe_text(
            row.get(
                "normalised_source_account_label"
            )
        )
        or row.get("source_account_label")
    )

    return canonicalise_source_label(value)


def deterministic_cluster_key(
    row: pd.Series,
) -> str:
    payload = [
        safe_text(
            row.get("exception_type")
        )
        or "",
        normalised_cluster_label(row)
        or "",
        safe_text(
            row.get("statement_type")
        )
        or "",
        safe_text(
            row.get("reporting_scope")
        )
        or "",
        normalise_identifier(
            row.get("standard_concept")
        )
        or "",
    ]

    return stable_hash(
        "||".join(payload)
    )


ai_exception_signature_queue_df[
    "ai_eligible"
] = [
    signature_eligible(row)
    for _, row in (
        ai_exception_signature_queue_df
        .iterrows()
    )
]

ai_exception_signature_queue_df[
    "processing_key"
] = [
    current_processing_key(
        safe_text(
            row.get("exception_type")
        )
        or "UNKNOWN"
    )
    for _, row in (
        ai_exception_signature_queue_df
        .iterrows()
    )
]

ai_exception_signature_queue_df[
    "normalised_cluster_label"
] = [
    normalised_cluster_label(row)
    for _, row in (
        ai_exception_signature_queue_df
        .iterrows()
    )
]

ai_exception_signature_queue_df[
    "deterministic_cluster_id"
] = [
    deterministic_cluster_key(row)
    for _, row in (
        ai_exception_signature_queue_df
        .iterrows()
    )
]

ai_exception_signature_queue_df[
    "priority_score"
] = (
    np.log1p(
        safe_numeric(
            ai_exception_signature_queue_df[
                "raw_row_count"
            ]
        ).fillna(0)
    )
    + 0.50
    * np.log1p(
        safe_numeric(
            ai_exception_signature_queue_df[
                "unique_issuer_count"
            ]
        ).fillna(0)
    )
    + 0.25
    * np.log1p(
        safe_numeric(
            ai_exception_signature_queue_df[
                "unique_filing_count"
            ]
        ).fillna(0)
    )
    + ai_exception_signature_queue_df[
        "has_source_evidence"
    ].astype(int)
)


COMPLETION_REGISTRY_COLUMNS = [
    "issue_signature",
    "exception_type",
    "processing_key",
    "processing_generation",
    "provider",
    "deployment",
    "model",
    "schema_name",
    "prompt_hash",
    "completed_at_utc",
    "ai_decision",
    "ai_confidence",
    "acceptance_status",
    "publication_status",
    "completion_status",
    "completion_method",
    "prototype_issue_signature",
    "cluster_id",
    "cluster_similarity_score",
]

completion_registry_path = Path(
    globals().get(
        "COMPLETION_REGISTRY_PATH",
        BLOCK_9_OUTPUT_DIR
        / "completed_ai_signatures.parquet",
    )
)

if completion_registry_path.exists():
    try:
        completed_ai_signatures_prior_df = (
            pd.read_parquet(
                completion_registry_path
            )
        )
    except Exception as exc:
        print(
            "Completion registry could not "
            "be read; starting empty:",
            repr(exc),
        )

        completed_ai_signatures_prior_df = (
            pd.DataFrame(
                columns=(
                    COMPLETION_REGISTRY_COLUMNS
                )
            )
        )
else:
    completed_ai_signatures_prior_df = (
        pd.DataFrame(
            columns=COMPLETION_REGISTRY_COLUMNS
        )
    )

for column in COMPLETION_REGISTRY_COLUMNS:
    if (
        column
        not in completed_ai_signatures_prior_df
    ):
        completed_ai_signatures_prior_df[
            column
        ] = pd.NA


successful_completion_keys = set()

if (
    SKIP_COMPLETED_SIGNATURES
    and not completed_ai_signatures_prior_df.empty
):
    completed_successfully = (
        completed_ai_signatures_prior_df.loc[
            completed_ai_signatures_prior_df[
                "completion_status"
            ].isin(
                [
                    "SUCCEEDED",
                    "CACHE_HIT",
                    "PROPAGATED",
                ]
            )
        ]
    )

    successful_completion_keys = {
        (
            str(signature),
            str(processing_key),
        )
        for signature, processing_key in zip(
            completed_successfully[
                "issue_signature"
            ],
            completed_successfully[
                "processing_key"
            ],
        )
        if (
            pd.notna(signature)
            and pd.notna(processing_key)
        )
    }


ai_exception_signature_queue_df[
    "already_completed"
] = [
    (
        str(signature),
        str(processing_key),
    )
    in successful_completion_keys
    for signature, processing_key in zip(
        ai_exception_signature_queue_df[
            "issue_signature"
        ],
        ai_exception_signature_queue_df[
            "processing_key"
        ],
    )
]


eligible_signatures_df = (
    ai_exception_signature_queue_df.loc[
        ai_exception_signature_queue_df[
            "ai_eligible"
        ].fillna(False)
    ]
    .copy()
)

remaining_ai_work_queue_df = (
    eligible_signatures_df.loc[
        ~eligible_signatures_df[
            "already_completed"
        ].fillna(False)
    ]
    .copy()
)


cluster_sizes = (
    remaining_ai_work_queue_df.groupby(
        "deterministic_cluster_id",
        dropna=False,
    )["issue_signature"]
    .size()
    .rename("cluster_size")
)

remaining_ai_work_queue_df = (
    remaining_ai_work_queue_df.merge(
        cluster_sizes,
        on="deterministic_cluster_id",
        how="left",
    )
)

remaining_ai_work_queue_df[
    "required_cluster_prototypes"
] = np.where(
    remaining_ai_work_queue_df[
        "cluster_size"
    ]
    >= LARGE_CLUSTER_MEMBER_THRESHOLD,
    LARGE_CLUSTER_REQUIRED_PROTOTYPES,
    SMALL_CLUSTER_REQUIRED_PROTOTYPES,
)


prototype_ranked_df = (
    remaining_ai_work_queue_df.sort_values(
        [
            "deterministic_cluster_id",
            "priority_score",
            "has_source_evidence",
            "raw_row_count",
            "issue_signature",
        ],
        ascending=[
            True,
            False,
            False,
            False,
            True,
        ],
        kind="mergesort",
    )
    .copy()
)

prototype_ranked_df[
    "prototype_rank_within_cluster"
] = (
    prototype_ranked_df.groupby(
        "deterministic_cluster_id"
    ).cumcount()
    + 1
)

prototype_candidates_df = (
    prototype_ranked_df.loc[
        prototype_ranked_df[
            "prototype_rank_within_cluster"
        ]
        <= prototype_ranked_df[
            "required_cluster_prototypes"
        ]
    ]
    .sort_values(
        [
            "priority_score",
            "cluster_size",
            "raw_row_count",
            "issue_signature",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# Final fresh-batch selection happens in Step 12,
# after exact prompt hashes and cache keys can be built.
selected_ai_exceptions_df = (
    prototype_candidates_df.iloc[0:0]
    .copy()
)

cached_but_unregistered_df = (
    prototype_candidates_df.iloc[0:0]
    .copy()
)


ai_similarity_clusters_df = (
    prototype_ranked_df[
        [
            "issue_signature",
            "exception_type",
            "deterministic_cluster_id",
            "normalised_cluster_label",
            "cluster_size",
            "required_cluster_prototypes",
            "prototype_rank_within_cluster",
            "statement_type",
            "reporting_scope",
            "standard_concept",
            "priority_score",
        ]
    ]
    .copy()
)


all_task_types = sorted(
    set(
        ai_exception_signature_queue_df[
            "exception_type"
        ]
        .dropna()
        .astype(str)
    )
    | {
        "ACCOUNT_MAPPING",
        "FACT_QC",
        "VALUE_ANOMY",
    }
)

task_funnel_rows = []

for task_type in all_task_types:
    task_all = (
        ai_exception_signature_queue_df.loc[
            ai_exception_signature_queue_df[
                "exception_type"
            ].eq(task_type)
        ]
    )

    task_eligible = (
        eligible_signatures_df.loc[
            eligible_signatures_df[
                "exception_type"
            ].eq(task_type)
        ]
    )

    task_remaining = (
        remaining_ai_work_queue_df.loc[
            remaining_ai_work_queue_df[
                "exception_type"
            ].eq(task_type)
        ]
    )

    task_prototypes = (
        prototype_candidates_df.loc[
            prototype_candidates_df[
                "exception_type"
            ].eq(task_type)
        ]
    )

    task_funnel_rows.append(
        {
            "exception_type":
                task_type,
            "representative_signatures":
                len(task_all),
            "ai_eligible_signatures":
                len(task_eligible),
            "already_completed_signatures":
                int(
                    task_eligible[
                        "already_completed"
                    ]
                    .fillna(False)
                    .sum()
                ),
            "remaining_signatures":
                len(task_remaining),
            "deterministic_clusters":
                int(
                    task_remaining[
                        "deterministic_cluster_id"
                    ].nunique()
                ),
            "prototype_candidates":
                len(task_prototypes),
            "cached_but_unregistered":
                0,
            "selected_fresh_azure_requests":
                0,
        }
    )

ai_task_funnel_df = pd.DataFrame(
    task_funnel_rows
)


print(
    "AI-eligible signatures:",
    f"{len(eligible_signatures_df):,}",
)

print(
    "Previously completed for this generation:",
    f"{int(eligible_signatures_df['already_completed'].sum()):,}",
)

print(
    "Remaining eligible signatures:",
    f"{len(remaining_ai_work_queue_df):,}",
)

print(
    "Prototype candidates before cache filtering:",
    f"{len(prototype_candidates_df):,}",
)

display(ai_task_funnel_df)


AI-eligible signatures: 59,570
Previously completed for this generation: 59,561
Remaining eligible signatures: 9
Prototype candidates before cache filtering: 1


,exception_type,representative_signatures,ai_eligible_signatures,already_completed_signatures,remaining_signatures,deterministic_clusters,prototype_candidates,cached_but_unregistered,selected_fresh_azure_requests
0,ACCOUNT_MAPPING,69873,59570,59561,9,1,1,0,0
1,FACT_QC,0,0,0,0,0,0,0,0
2,VALUE_ANOMY,0,0,0,0,0,0,0,0


In [11]:
# ============================================================
# 11. EVIDENCE, CANDIDATE RETRIEVAL AND STRICT PROMPTS
# ============================================================

from difflib import SequenceMatcher


def retrieve_evidence_context(row: pd.Series) -> str:
    parts = []

    for label, col in [
        ("Issuer", "global_issuer_id"),
        ("Region", "region"),
        ("Source account label", "source_account_label"),
        ("Existing standard concept", "standard_concept"),
        ("Reported value", "reported_value_numeric"),
        ("Currency", "currency"),
        ("Unit scale", "unit_scale"),
        ("Statement type", "statement_type"),
        ("Reporting scope", "reporting_scope"),
        ("Period end", "period_end"),
        ("Exception code", "exception_code"),
        ("QC rule", "qc_rule_id"),
        ("QC failure type", "qc_failure_type"),
        ("Anomaly type", "anomaly_type"),
        ("Relative difference", "relative_difference"),
        ("Represented raw rows", "raw_row_count"),
        ("Unique issuers", "unique_issuer_count"),
        ("Unique filings", "unique_filing_count"),
    ]:
        value = row.get(col)

        if value is not None and not pd.isna(value):
            parts.append(f"{label}: {value}")

    source_text = safe_text(row.get("source_text"))

    if source_text:
        parts.append(
            "Source evidence:\n"
            + source_text[:MAX_SOURCE_TEXT_CHARACTERS]
        )

    return "\n".join(parts)


def concept_display_text(value: Any) -> str:
    text = normalise_identifier(value)
    return text.replace("_", " ") if text else ""


def lexical_similarity(left: Any, right: Any) -> float:
    left_text = normalise_account_label(left) or ""
    right_text = normalise_account_label(right) or ""

    left_text = left_text.replace("_", " ").casefold()
    right_text = right_text.replace("_", " ").casefold()

    if not left_text or not right_text:
        return 0.0

    return SequenceMatcher(
        None,
        left_text,
        right_text,
    ).ratio()


canonical_concepts = sorted(
    {
        normalise_identifier(value)
        for value in regional_source_records_df[
            "standard_concept"
        ].dropna()
        if normalise_identifier(value)
    }
)

concepts_by_statement = (
    regional_source_records_df[
        ["statement_type", "standard_concept"]
    ]
    .dropna()
    .assign(
        standard_concept=lambda frame: (
            frame["standard_concept"]
            .map(normalise_identifier)
        )
    )
    .dropna()
    .groupby("statement_type")["standard_concept"]
    .apply(lambda values: sorted(set(values)))
    .to_dict()
)

# Build the lookup locally because the consolidated regional table does not
# necessarily persist this derived normalised-label column.
label_concept_source_df = (
    regional_source_records_df[
        [
            "source_account_label",
            "standard_concept",
        ]
    ]
    .copy()
)

label_concept_source_df[
    "normalised_source_account_label"
] = (
    label_concept_source_df[
        "source_account_label"
    ]
    .map(normalise_account_label)
)

label_concept_source_df[
    "standard_concept"
] = (
    label_concept_source_df[
        "standard_concept"
    ]
    .map(normalise_identifier)
)

concepts_by_normalised_label = (
    label_concept_source_df[
        [
            "normalised_source_account_label",
            "standard_concept",
        ]
    ]
    .dropna()
    .groupby(
        "normalised_source_account_label"
    )["standard_concept"]
    .apply(
        lambda values: sorted(set(values))
    )
    .to_dict()
)


def retrieve_canonical_candidates(
    row: pd.Series,
) -> list[str]:
    existing = normalise_identifier(
        row.get("standard_concept")
    )

    normalised_label = (
        safe_text(
            row.get(
                "normalised_source_account_label"
            )
        )
        or normalise_account_label(
            row.get("source_account_label")
        )
    )

    source_label = (
        safe_text(row.get("source_account_label"))
        or normalised_label
        or ""
    )

    deterministic_concept = (
        deterministic_phrase_mapping(
            source_label
        )
    )

    statement_type = safe_text(
        row.get("statement_type")
    )

    ranked = []
    seen = set()

    def add_candidate(
        concept: Any,
        source_rank: int,
        similarity: float,
    ) -> None:
        concept_id = normalise_identifier(concept)

        if not concept_id or concept_id in seen:
            return

        seen.add(concept_id)
        ranked.append(
            (
                source_rank,
                -float(similarity),
                concept_id,
            )
        )

    if existing:
        add_candidate(existing, 0, 1.0)

    for concept in concepts_by_normalised_label.get(
        normalised_label,
        [],
    ):
        add_candidate(concept, 1, 1.0)

    for concept in concepts_by_statement.get(
        statement_type,
        [],
    ):
        similarity = lexical_similarity(
            source_label,
            concept_display_text(concept),
        )

        if similarity >= MIN_CANDIDATE_SIMILARITY:
            add_candidate(
                concept,
                2,
                similarity,
            )

    if deterministic_concept is not None:
        add_candidate(
            deterministic_concept,
            source_rank=0,
            similarity=1.00,
        )

    for concept in candidate_recall_hints(
        source_label
    ):
        add_candidate(
            concept,
            source_rank=2,
            similarity=0.95,
        )

    if len(ranked) < MAX_CANONICAL_CANDIDATES_PER_PROMPT:
        global_scored = sorted(
            (
                lexical_similarity(
                    source_label,
                    concept_display_text(concept),
                ),
                concept,
            )
            for concept in canonical_concepts
            if concept not in seen
        )

        for similarity, concept in reversed(
            global_scored
        ):
            if (
                similarity < MIN_CANDIDATE_SIMILARITY
                and ranked
            ):
                break

            add_candidate(
                concept,
                3,
                similarity,
            )

            if (
                len(ranked)
                >= MAX_CANONICAL_CANDIDATES_PER_PROMPT
            ):
                break

    ranked = sorted(ranked)

    return [
        concept
        for _, _, concept in ranked[
            :MAX_CANONICAL_CANDIDATES_PER_PROMPT
        ]
    ]


# ------------------------------------------------------------
# Persistent accepted-synonym registry
# ------------------------------------------------------------

accepted_synonym_registry_path = Path(
    globals().get(
        "ACCEPTED_SYNONYM_REGISTRY_PATH",
        BLOCK_9_OUTPUT_DIR
        / "accepted_synonym_registry.parquet",
    )
)

ACCEPTED_SYNONYM_REGISTRY_COLUMNS = [
    "normalised_source_account_label",
    "standard_concept",
    "statement_type",
    "reporting_scope",
    "accepted_observations",
    "unique_issuers",
    "unique_filings",
    "first_accepted_at_utc",
    "last_accepted_at_utc",
    "registry_generation",
]

if accepted_synonym_registry_path.exists():
    try:
        accepted_synonym_registry_prior_df = (
            pd.read_parquet(
                accepted_synonym_registry_path
            )
        )
    except Exception as exc:
        print(
            "Accepted synonym registry could not be read; "
            "starting empty:",
            repr(exc),
        )
        accepted_synonym_registry_prior_df = pd.DataFrame(
            columns=ACCEPTED_SYNONYM_REGISTRY_COLUMNS
        )
else:
    accepted_synonym_registry_prior_df = pd.DataFrame(
        columns=ACCEPTED_SYNONYM_REGISTRY_COLUMNS
    )

for column in ACCEPTED_SYNONYM_REGISTRY_COLUMNS:
    if column not in accepted_synonym_registry_prior_df.columns:
        accepted_synonym_registry_prior_df[column] = pd.NA

trusted_historical_synonym_pairs = set()

if not accepted_synonym_registry_prior_df.empty:
    trusted_registry = accepted_synonym_registry_prior_df.loc[
        (
            safe_numeric(
                accepted_synonym_registry_prior_df[
                    "accepted_observations"
                ]
            ).fillna(0)
            >= MIN_ACCEPTED_SYNONYM_OBSERVATIONS
        )
        & (
            safe_numeric(
                accepted_synonym_registry_prior_df[
                    "unique_issuers"
                ]
            ).fillna(0)
            >= MIN_ACCEPTED_SYNONYM_ISSUERS
        )
        & (
            safe_numeric(
                accepted_synonym_registry_prior_df[
                    "unique_filings"
                ]
            ).fillna(0)
            >= MIN_ACCEPTED_SYNONYM_FILINGS
        )
    ]

    trusted_historical_synonym_pairs = {
        (
            safe_text(label),
            normalise_identifier(concept),
            safe_text(statement_type),
            safe_text(reporting_scope),
        )
        for (
            label,
            concept,
            statement_type,
            reporting_scope,
        ) in zip(
            trusted_registry[
                "normalised_source_account_label"
            ],
            trusted_registry[
                "standard_concept"
            ],
            trusted_registry[
                "statement_type"
            ],
            trusted_registry[
                "reporting_scope"
            ],
        )
        if (
            safe_text(label) is not None
            and normalise_identifier(concept) is not None
        )
    }


# ------------------------------------------------------------
# High-certainty accounting phrase mappings
# ------------------------------------------------------------

HIGH_CERTAINTY_ACCOUNT_PHRASES = {
    "purchase of fixed assets":
        "capital_expenditure",
    "purchase of property plant and equipment":
        "capital_expenditure",
    "acquisition of property plant and equipment":
        "capital_expenditure",
    "additions to property plant and equipment":
        "capital_expenditure",
    "net cash generated from operating activities":
        "operating_cash_flow",
    "net cash inflow generated from operating activities":
        "operating_cash_flow",
    "net cash used in operating activities":
        "operating_cash_flow",
    "total comprehensive income attributable to owners":
        "comprehensive_income_attributable_to_owners",
    "total comprehensive income attributable to minority interests":
        "comprehensive_income_attributable_to_noncontrolling_interests",
    "total comprehensive income attributable to non controlling interests":
        "comprehensive_income_attributable_to_noncontrolling_interests",
    "amortization of intangible assets":
        "amortization_expense",
    "amortisation of intangible assets":
        "amortization_expense",
    "right of use assets depreciation":
        "depreciation_expense",
    "depreciation of right of use assets":
        "depreciation_expense",
    "impairment of intangible assets":
        "impairment_loss",
    "gain on disposal of intangible assets":
        "gain_on_asset_disposal",
    "loss on disposal of intangible assets":
        "loss_on_asset_disposal",
    "selling expenses":
        "selling_expense",
}


def deterministic_phrase_mapping(
    source_label: Any,
) -> Optional[str]:
    normalised = canonicalise_source_label(
        source_label
    )

    if normalised is None:
        return None

    exact = HIGH_CERTAINTY_ACCOUNT_PHRASES.get(
        normalised
    )

    if exact in canonical_concepts:
        return exact

    for phrase, concept in (
        HIGH_CERTAINTY_ACCOUNT_PHRASES.items()
    ):
        if (
            normalised.startswith(
                phrase + " "
            )
            and concept in canonical_concepts
        ):
            return concept

    return None


# ------------------------------------------------------------
# Broader concept search aliases for candidate recall
# ------------------------------------------------------------

CANDIDATE_RECALL_HINTS = {
    "depreciation": [
        "depreciation_expense",
        "depreciation_and_amortization",
        "property_plant_and_equipment",
        "right_of_use_assets",
    ],
    "amortization": [
        "amortization_expense",
        "depreciation_and_amortization",
        "intangible_assets",
    ],
    "intangible": [
        "intangible_assets",
        "amortization_expense",
        "impairment_loss",
    ],
    "provision": [
        "provisions",
        "other_liabilities",
        "impairment_loss",
    ],
    "purchase of fixed assets": [
        "capital_expenditure",
        "purchase_of_property_plant_and_equipment",
        "investing_cash_flow",
    ],
    "property plant and equipment": [
        "property_plant_and_equipment",
        "capital_expenditure",
        "purchase_of_property_plant_and_equipment",
    ],
    "comprehensive income attributable": [
        "comprehensive_income_attributable_to_owners",
        "comprehensive_income_attributable_to_noncontrolling_interests",
        "comprehensive_income",
    ],
    "other current liabilities": [
        "other_current_liabilities",
        "current_liabilities",
        "other_payables",
    ],
    "other non-current liabilities": [
        "other_noncurrent_liabilities",
        "noncurrent_liabilities",
        "other_payables",
    ],
    "interest income": [
        "interest_income",
        "finance_income",
    ],
    "gain on disposal": [
        "gain_on_asset_disposal",
        "other_income",
    ],
    "loss on disposal": [
        "loss_on_asset_disposal",
        "other_expense",
    ],
    "lease liabilities": [
        "current_lease_liabilities",
        "noncurrent_lease_liabilities",
    ],
}


def candidate_recall_hints(
    source_label: Any,
) -> list[str]:
    normalised = (
        normalise_account_label(source_label)
        or ""
    ).casefold()

    hints = []

    for phrase, concepts in CANDIDATE_RECALL_HINTS.items():
        if phrase in normalised:
            hints.extend(concepts)

    return [
        concept
        for concept in hints
        if concept in canonical_concepts
    ]


SYSTEM_PROMPTS = {
    "ACCOUNT_MAPPING": """
You are a conservative financial-statement account-mapping reviewer.
Return only the required structured object.

This is constrained candidate selection, not concept invention.

For MAP:
- choose exactly one supplied candidate_index;
- never invent a canonical concept;
- the candidate must be supported by the evidence and statement-compatible.

For KEEP_SOURCE:
- use when the existing standard concept is already defensible;
- return the candidate_index corresponding to the existing concept.

For REJECT:
- use only when the supplied evidence positively demonstrates that the row is not a valid accounting fact;
- do not reject merely because the label is broad, unfamiliar, truncated, ambiguous or lacks a defensible candidate;
- broad accounts, totals and "other" categories remain valid accounts;
- return candidate_index=null.

For UNRESOLVED:
- use when evidence is insufficient, the label is ambiguous or incomplete, or no supplied candidate is defensible;
- prefer UNRESOLVED over REJECT whenever invalidity is not directly demonstrated;
- return candidate_index=null.

Do not infer facts absent from the evidence.
""".strip(),
    "FACT_QC": """
You are a conservative financial-fact quality-control reviewer.
Return only the required structured object.
KEEP_SOURCE only when evidence supports leaving the source unchanged.
PROPOSE_CORRECTION only when a concrete corrected field is explicitly supported.
REJECT_FACT only when the record is not a valid financial fact.
Use UNRESOLVED whenever evidence is insufficient.
Never invent values, units, currency, sign or statement classification.
""".strip(),
    "VALUE_ANOMALY": """
You are a conservative financial-value anomaly reviewer.
Return only the required structured object.
An unusual value is not automatically wrong.
KEEP_SOURCE only when evidence supports it.
PROPOSE_CORRECTION only for a concrete evidence-based correction.
REJECT_FACT only when the record is invalid.
Otherwise use UNRESOLVED.
Never manufacture a replacement value.
""".strip(),
}


def build_ai_prompt(
    row: pd.Series,
    evidence_text: str,
) -> tuple[str, str, list[str]]:
    exception_type = row["exception_type"]

    candidates = (
        retrieve_canonical_candidates(row)
        if exception_type == "ACCOUNT_MAPPING"
        else []
    )

    candidate_payload = [
        {
            "candidate_index": index,
            "canonical_concept": concept,
        }
        for index, concept in enumerate(candidates)
    ]

    payload = {
        "task": exception_type,
        "issue_signature": row.get(
            "issue_signature"
        ),
        "signature_payload": row.get(
            "signature_payload"
        ),
        "represented_raw_rows": row.get(
            "raw_row_count"
        ),
        "evidence": evidence_text,
        "allowed_canonical_candidates":
            candidate_payload,
        "instructions": [
            "Use only supplied evidence.",
            "Return no prose outside the schema.",
            "Never invent a canonical concept.",
            "For MAP or KEEP_SOURCE, return one valid candidate_index.",
            "For REJECT or UNRESOLVED, return candidate_index=null.",
            "Use UNRESOLVED rather than guessing.",
        ],
    }

    user_prompt = (
        "Review this representative exception:\n\n"
        + json.dumps(
            payload,
            ensure_ascii=False,
            default=str,
            indent=2,
        )
    )

    return (
        SYSTEM_PROMPTS[exception_type],
        user_prompt[:MAX_PROMPT_CHARACTERS],
        candidates,
    )


def resolve_candidate_selection(
    parsed: dict,
    candidates: list[str],
) -> dict:
    resolved = dict(parsed)

    decision = safe_text(
        resolved.get("decision")
    )

    decision = decision.upper() if decision else None

    candidate_index = resolved.get(
        "candidate_index"
    )

    if decision in {"MAP", "KEEP_SOURCE"}:
        valid_index = (
            isinstance(candidate_index, int)
            and 0 <= candidate_index < len(candidates)
        )

        if not valid_index:
            resolved["decision"] = "UNRESOLVED"
            resolved["candidate_index"] = None
            resolved["proposed_standard_concept"] = None
            resolved["reasoning_summary"] = (
                (
                    safe_text(
                        resolved.get(
                            "reasoning_summary"
                        )
                    )
                    or ""
                )
                + " Invalid candidate selection; deterministically converted to UNRESOLVED."
            )
        else:
            resolved[
                "proposed_standard_concept"
            ] = candidates[candidate_index]
    else:
        resolved["candidate_index"] = None
        resolved["proposed_standard_concept"] = None

    return resolved


In [12]:
# ============================================================
# 12. AZURE OPENAI AND PERSISTENT CACHE
# ============================================================

@dataclass
class AzureResult:
    parsed: dict[str, Any]
    deployment: str
    model: str
    prompt_tokens: Optional[int] = None
    completion_tokens: Optional[int] = None
    total_tokens: Optional[int] = None


class AzureOpenAIClient:
    """Single Azure OpenAI structured-output request path."""

    provider_name = "Azure OpenAI"

    @staticmethod
    def _normalise_base_url(
        endpoint: str,
    ) -> str:
        value = (
            str(endpoint)
            .strip()
            .split("?", 1)[0]
            .rstrip("/")
        )

        if not value.startswith(
            ("https://", "http://")
        ):
            raise RuntimeError(
                "AZURE_OPENAI_ENDPOINT must be "
                "a full HTTPS URL."
            )

        match = re.match(
            r"^(https?://[^/]+)",
            value,
            flags=re.IGNORECASE,
        )

        if not match:
            raise RuntimeError(
                "Could not parse "
                "AZURE_OPENAI_ENDPOINT."
            )

        resource_root = (
            match.group(1).rstrip("/")
        )

        hostname = resource_root.lower()

        accepted_hosts = (
            ".openai.azure.com",
            ".services.ai.azure.com",
            ".api.cognitive.microsoft.com",
        )

        if not any(
            suffix in hostname
            for suffix in accepted_hosts
        ):
            raise RuntimeError(
                "AZURE_OPENAI_ENDPOINT must be "
                "an Azure OpenAI or Microsoft "
                "Foundry resource endpoint."
            )

        return (
            resource_root
            + "/openai/v1/"
        )

    def __init__(self) -> None:
        secret_values = {
            "AZURE_OPENAI_ENDPOINT":
                globals().get(
                    "AZURE_OPENAI_ENDPOINT"
                ),
            "AZURE_OPENAI_API_KEY":
                globals().get(
                    "AZURE_OPENAI_API_KEY"
                ),
            "AZURE_OPENAI_DEPLOYMENT":
                globals().get(
                    "AZURE_OPENAI_DEPLOYMENT"
                ),
        }

        missing = [
            name
            for name, value
            in secret_values.items()
            if not value
        ]

        if missing:
            raise RuntimeError(
                "Missing required Colab Secrets: "
                + ", ".join(missing)
            )

        from openai import OpenAI

        self.base_url = (
            self._normalise_base_url(
                secret_values[
                    "AZURE_OPENAI_ENDPOINT"
                ]
            )
        )

        self.client = OpenAI(
            api_key=secret_values[
                "AZURE_OPENAI_API_KEY"
            ],
            base_url=self.base_url,
            timeout=90.0,
            max_retries=0,
        )

        self.deployment_name = str(
            secret_values[
                "AZURE_OPENAI_DEPLOYMENT"
            ]
        ).strip()

        self.model_name = (
            self.deployment_name
        )

    def generate_structured_response(
        self,
        *,
        exception_type: str,
        system_prompt: str,
        user_prompt: str,
        schema: Type[BaseModel],
    ) -> AzureResult:
        response = (
            self.client.beta.chat.completions.parse(
                model=self.deployment_name,
                messages=[
                    {
                        "role": "system",
                        "content": system_prompt,
                    },
                    {
                        "role": "user",
                        "content": user_prompt,
                    },
                ],
                response_format=schema,
                reasoning_effort="low",
                max_completion_tokens=4096,
            )
        )

        if not response.choices:
            raise RuntimeError(
                "Azure returned no completion choices."
            )

        parsed_object = (
            response.choices[0]
            .message
            .parsed
        )

        if parsed_object is None:
            raise RuntimeError(
                "Azure returned no parsed "
                "structured response."
            )

        usage = getattr(
            response,
            "usage",
            None,
        )

        return AzureResult(
            parsed=parsed_object.model_dump(),
            deployment=self.deployment_name,
            model=self.model_name,
            prompt_tokens=getattr(
                usage,
                "prompt_tokens",
                None,
            ),
            completion_tokens=getattr(
                usage,
                "completion_tokens",
                None,
            ),
            total_tokens=getattr(
                usage,
                "total_tokens",
                None,
            ),
        )


class SQLiteAICache:
    """Persistent Azure response cache."""

    def __init__(
        self,
        path: Path,
    ) -> None:
        self.path = Path(path)

        self.path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        with sqlite3.connect(
            self.path
        ) as connection:
            connection.execute(
                """
                CREATE TABLE IF NOT EXISTS ai_cache (
                    cache_key TEXT PRIMARY KEY,
                    issue_signature TEXT NOT NULL,
                    provider TEXT NOT NULL,
                    deployment TEXT,
                    model TEXT NOT NULL,
                    schema_name TEXT NOT NULL,
                    prompt_hash TEXT NOT NULL,
                    response_json TEXT NOT NULL,
                    created_at_utc TEXT NOT NULL
                )
                """
            )

    def contains(
        self,
        cache_key: str,
    ) -> bool:
        with sqlite3.connect(
            self.path
        ) as connection:
            row = connection.execute(
                """
                SELECT 1
                FROM ai_cache
                WHERE cache_key = ?
                LIMIT 1
                """,
                (cache_key,),
            ).fetchone()

        return row is not None

    def get(
        self,
        cache_key: str,
    ) -> Optional[dict[str, Any]]:
        with sqlite3.connect(
            self.path
        ) as connection:
            row = connection.execute(
                """
                SELECT response_json
                FROM ai_cache
                WHERE cache_key = ?
                """,
                (cache_key,),
            ).fetchone()

        return (
            json.loads(row[0])
            if row
            else None
        )

    def put(
        self,
        *,
        cache_key: str,
        issue_signature: str,
        schema_name: str,
        prompt_hash: str,
        response: dict[str, Any],
    ) -> None:
        with sqlite3.connect(
            self.path
        ) as connection:
            connection.execute(
                """
                INSERT OR REPLACE INTO ai_cache (
                    cache_key,
                    issue_signature,
                    provider,
                    deployment,
                    model,
                    schema_name,
                    prompt_hash,
                    response_json,
                    created_at_utc
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
                """,
                (
                    cache_key,
                    issue_signature,
                    "Azure OpenAI",
                    azure_client.deployment_name,
                    azure_client.model_name,
                    schema_name,
                    prompt_hash,
                    json.dumps(
                        response,
                        ensure_ascii=False,
                    ),
                    utc_now().isoformat(),
                ),
            )


def make_cache_key(
    issue_signature: str,
    schema: Type[BaseModel],
    prompt_hash: str,
) -> str:
    return stable_hash(
        "|".join(
            [
                issue_signature,
                "Azure OpenAI",
                azure_client.deployment_name,
                azure_client.model_name,
                schema.__name__,
                prompt_hash,
            ]
        )
    )


azure_client: Optional[
    AzureOpenAIClient
] = None

provider_validation_df = pd.DataFrame()

# ------------------------------------------------------------
# Validate that the live cache is local and healthy
# ------------------------------------------------------------

if ENABLE_CACHE:
    if Path(AI_CACHE_PATH) != Path(
        LOCAL_AI_CACHE_PATH
    ):
        raise RuntimeError(
            "Block 9 must use the local runtime SQLite cache. "
            f"Current AI_CACHE_PATH: {AI_CACHE_PATH}"
        )

    if Path(AI_CACHE_PATH).exists():
        cache_integrity = (
            sqlite_integrity_check(
                Path(AI_CACHE_PATH)
            )
        )

        if cache_integrity != "ok":
            raise RuntimeError(
                "The local SQLite cache is not healthy: "
                + cache_integrity
            )

print(
    "Step 12 live cache:",
    AI_CACHE_PATH,
)

ai_cache = (
    SQLiteAICache(AI_CACHE_PATH)
    if ENABLE_CACHE
    else None
)

if AI_ENABLED:
    azure_client = AzureOpenAIClient()

    print(
        "Azure OpenAI initialised:",
        azure_client.deployment_name,
    )

    print(
        "Fresh-batch selection will exclude "
        "exact cache matches."
    )


# ------------------------------------------------------------
# Exact cache-aware fresh-batch selection
# ------------------------------------------------------------

fresh_rows = []
cached_rows = []
fresh_counts_by_task = {}

if not prototype_candidates_df.empty:
    for _, row in (
        prototype_candidates_df.iterrows()
    ):
        exception_type = (
            row["exception_type"]
        )

        current_count = (
            fresh_counts_by_task.get(
                exception_type,
                0,
            )
        )

        if (
            current_count
            >= MAX_AI_RECORDS_PER_TASK
        ):
            continue

        evidence_text = (
            retrieve_evidence_context(row)
        )

        (
            system_prompt,
            user_prompt,
            candidates,
        ) = build_ai_prompt(
            row,
            evidence_text,
        )

        prompt_hash = stable_hash(
            system_prompt
            + "\n\n"
            + user_prompt
        )

        schema = RESPONSE_SCHEMA_BY_TASK[
            exception_type
        ]

        if (
            AI_ENABLED
            and azure_client is not None
        ):
            cache_key = make_cache_key(
                row["issue_signature"],
                schema,
                prompt_hash,
            )
        else:
            cache_key = stable_hash(
                "|".join(
                    [
                        row["issue_signature"],
                        schema.__name__,
                        prompt_hash,
                        "dry_run",
                    ]
                )
            )

        enriched_row = row.copy()

        enriched_row[
            "selection_prompt_hash"
        ] = prompt_hash

        enriched_row[
            "selection_cache_key"
        ] = cache_key

        enriched_row[
            "selection_candidate_concepts_json"
        ] = json.dumps(
            candidates,
            ensure_ascii=False,
        )

        already_cached = (
            bool(
                ai_cache.contains(cache_key)
            )
            if (
                AI_ENABLED
                and ai_cache is not None
            )
            else False
        )

        enriched_row[
            "already_cached_exact_prompt"
        ] = already_cached

        if already_cached:
            cached_rows.append(
                enriched_row
            )
            continue

        fresh_rows.append(
            enriched_row
        )

        fresh_counts_by_task[
            exception_type
        ] = current_count + 1


selected_ai_exceptions_df = pd.DataFrame(
    fresh_rows
)

cached_but_unregistered_df = pd.DataFrame(
    cached_rows
)

if selected_ai_exceptions_df.empty:
    selected_ai_exceptions_df = (
        prototype_candidates_df.iloc[0:0]
        .copy()
    )

if cached_but_unregistered_df.empty:
    cached_but_unregistered_df = (
        prototype_candidates_df.iloc[0:0]
        .copy()
    )


# Cache matches that are absent from the completion registry are
# interruption-recovery work. Replay them locally before executing
# the newly selected Azure requests.
recovery_ai_exceptions_df = (
    cached_but_unregistered_df.copy()
)

execution_ai_exceptions_df = (
    pd.concat(
        [
            recovery_ai_exceptions_df,
            selected_ai_exceptions_df,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["issue_signature"],
        keep="first",
    )
    .reset_index(drop=True)
)


ai_cluster_prototypes_df = (
    execution_ai_exceptions_df[
        [
            column
            for column in [
                "issue_signature",
                "exception_type",
                "deterministic_cluster_id",
                "normalised_cluster_label",
                "cluster_size",
                "required_cluster_prototypes",
                "prototype_rank_within_cluster",
                "statement_type",
                "reporting_scope",
                "standard_concept",
                "priority_score",
                "selection_prompt_hash",
                "selection_cache_key",
            ]
            if column
            in selected_ai_exceptions_df.columns
        ]
    ]
    .copy()
)


for task_type in (
    ai_task_funnel_df[
        "exception_type"
    ].tolist()
):
    cached_count = int(
        (
            cached_but_unregistered_df
            .get(
                "exception_type",
                pd.Series(dtype=str),
            )
            .eq(task_type)
        )
        .sum()
    )

    selected_count = int(
        (
            selected_ai_exceptions_df
            .get(
                "exception_type",
                pd.Series(dtype=str),
            )
            .eq(task_type)
        )
        .sum()
    )

    ai_task_funnel_df.loc[
        ai_task_funnel_df[
            "exception_type"
        ].eq(task_type),
        "cached_but_unregistered",
    ] = cached_count

    ai_task_funnel_df.loc[
        ai_task_funnel_df[
            "exception_type"
        ].eq(task_type),
        "selected_fresh_azure_requests",
    ] = selected_count


print(
    "Exact cached prompts excluded before "
    "batch selection:",
    len(cached_but_unregistered_df),
)

print(
    "Selected fresh Azure requests:",
    len(selected_ai_exceptions_df),
)

print(
    "Interrupted requests available for cache replay:",
    len(recovery_ai_exceptions_df),
)

print(
    "Total requests to process this execution:",
    len(execution_ai_exceptions_df),
)

display(ai_task_funnel_df)


Step 12 live cache: /content/block_9_ai_cache.sqlite
Azure OpenAI initialised: gpt-5-mini
Fresh-batch selection will exclude exact cache matches.
Exact cached prompts excluded before batch selection: 1
Selected fresh Azure requests: 0
Interrupted requests available for cache replay: 1
Total requests to process this execution: 1


,exception_type,representative_signatures,ai_eligible_signatures,already_completed_signatures,remaining_signatures,deterministic_clusters,prototype_candidates,cached_but_unregistered,selected_fresh_azure_requests
0,ACCOUNT_MAPPING,69873,59570,59561,9,1,1,1,0
1,FACT_QC,0,0,0,0,0,0,0,0
2,VALUE_ANOMY,0,0,0,0,0,0,0,0


# ^ STEP 12 ERROR NOTE:
# If Azure reports missing Colab secrets, rerun Step 2 then Step 12.

In [13]:
# ============================================================
# 13. CONTROLLED AI EXECUTION, CACHE AND REQUEST AUDIT
# ============================================================

def classify_ai_error(error: Exception) -> str:
    text = repr(error).lower()

    if any(
        token in text
        for token in [
            "429",
            "resource_exhausted",
            "rate limit",
            "quota",
        ]
    ):
        return "QUOTA_OR_RATE_LIMIT"

    if any(
        token in text
        for token in [
            "401",
            "403",
            "unauthorized",
            "forbidden",
            "authentication",
        ]
    ):
        return "AUTHENTICATION"

    if any(
        token in text
        for token in [
            "schema",
            "validation",
            "parsed",
            "json",
            "no completion choices",
            "no parsed structured response",
            "empty structured response",
        ]
    ):
        return "SCHEMA_OR_PARSING"

    if any(
        token in text
        for token in [
            "timeout",
            "connection",
            "dns",
            "reachable",
        ]
    ):
        return "CONNECTION"

    return "OTHER"


def estimate_cost_usd(
    provider_name: str,
    prompt_tokens: Any,
    completion_tokens: Any,
) -> float:
    prompt_count = float(
        prompt_tokens or 0
    )

    completion_count = float(
        completion_tokens or 0
    )

    input_rate = float(
        globals().get(
            "AZURE_INPUT_COST_PER_MILLION_USD",
            0.25,
        )
    )

    output_rate = float(
        globals().get(
            "AZURE_OUTPUT_COST_PER_MILLION_USD",
            2.00,
        )
    )

    return (
        prompt_count * input_rate
        + completion_count * output_rate
    ) / 1_000_000


def proposal_row_from_result(
    row: pd.Series,
    parsed: dict,
    provider_name: str,
    deployment_name: str,
    model_name: str,
    prompt_hash: str,
    cache_hit: bool,
    processed_at_utc: Any,
    candidates: list[str],
) -> dict:
    return {
        "review_record_id":
            row["review_record_id"],
        "issue_signature":
            row["issue_signature"],
        "exception_type":
            row["exception_type"],
        "region":
            row.get("region"),
        "upstream_block":
            row.get("upstream_block"),
        "source_table":
            row.get("source_table"),
        "represented_raw_rows":
            row.get("raw_row_count"),
        "unique_issuer_count":
            row.get("unique_issuer_count"),
        "unique_filing_count":
            row.get("unique_filing_count"),
        "model_provider":
            provider_name,
        "model_deployment":
            deployment_name,
        "model_name":
            model_name,
        "model_response":
            json.dumps(
                parsed,
                ensure_ascii=False,
            ),
        "candidate_count":
            len(candidates),
        "candidate_concepts_json":
            json.dumps(
                candidates,
                ensure_ascii=False,
            ),
        "selected_candidate_index":
            parsed.get("candidate_index"),
        "ai_decision":
            parsed.get("decision"),
        "ai_confidence":
            parsed.get("confidence"),
        "ai_evidence_text":
            parsed.get("evidence_text"),
        "ai_reasoning_summary":
            parsed.get("reasoning_summary"),
        "proposed_standard_concept":
            parsed.get(
                "proposed_standard_concept"
            ),
        "proposed_value":
            parsed.get("proposed_value"),
        "proposed_currency":
            parsed.get("proposed_currency"),
        "proposed_unit_scale":
            parsed.get("proposed_unit_scale"),
        "proposed_sign_multiplier":
            parsed.get(
                "proposed_sign_multiplier"
            ),
        "proposed_statement_type":
            parsed.get("statement_type"),
        "proposed_reporting_scope":
            parsed.get("reporting_scope"),
        "prompt_hash":
            prompt_hash,
        "cache_hit":
            cache_hit,
        "processed_at_utc":
            processed_at_utc,
    }


ai_result_rows = []
ai_audit_rows = []
consecutive_quota_errors = 0
pipeline_stopped_early = False
pipeline_stop_reason = None
estimated_run_cost_usd = 0.0

if (
    AI_ENABLED
    and azure_client is not None
    and not execution_ai_exceptions_df.empty
):
    iterator = execution_ai_exceptions_df.iterrows()

    for _, row in tqdm(
        iterator,
        total=len(execution_ai_exceptions_df),
        desc="AI signature review",
    ):
        exception_type = row["exception_type"]
        schema = RESPONSE_SCHEMA_BY_TASK[
            exception_type
        ]

        evidence_text = retrieve_evidence_context(
            row
        )

        (
            system_prompt,
            user_prompt,
            candidates,
        ) = build_ai_prompt(
            row,
            evidence_text,
        )

        prompt_hash = (
            safe_text(
                row.get(
                    "selection_prompt_hash"
                )
            )
            or stable_hash(
                system_prompt
                + "\n\n"
                + user_prompt
            )
        )

        cache_key = make_cache_key(
            row["issue_signature"],
            schema,
            prompt_hash,
        )

        cached = (
            ai_cache.get(cache_key)
            if ai_cache is not None
            else None
        )

        if cached is not None:
            print(
                "Recovery cache hit after fresh-batch selection: "
                + row["issue_signature"][:12]
            )

            parsed = schema.model_validate(
                cached
            ).model_dump()

            parsed = resolve_candidate_selection(
                parsed,
                candidates,
            )

            ai_result_rows.append(
                proposal_row_from_result(
                    row=row,
                    parsed=parsed,
                    provider_name="Azure OpenAI",
                    deployment_name=azure_client.deployment_name,
                    model_name=azure_client.model_name,
                    prompt_hash=prompt_hash,
                    cache_hit=True,
                    processed_at_utc=utc_now(),
                    candidates=candidates,
                )
            )

            now = utc_now()

            ai_audit_rows.append(
                {
                    "review_record_id":
                        row["review_record_id"],
                    "issue_signature":
                        row["issue_signature"],
                    "exception_type":
                        exception_type,
                    "provider":
                        "Azure OpenAI",
                    "deployment":
                        azure_client.deployment_name,
                    "model":
                        azure_client.model_name,
                    "attempt_number": 0,
                    "retry_count": 0,
                    "prompt_hash": prompt_hash,
                    "status": "CACHE_HIT",
                    "error_category": None,
                    "error_message": None,
                    "started_at_utc": now,
                    "completed_at_utc": now,
                    "latency_seconds": 0.0,
                    "prompt_token_count": 0,
                    "candidate_token_count": 0,
                    "total_token_count": 0,
                    "estimated_cost_usd": 0.0,
                    "cache_hit": True,
                }
            )
            continue

        print(
            "Azure OpenAI"
            + " request: "
            + row["issue_signature"][:12]
        )

        completed = False

        for attempt in range(
            1,
            MAX_RETRIES_PER_RECORD + 1,
        ):
            started = utc_now()
            result = None
            error_category = None
            error_message = None

            try:
                result = azure_client.generate_structured_response(
                    exception_type=exception_type,
                    system_prompt=system_prompt,
                    user_prompt=user_prompt,
                    schema=schema,
                )

                parsed = schema.model_validate(
                    result.parsed
                ).model_dump()

                parsed = resolve_candidate_selection(
                    parsed,
                    candidates,
                )

                cache_payload = {
                    key: value
                    for key, value in parsed.items()
                    if key != "proposed_standard_concept"
                }

                if ai_cache is not None:
                    ai_cache.put(
                        cache_key=cache_key,
                        issue_signature=
                            row["issue_signature"],

                        schema_name=schema.__name__,
                        prompt_hash=prompt_hash,
                        response=cache_payload,
                    )

                status = "SUCCEEDED"
                consecutive_quota_errors = 0
                completed = True

                ai_result_rows.append(
                    proposal_row_from_result(
                        row=row,
                        parsed=parsed,
                        provider_name="Azure OpenAI",
                        deployment_name=result.deployment,
                        model_name=result.model,
                        prompt_hash=prompt_hash,
                        cache_hit=False,
                        processed_at_utc=started,
                        candidates=candidates,
                    )
                )

            except Exception as error:
                status = "FAILED"
                error_message = repr(error)
                error_category = classify_ai_error(
                    error
                )

                if (
                    error_category
                    == "QUOTA_OR_RATE_LIMIT"
                ):
                    consecutive_quota_errors += 1
                else:
                    consecutive_quota_errors = 0

            completed_at = utc_now()

            prompt_tokens = (
                result.prompt_tokens
                if result is not None
                else None
            )

            completion_tokens = (
                result.completion_tokens
                if result is not None
                else None
            )

            total_tokens = (
                result.total_tokens
                if result is not None
                else None
            )

            ai_audit_rows.append(
                {
                    "review_record_id":
                        row["review_record_id"],
                    "issue_signature":
                        row["issue_signature"],
                    "exception_type":
                        exception_type,
                    "provider":
                        "Azure OpenAI",
                    "deployment":
                        azure_client.deployment_name,
                    "model":
                        azure_client.model_name,
                    "attempt_number": attempt,
                    "retry_count": attempt - 1,
                    "prompt_hash": prompt_hash,
                    "status": status,
                    "error_category":
                        error_category,
                    "error_message":
                        error_message,
                    "started_at_utc":
                        started,
                    "completed_at_utc":
                        completed_at,
                    "latency_seconds": (
                        completed_at - started
                    ).total_seconds(),
                    "prompt_token_count":
                        prompt_tokens,
                    "candidate_token_count":
                        completion_tokens,
                    "total_token_count":
                        total_tokens,
                    "estimated_cost_usd":
                        estimate_cost_usd(
                            "Azure OpenAI",
                            prompt_tokens,
                            completion_tokens,
                        ),
                    "cache_hit": False,
                }
            )

            latest_cost = (
                float(
                    ai_audit_rows[-1].get(
                        "estimated_cost_usd",
                        0.0,
                    )
                    or 0.0
                )
                if ai_audit_rows
                else 0.0
            )
            estimated_run_cost_usd += latest_cost

            if (
                estimated_run_cost_usd
                >= float(globals().get("MAX_ESTIMATED_RUN_COST_USD", 25.0))
            ):
                pipeline_stopped_early = True
                pipeline_stop_reason = (
                    "Stopped after reaching the configured "
                    "estimated run-cost ceiling of USD "
                    + f"{float(globals().get("MAX_ESTIMATED_RUN_COST_USD", 25.0)):,.2f}."
                )
                break

            if completed:
                break

            if (
                consecutive_quota_errors
                >= MAX_CONSECUTIVE_QUOTA_ERRORS
            ):
                pipeline_stopped_early = True
                pipeline_stop_reason = (
                    "Stopped after "
                    + str(consecutive_quota_errors)
                    + " consecutive quota/rate-limit errors."
                )
                break

            if error_category in {
                "AUTHENTICATION",
                "SCHEMA_OR_PARSING",
                "CONNECTION",
            }:
                break

            time.sleep(
                RETRY_BASE_SECONDS
                * (2 ** (attempt - 1))
            )

        if pipeline_stopped_early:
            break

        time.sleep(AI_SLEEP_SECONDS)


ai_enrichment_proposals_df = pd.DataFrame(
    ai_result_rows
)

ai_model_audit_log_df = pd.DataFrame(
    ai_audit_rows
)

print(
    "AI proposals:",
    len(ai_enrichment_proposals_df),
)

print(
    "AI audit records:",
    len(ai_model_audit_log_df),
)

print(
    "Estimated run cost (USD):",
    f"{estimated_run_cost_usd:,.4f}",
)

print(
    "Stopped early:",
    pipeline_stopped_early,
)

if pipeline_stop_reason:
    print(pipeline_stop_reason)


AI signature review:   0%|          | 0/1 [00:00<?, ?it/s]

Recovery cache hit after fresh-batch selection: 9bad0e7c5aeb
AI proposals: 1
AI audit records: 1
Estimated run cost (USD): 0.0000
Stopped early: False


In [14]:
# ============================================================
# 14. CHECKPOINT LOCAL AI CACHE TO GOOGLE DRIVE
# ============================================================

if ENABLE_CACHE:
    print(
        "Local cache integrity before checkpoint:",
        sqlite_integrity_check(
            Path(AI_CACHE_PATH)
        ),
    )

    print(
        "Local cached responses:",
        f"{sqlite_cache_row_count(Path(AI_CACHE_PATH)):,}",
    )

    sync_local_ai_cache_to_drive()

Local cache integrity before checkpoint: ok
Local cached responses: 58,962
Local AI cache synchronised safely to Google Drive.
Persistent rows: 58,962


In [15]:
# ============================================================
# 15. STATEMENT–CONCEPT COMPATIBILITY
# ============================================================
STATEMENT_ALIASES = {
    "balance sheet": "BALANCE_SHEET", "statement of financial position": "BALANCE_SHEET",
    "income statement": "INCOME_STATEMENT", "profit and loss": "INCOME_STATEMENT",
    "statement of comprehensive income": "INCOME_STATEMENT",
    "cash flow": "CASH_FLOW", "cash flow statement": "CASH_FLOW",
    "statement of cash flows": "CASH_FLOW", "notes": "NOTES",
}

CONCEPT_ALLOWED_STATEMENTS = {
    "property_plant_equipment": {"BALANCE_SHEET", "NOTES"},
    "inventory": {"BALANCE_SHEET", "NOTES"},
    "accounts_receivable": {"BALANCE_SHEET", "NOTES"},
    "accounts_payable": {"BALANCE_SHEET", "NOTES"},
    "short_term_debt": {"BALANCE_SHEET", "NOTES"},
    "long_term_debt": {"BALANCE_SHEET", "NOTES"},
    "cash_and_cash_equivalents": {"BALANCE_SHEET", "CASH_FLOW", "NOTES"},
    "revenue": {"INCOME_STATEMENT", "NOTES"},
    "cost_of_revenue": {"INCOME_STATEMENT", "NOTES"},
    "gross_profit": {"INCOME_STATEMENT", "NOTES"},
    "operating_income": {"INCOME_STATEMENT", "NOTES"},
    "net_income": {"INCOME_STATEMENT", "NOTES"},
    "depreciation_and_amortisation": {"INCOME_STATEMENT", "CASH_FLOW", "NOTES"},
    "operating_cash_flow": {"CASH_FLOW"},
    "investing_cash_flow": {"CASH_FLOW"},
    "financing_cash_flow": {"CASH_FLOW"},
    "capital_expenditure": {"CASH_FLOW", "NOTES"},
}

STATEMENT_CONCEPT_DENY_PATTERNS = {
    "CASH_FLOW": [r"property.*plant", r"inventory$", r"receivable$", r"payable$"],
    "INCOME_STATEMENT": [r"cash_flow", r"property.*plant", r"total_assets", r"total_liabilities"],
    "BALANCE_SHEET": [r"operating_cash_flow", r"investing_cash_flow", r"financing_cash_flow"],
}


def normalise_statement_type(value: Any) -> Optional[str]:
    text = normalise_identifier(value)
    if text is None: return None
    return STATEMENT_ALIASES.get(text, text.upper().replace(" ", "_"))


def concept_statement_compatible(concept: Any, statement: Any) -> bool:
    concept_norm = normalise_identifier(concept)
    statement_norm = normalise_statement_type(statement)
    if concept_norm is None: return True
    if statement_norm is None: return False
    allowed = CONCEPT_ALLOWED_STATEMENTS.get(concept_norm)
    if allowed is not None and statement_norm not in allowed: return False
    return not any(re.search(pattern, concept_norm) for pattern in STATEMENT_CONCEPT_DENY_PATTERNS.get(statement_norm, []))


In [16]:
# ============================================================
# 16. HISTORICAL CACHE REPUBLICATION UNDER CURRENT POLICY
# ============================================================

def load_all_ai_cache_rows(
    cache_path: Path,
) -> pd.DataFrame:
    """Load the complete SQLite cache without generating Azure requests."""
    if not cache_path.exists():
        return pd.DataFrame()

    with sqlite3.connect(cache_path) as connection:
        frame = pd.read_sql_query(
            """
            SELECT
                cache_key,
                issue_signature,
                provider,
                deployment,
                model,
                schema_name,
                prompt_hash,
                response_json,
                created_at_utc
            FROM ai_cache
            ORDER BY created_at_utc
            """,
            connection,
        )

    if frame.empty:
        return frame

    frame["created_at_utc"] = pd.to_datetime(
        frame["created_at_utc"],
        errors="coerce",
        utc=True,
    )

    # A signature may have multiple prompt-specific cache records. The most
    # recent valid response is used for the current local republication.
    return (
        frame
        .sort_values("created_at_utc")
        .drop_duplicates(
            "issue_signature",
            keep="last",
        )
        .reset_index(drop=True)
    )


def rebuild_cached_proposal_row(
    cache_row: pd.Series,
    queue_lookup: pd.DataFrame,
) -> Optional[dict]:
    issue_signature = safe_text(
        cache_row.get("issue_signature")
    )

    if issue_signature is None:
        return None

    if issue_signature not in queue_lookup.index:
        return None

    queue_row = queue_lookup.loc[
        issue_signature
    ]

    if isinstance(queue_row, pd.DataFrame):
        queue_row = queue_row.iloc[-1]

    exception_type = safe_text(
        queue_row.get("exception_type")
    )

    if exception_type not in RESPONSE_SCHEMA_BY_TASK:
        return None

    schema = RESPONSE_SCHEMA_BY_TASK[
        exception_type
    ]

    try:
        cached_payload = json.loads(
            cache_row["response_json"]
        )

        parsed = schema.model_validate(
            cached_payload
        ).model_dump()

        evidence_text = retrieve_evidence_context(
            queue_row
        )

        (
            _,
            _,
            candidates,
        ) = build_ai_prompt(
            queue_row,
            evidence_text,
        )

        parsed = resolve_candidate_selection(
            parsed,
            candidates,
        )

        proposal = proposal_row_from_result(
            row=queue_row,
            parsed=parsed,
            provider_name=(
                safe_text(
                    cache_row.get("provider")
                )
                or "Azure OpenAI"
            ),
            deployment_name=(
                safe_text(
                    cache_row.get("deployment")
                )
                or AZURE_OPENAI_DEPLOYMENT
            ),
            model_name=(
                safe_text(
                    cache_row.get("model")
                )
                or AZURE_OPENAI_DEPLOYMENT
            ),
            prompt_hash=(
                safe_text(
                    cache_row.get("prompt_hash")
                )
                or ""
            ),
            cache_hit=True,
            processed_at_utc=cache_row.get(
                "created_at_utc"
            ),
            candidates=candidates,
        )

        proposal[
            "republication_source"
        ] = HISTORICAL_CACHE_REPUBLICATION_SOURCE

        proposal[
            "republication_policy_version"
        ] = REPUBLICATION_POLICY_VERSION

        proposal[
            "historical_cache_republication"
        ] = True

        return proposal

    except Exception as exc:
        return {
            "issue_signature": issue_signature,
            "exception_type": exception_type,
            "model_provider": (
                safe_text(
                    cache_row.get("provider")
                )
                or "Azure OpenAI"
            ),
            "model_deployment": (
                safe_text(
                    cache_row.get("deployment")
                )
                or AZURE_OPENAI_DEPLOYMENT
            ),
            "model_name": (
                safe_text(
                    cache_row.get("model")
                )
                or AZURE_OPENAI_DEPLOYMENT
            ),
            "prompt_hash": cache_row.get(
                "prompt_hash"
            ),
            "cache_hit": True,
            "processed_at_utc": cache_row.get(
                "created_at_utc"
            ),
            "provider_status": "FAILED",
            "provider_error": repr(exc),
            "republication_source":
                HISTORICAL_CACHE_REPUBLICATION_SOURCE,
            "republication_policy_version":
                REPUBLICATION_POLICY_VERSION,
            "historical_cache_republication": True,
        }


historical_ai_cache_df = (
    load_all_ai_cache_rows(
        AI_CACHE_PATH
    )
    if REPUBLISH_ALL_COMPLETED_CACHE_RESPONSES
    else pd.DataFrame()
)

queue_lookup_df = (
    ai_exception_signature_queue_df
    .drop_duplicates(
        "issue_signature",
        keep="last",
    )
    .set_index(
        "issue_signature",
        drop=False,
    )
)

historical_proposal_rows = []

if not historical_ai_cache_df.empty:
    for _, cache_row in tqdm(
        historical_ai_cache_df.iterrows(),
        total=len(historical_ai_cache_df),
        desc="Rebuilding cached AI proposals",
    ):
        proposal = rebuild_cached_proposal_row(
            cache_row,
            queue_lookup_df,
        )

        if proposal is not None:
            historical_proposal_rows.append(
                proposal
            )

historical_ai_enrichment_proposals_df = (
    pd.DataFrame(
        historical_proposal_rows
    )
)

current_ai_enrichment_proposals_df = (
    ai_enrichment_proposals_df.copy()
)

if not current_ai_enrichment_proposals_df.empty:
    current_ai_enrichment_proposals_df[
        "republication_source"
    ] = "CURRENT_EXECUTION"

    current_ai_enrichment_proposals_df[
        "republication_policy_version"
    ] = REPUBLICATION_POLICY_VERSION

    current_ai_enrichment_proposals_df[
        "historical_cache_republication"
    ] = False


proposal_frames = [
    frame
    for frame in [
        historical_ai_enrichment_proposals_df,
        current_ai_enrichment_proposals_df,
    ]
    if not frame.empty
]

if proposal_frames:
    ai_enrichment_proposals_df = (
        pd.concat(
            proposal_frames,
            ignore_index=True,
            sort=False,
        )
        .sort_values(
            [
                "issue_signature",
                "historical_cache_republication",
            ],
            ascending=[
                True,
                True,
            ],
        )
        .drop_duplicates(
            "issue_signature",
            keep="first",
        )
        .reset_index(drop=True)
    )
else:
    ai_enrichment_proposals_df = pd.DataFrame()


historical_republication_summary_df = pd.DataFrame({
    "metric": [
        "cache_rows_loaded",
        "historical_proposals_rebuilt",
        "current_execution_proposals",
        "combined_unique_proposals_for_republication",
        "republication_policy_version",
        "new_azure_requests_created_by_republication",
    ],
    "value_text": [
        str(len(historical_ai_cache_df)),
        str(len(historical_ai_enrichment_proposals_df)),
        str(len(current_ai_enrichment_proposals_df)),
        str(len(ai_enrichment_proposals_df)),
        str(REPUBLICATION_POLICY_VERSION),
        "0",
    ],
    "value_numeric": [
        float(len(historical_ai_cache_df)),
        float(len(historical_ai_enrichment_proposals_df)),
        float(len(current_ai_enrichment_proposals_df)),
        float(len(ai_enrichment_proposals_df)),
        np.nan,
        0.0,
    ],
})

print(
    "Historical cached proposals rebuilt:",
    f"{len(historical_ai_enrichment_proposals_df):,}",
)

print(
    "Combined unique proposals to re-evaluate:",
    f"{len(ai_enrichment_proposals_df):,}",
)

display(
    historical_republication_summary_df
)


Rebuilding cached AI proposals:   0%|          | 0/58962 [00:00<?, ?it/s]

Historical cached proposals rebuilt: 58,962
Combined unique proposals to re-evaluate: 58,962


,metric,value_text,value_numeric
0,cache_rows_loaded,58962,58962.0
1,historical_proposals_rebuilt,58962,58962.0
2,current_execution_proposals,1,1.0
3,combined_unique_proposals_for_republication,58962,58962.0
4,republication_policy_version,block9_acceptance_policy_v2,NaN
5,new_azure_requests_created_by_republication,0,0.0


In [17]:
# ============================================================
# 17. DETERMINISTIC ACCEPTANCE ENGINE
# ============================================================

def evidence_supported(value: Any) -> bool:
    """Require a substantive source-grounded evidence field."""
    text = safe_text(value)
    return (
        text is not None
        and len(text) >= MIN_EVIDENCE_CHARACTERS
    )


def normalised_concept(value: Any) -> Optional[str]:
    """Normalise a concept identifier for deterministic comparisons."""
    text = normalise_identifier(value)
    return text.casefold() if text else None


def normalised_label_for_concept_match(value: Any) -> Optional[str]:
    """Normalise source labels and concept identifiers to comparable words."""
    text = normalise_account_label(value)
    if text is None:
        return None

    text = re.sub(r"[_\-]+", " ", text.casefold())
    text = re.sub(r"\s+", " ", text).strip()
    return text or None


def deterministic_invalid_source_reason(value: Any) -> Optional[str]:
    """Identify only narrow, high-precision invalid-source patterns.

    Broad financial-statement labels, subtotals and 'other' categories are not
    rejected by this function.
    """
    text = safe_text(value)

    if text is None:
        return "MISSING_SOURCE_LABEL"

    compact = unicodedata.normalize("NFKC", text).strip()
    lowered = compact.casefold()

    if not re.search(r"[A-Za-z\u3400-\u9fff\u3040-\u30ff\uac00-\ud7af]", compact):
        if re.fullmatch(r"[\W\d_]+", compact):
            return "NUMERIC_OR_PUNCTUATION_ONLY"

    if re.fullmatch(
        r"(page|p\.?)\s*\d+(\s*(of|/)\s*\d+)?",
        lowered,
    ):
        return "PAGE_NUMBER"

    if re.fullmatch(
        r"(rmb|cny|usd|eur|jpy|krw|aud|hkd|gbp|"
        r"in\s+(thousands?|millions?|billions?)|"
        r"单位[:：]?\s*(元|千元|百万元))",
        lowered,
    ):
        return "UNIT_LABEL"

    if re.fullmatch(
        r"(19|20)\d{2}([\-/.年](0?[1-9]|1[0-2]))?"
        r"([\-/.月](0?[1-9]|[12]\d|3[01])日?)?",
        lowered,
    ):
        return "DATE_LABEL"

    if lowered in {
        "notes",
        "note",
        "continued",
        "continuation",
        "table of contents",
        "contents",
    }:
        return "DOCUMENT_LAYOUT_LABEL"

    return None


def fragment_quality_reason(
    value: Any,
) -> Optional[str]:
    original = safe_text(value)

    if original is None:
        return "MISSING_LABEL"

    cleaned = strip_reporting_period_suffix(
        original
    )
    cleaned = remove_roman_note_tokens(
        cleaned
    )
    cleaned = safe_text(cleaned)

    if cleaned is None:
        return "MISSING_LABEL_AFTER_NORMALISATION"

    lowered = cleaned.casefold().strip()

    if len(lowered) < MIN_COMPLETE_LABEL_CHARACTERS:
        return "TOO_SHORT"

    words = re.findall(r"[a-zA-Z]+", lowered)

    if words:
        trailing_one = words[-1]
        trailing_two = (
            " ".join(words[-2:])
            if len(words) >= 2
            else trailing_one
        )

        if (
            trailing_one in FRAGMENT_TRAILING_TERMS
            or trailing_two in FRAGMENT_TRAILING_TERMS
        ):
            return "TRAILING_FRAGMENT_TERM"

    if re.fullmatch(
        r"(activities|liabilities|assets|equipment|"
        r"income|profit|revenue|equity|borrowings|"
        r"equivalents)\s+of",
        lowered,
    ):
        return "INCOMPLETE_NOUN_PHRASE"

    if re.fullmatch(
        r"(in|with|during|from|for)\s+.+",
        lowered,
    ):
        return "LEADING_PREPOSITION_FRAGMENT"

    return None


FLOW_CUE_TERMS = {
    "depreciation",
    "amortization",
    "amortisation",
    "impairment",
    "gain on disposal",
    "loss on disposal",
    "purchase",
    "acquisition",
    "additions",
    "cash generated",
    "cash used",
    "expense",
    "income attributable",
}

STOCK_CONCEPT_TERMS = {
    "assets",
    "liabilities",
    "receivables",
    "payables",
    "inventory",
    "cash_and_cash_equivalents",
    "debt",
    "equity",
    "property_plant_and_equipment",
    "right_of_use_assets",
    "intangible_assets",
}


def stock_flow_compatibility_reason(
    source_label: Any,
    proposed_concept: Any,
) -> Optional[str]:
    label = canonicalise_source_label(
        source_label
    )
    concept = normalise_identifier(
        proposed_concept
    )

    if label is None or concept is None:
        return None

    has_flow_cue = any(
        cue in label
        for cue in FLOW_CUE_TERMS
    )

    concept_looks_stock = any(
        term in concept
        for term in STOCK_CONCEPT_TERMS
    )

    explicit_balance_cues = any(
        cue in label
        for cue in [
            "carrying amount",
            "net book value",
            "closing balance",
            "ending balance",
            "balance at",
            "value of",
        ]
    )

    if (
        has_flow_cue
        and concept_looks_stock
        and not explicit_balance_cues
    ):
        return "FLOW_LABEL_MAPPED_TO_STOCK_CONCEPT"

    return None


# ------------------------------------------------------------
# Canonical concept and observed synonym dictionaries
# ------------------------------------------------------------

canonical_concept_set = {
    concept
    for concept in (
        regional_source_records_df["standard_concept"]
        .map(normalised_concept)
        .dropna()
        .tolist()
    )
    if concept
}

observed_label_concept_pairs = {
    (
        normalised_label_for_concept_match(label),
        normalised_concept(concept),
    )
    for label, concept in zip(
        regional_source_records_df["source_account_label"],
        regional_source_records_df["standard_concept"],
    )
    if (
        normalised_label_for_concept_match(label) is not None
        and normalised_concept(concept) is not None
    )
}


# ------------------------------------------------------------
# Merge complete representative source context into AI proposals
# ------------------------------------------------------------

context_source_df = (
    ai_exception_signature_queue_df.copy()
)

proposal_signature_set = set(
    ai_enrichment_proposals_df.get(
        "issue_signature",
        pd.Series(dtype="string"),
    )
    .dropna()
    .astype("string")
)

if proposal_signature_set:
    context_source_df = context_source_df.loc[
        context_source_df[
            "issue_signature"
        ]
        .astype("string")
        .isin(proposal_signature_set)
    ].copy()
else:
    context_source_df = context_source_df.iloc[
        0:0
    ].copy()

context_columns = [
    column
    for column in [
        "review_record_id",
        "issue_signature",
        "exception_type",
        "region",
        "upstream_block",
        "source_table",
        "raw_row_count",
        "represented_raw_rows",
        "unique_issuer_count",
        "unique_filing_count",
        "global_issuer_id",
        "filing_id",
        "source_account_label",
        "standard_concept",
        "statement_type",
        "reporting_scope",
        "reported_value",
        "reported_value_numeric",
        "currency",
        "unit_scale",
        "period_end",
        "source_text",
        "exception_code",
        "qc_rule_id",
        "qc_failure_type",
        "anomaly_type",
        "pre_ai_fragment_reason",
        "pre_ai_fragment_excluded",
    ]
    if column in context_source_df.columns
]

decision_base_df = (
    context_source_df[
        context_columns
    ]
    .drop_duplicates(
        "issue_signature",
        keep="last",
    )
    .copy()
)

if (
    "represented_raw_rows"
    not in decision_base_df.columns
    and "raw_row_count"
    in decision_base_df.columns
):
    decision_base_df = decision_base_df.rename(
        columns={
            "raw_row_count":
                "represented_raw_rows"
        }
    )

proposal_frame_for_merge = (
    ai_enrichment_proposals_df
    .drop_duplicates(
        "issue_signature",
        keep="last",
    )
    .copy()
)

proposal_columns = [
    column
    for column in proposal_frame_for_merge.columns
    if column not in {
        "review_record_id",
        "exception_type",
        "region",
        "upstream_block",
        "source_table",
        "represented_raw_rows",
        "raw_row_count",
        "unique_issuer_count",
        "unique_filing_count",
        "global_issuer_id",
        "filing_id",
        "source_account_label",
        "standard_concept",
        "statement_type",
        "reporting_scope",
        "reported_value",
        "reported_value_numeric",
        "currency",
        "unit_scale",
        "period_end",
        "source_text",
        "exception_code",
        "qc_rule_id",
        "qc_failure_type",
        "anomaly_type",
        "pre_ai_fragment_reason",
        "pre_ai_fragment_excluded",
    }
]

if proposal_frame_for_merge.empty:
    decisions = decision_base_df.copy()
else:
    decisions = decision_base_df.merge(
        proposal_frame_for_merge[
            proposal_columns
        ],
        on="issue_signature",
        how="inner",
        validate="one_to_one",
    )


# ------------------------------------------------------------
# Provider-outcome diagnostics
# ------------------------------------------------------------

successful_signatures = set(
    ai_enrichment_proposals_df.get(
        "issue_signature",
        pd.Series(dtype="string"),
    )
    .dropna()
    .astype(str)
)

failed_signatures = set()

if not ai_model_audit_log_df.empty:
    failed_signature_status = (
        ai_model_audit_log_df
        .groupby("issue_signature", dropna=False)["status"]
        .apply(lambda values: set(values.dropna().astype(str)))
    )

    failed_signatures = {
        str(signature)
        for signature, statuses in failed_signature_status.items()
        if (
            "SUCCEEDED" not in statuses
            and "CACHE_HIT" not in statuses
            and "FAILED" in statuses
        )
    }

if (
    "pre_ai_fragment_reason"
    not in decisions.columns
):
    decisions[
        "pre_ai_fragment_reason"
    ] = decisions[
        "source_account_label"
    ].map(pre_ai_fragment_reason)

decisions[
    "pre_ai_fragment_excluded"
] = decisions[
    "pre_ai_fragment_reason"
].notna()


proposal_provider_failed = (
    decisions.get(
        "provider_status",
        pd.Series(
            pd.NA,
            index=decisions.index,
            dtype="string",
        ),
    )
    .astype("string")
    .str.upper()
    .eq("FAILED")
)

decisions["provider_failed"] = (
    decisions["issue_signature"]
    .astype(str)
    .isin(failed_signatures)
    | proposal_provider_failed.fillna(False)
)

decisions["ai_response_present"] = (
    decisions["issue_signature"]
    .astype(str)
    .isin(successful_signatures)
)


# ------------------------------------------------------------
# Core gates and mapping basis
# ------------------------------------------------------------

decisions["ai_confidence"] = safe_numeric(
    decisions.get(
        "ai_confidence",
        pd.Series(index=decisions.index, dtype=float),
    )
).fillna(0.0)

decisions["normalised_ai_decision"] = (
    decisions.get(
        "ai_decision",
        pd.Series(index=decisions.index, dtype="string"),
    )
    .astype("string")
    .str.upper()
    .str.strip()
)

decisions["current_concept_normalised"] = (
    decisions["standard_concept"].map(normalised_concept)
)

decisions["proposed_concept_normalised"] = (
    decisions.get(
        "proposed_standard_concept",
        pd.Series(index=decisions.index, dtype="string"),
    )
    .map(normalised_concept)
)

decisions["source_label_normalised"] = (
    decisions["source_account_label"]
    .map(normalised_label_for_concept_match)
)

decisions["proposed_label_form"] = (
    decisions["proposed_concept_normalised"]
    .map(normalised_label_for_concept_match)
)

decisions["identity_mapping_gate_passed"] = (
    decisions["current_concept_normalised"].notna()
    & decisions["proposed_concept_normalised"].notna()
    & decisions["current_concept_normalised"].eq(
        decisions["proposed_concept_normalised"]
    )
)

decisions["canonical_dictionary_gate_passed"] = (
    decisions["proposed_concept_normalised"]
    .isin(canonical_concept_set)
)

decisions["exact_label_gate_passed"] = (
    decisions["source_label_normalised"].notna()
    & decisions["proposed_label_form"].notna()
    & decisions["source_label_normalised"].eq(
        decisions["proposed_label_form"]
    )
)

decisions["known_synonym_gate_passed"] = [
    (label, concept) in observed_label_concept_pairs
    if label is not None and concept is not None
    else False
    for label, concept in zip(
        decisions["source_label_normalised"],
        decisions["proposed_concept_normalised"],
    )
]


decisions[
    "deterministic_phrase_concept"
] = decisions[
    "source_account_label"
].map(deterministic_phrase_mapping)

decisions[
    "deterministic_phrase_gate_passed"
] = [
    (
        normalise_identifier(
            deterministic_concept
        )
        == normalise_identifier(
            proposed_concept
        )
    )
    if (
        deterministic_concept is not None
        and proposed_concept is not None
    )
    else False
    for deterministic_concept, proposed_concept in zip(
        decisions[
            "deterministic_phrase_concept"
        ],
        decisions[
            "proposed_standard_concept"
        ],
    )
]

decisions[
    "historical_synonym_gate_passed"
] = [
    (
        (
            label,
            concept,
            safe_text(statement_type),
            safe_text(reporting_scope),
        )
        in trusted_historical_synonym_pairs
    )
    if (
        label is not None
        and concept is not None
    )
    else False
    for (
        label,
        concept,
        statement_type,
        reporting_scope,
    ) in zip(
        decisions["source_label_normalised"],
        decisions["proposed_concept_normalised"],
        decisions["statement_type"],
        decisions["reporting_scope"],
    )
]

decisions[
    "fragment_quality_reason"
] = decisions[
    "source_account_label"
].map(fragment_quality_reason)

decisions[
    "fragment_quality_gate_passed"
] = decisions[
    "fragment_quality_reason"
].isna()


decisions[
    "stock_flow_compatibility_reason"
] = [
    stock_flow_compatibility_reason(
        source_label,
        proposed_concept,
    )
    for source_label, proposed_concept in zip(
        decisions["source_account_label"],
        decisions[
            "proposed_standard_concept"
        ],
    )
]

decisions[
    "stock_flow_compatibility_gate_passed"
] = decisions[
    "stock_flow_compatibility_reason"
].isna()

decisions["deterministic_invalid_source_reason"] = (
    decisions["source_account_label"]
    .map(deterministic_invalid_source_reason)
)

decisions["deterministic_invalid_source_gate_passed"] = (
    decisions["deterministic_invalid_source_reason"].notna()
)

decisions["evidence_gate_passed"] = (
    decisions.get(
        "ai_evidence_text",
        pd.Series(index=decisions.index, dtype="string"),
    )
    .map(evidence_supported)
)

normalised_decision = decisions["normalised_ai_decision"]

is_reject = normalised_decision.isin(
    ["REJECT", "REJECT_FACT"]
)
is_unresolved = normalised_decision.eq("UNRESOLVED")
is_map = normalised_decision.eq("MAP")
is_keep = normalised_decision.eq("KEEP_SOURCE")
is_correction = normalised_decision.eq(
    "PROPOSE_CORRECTION"
)
is_positive = is_map | is_keep | is_correction

decisions["decision_gate_passed"] = is_positive

mapping_present = (
    decisions["proposed_concept_normalised"].notna()
)

correction_fields = [
    "proposed_value",
    "proposed_currency",
    "proposed_unit_scale",
    "proposed_sign_multiplier",
    "proposed_statement_type",
    "proposed_reporting_scope",
]

for column in correction_fields:
    if column not in decisions.columns:
        decisions[column] = pd.NA

correction_present = (
    decisions[correction_fields]
    .notna()
    .any(axis=1)
)

decisions["non_null_proposal_gate_passed"] = (
    np.select(
        [is_map, is_keep, is_correction],
        [mapping_present, True, correction_present],
        default=False,
    )
    .astype(bool)
)

effective_statement = (
    decisions["proposed_statement_type"]
    .where(
        decisions["proposed_statement_type"].notna(),
        decisions["statement_type"],
    )
)

decisions["statement_compatibility_gate_passed"] = [
    (
        concept_statement_compatible(
            concept,
            statement,
        )
        if map_decision
        else True
    )
    for concept, statement, map_decision in zip(
        decisions["proposed_standard_concept"],
        effective_statement,
        is_map,
    )
]


def determine_mapping_basis(row: pd.Series) -> str:
    decision = (
        safe_text(row.get("normalised_ai_decision"))
        or safe_text(row.get("ai_decision"))
        or ""
    ).upper()

    is_mapping_decision = decision == "MAP"

    if bool(row.get("identity_mapping_gate_passed", False)):
        return "IDENTITY_MAPPING"

    if bool(row.get("exact_label_gate_passed", False)):
        return "EXACT_LABEL_MAPPING"

    # High-certainty phrase status is reserved for an actual MAP
    # decision whose selected proposal equals the deterministic concept.
    if (
        is_mapping_decision
        and bool(
            row.get(
                "deterministic_phrase_gate_passed",
                False,
            )
        )
    ):
        return "HIGH_CERTAINTY_PHRASE_MAPPING"

    if (
        is_mapping_decision
        and bool(
            row.get(
                "historical_synonym_gate_passed",
                False,
            )
        )
    ):
        return "HISTORICAL_SYNONYM_MAPPING"

    if (
        is_mapping_decision
        and bool(
            row.get(
                "known_synonym_gate_passed",
                False,
            )
        )
    ):
        return "KNOWN_SYNONYM_MAPPING"

    return "MODEL_INFERRED_MAPPING"


decisions["mapping_basis"] = decisions.apply(
    determine_mapping_basis,
    axis=1,
)

mapping_thresholds = {
    "IDENTITY_MAPPING":
        IDENTITY_MAPPING_THRESHOLD,
    "EXACT_LABEL_MAPPING":
        EXACT_LABEL_MAPPING_THRESHOLD,
    "HIGH_CERTAINTY_PHRASE_MAPPING":
        HIGH_CERTAINTY_PHRASE_THRESHOLD,
    "HISTORICAL_SYNONYM_MAPPING":
        HISTORICAL_SYNONYM_THRESHOLD,
    "KNOWN_SYNONYM_MAPPING":
        KNOWN_SYNONYM_MAPPING_THRESHOLD,
    "MODEL_INFERRED_MAPPING":
        MODEL_INFERRED_MAPPING_THRESHOLD,
}

decisions["decision_auto_threshold"] = (
    decisions["mapping_basis"]
    .map(mapping_thresholds)
    .fillna(MODEL_INFERRED_MAPPING_THRESHOLD)
)

# Identity confirmations retain IDENTITY_MAPPING_THRESHOLD even when Azure
# expresses the result as KEEP_SOURCE. The generic KEEP_SOURCE threshold is
# applied only when the proposal is not an identity confirmation.
decisions.loc[
    is_keep
    & ~decisions[
        "identity_mapping_gate_passed"
    ],
    "decision_auto_threshold",
] = KEEP_SOURCE_THRESHOLD

decisions.loc[
    is_correction,
    "decision_auto_threshold",
] = CORRECTION_THRESHOLD

decisions["confidence_gate_passed"] = (
    decisions["ai_confidence"]
    .ge(decisions["decision_auto_threshold"])
)


# ------------------------------------------------------------
# Fully automated terminal classification
# ------------------------------------------------------------

acceptance_status = []
acceptance_reason = []

for _, row in decisions.iterrows():
    decision = safe_text(
        row.get("normalised_ai_decision")
    )
    decision = decision.upper() if decision else None

    confidence_ok = bool(
        row.get("confidence_gate_passed", False)
    )
    evidence_ok = bool(
        row.get("evidence_gate_passed", False)
    )
    proposal_ok = bool(
        row.get(
            "non_null_proposal_gate_passed",
            False,
        )
    )
    statement_ok = bool(
        row.get(
            "statement_compatibility_gate_passed",
            False,
        )
    )
    canonical_ok = bool(
        row.get(
            "canonical_dictionary_gate_passed",
            False,
        )
    )
    invalid_source_ok = bool(
        row.get(
            "deterministic_invalid_source_gate_passed",
            False,
        )
    )

    if bool(
        row.get(
            "pre_ai_fragment_excluded",
            False,
        )
    ):
        acceptance_status.append(
            "PRE_AI_EXCLUDED"
        )
        acceptance_reason.append(
            "pre_ai_fragment_excluded:"
            + str(
                row.get(
                    "pre_ai_fragment_reason"
                )
            )
        )
        continue

    if bool(row.get("provider_failed", False)):
        acceptance_status.append("PROVIDER_FAILED")
        acceptance_reason.append(
            "provider_failed_before_valid_structured_response"
        )
        continue

    if not bool(row.get("ai_response_present", False)):
        acceptance_status.append("MODEL_UNRESOLVED")
        acceptance_reason.append(
            "deterministic_dry_run_or_no_ai_proposal"
        )
        continue

    if decision in {"REJECT", "REJECT_FACT"}:
        if invalid_source_ok:
            acceptance_status.append("MODEL_REJECTED")
            acceptance_reason.append(
                "deterministically_invalid_source:"
                + str(
                    row.get(
                        "deterministic_invalid_source_reason"
                    )
                )
            )
        else:
            acceptance_status.append("MODEL_UNRESOLVED")
            acceptance_reason.append(
                "model_rejection_not_deterministically_supported"
            )
        continue

    if decision == "UNRESOLVED":
        acceptance_status.append("MODEL_UNRESOLVED")
        acceptance_reason.append(
            "model_could_not_resolve_mapping_or_fact"
        )
        continue

    if decision == "MAP":
        if not proposal_ok:
            acceptance_status.append("MODEL_UNRESOLVED")
            acceptance_reason.append(
                "map_decision_contains_no_standard_concept"
            )
            continue

        if not canonical_ok:
            acceptance_status.append("MODEL_UNRESOLVED")
            acceptance_reason.append(
                "proposal_not_in_canonical_dictionary"
            )
            continue

        if not statement_ok:
            acceptance_status.append("MODEL_UNRESOLVED")
            acceptance_reason.append(
                "statement_concept_compatibility_failed"
            )
            continue

        if not bool(
            row.get(
                "fragment_quality_gate_passed",
                False,
            )
        ):
            acceptance_status.append("MODEL_UNRESOLVED")
            acceptance_reason.append(
                "source_label_failed_fragment_quality_gate:"
                + str(
                    row.get(
                        "fragment_quality_reason"
                    )
                )
            )
            continue

        if not bool(
            row.get(
                "stock_flow_compatibility_gate_passed",
                True,
            )
        ):
            acceptance_status.append("MODEL_UNRESOLVED")
            acceptance_reason.append(
                "stock_flow_compatibility_failed:"
                + str(
                    row.get(
                        "stock_flow_compatibility_reason"
                    )
                )
            )
            continue

        if not evidence_ok:
            acceptance_status.append("MODEL_UNRESOLVED")
            acceptance_reason.append(
                "positive_decision_has_insufficient_evidence"
            )
            continue

        if confidence_ok:
            acceptance_status.append("AUTO_ACCEPTED")
            acceptance_reason.append(
                str(row.get("mapping_basis")).casefold()
                + "_all_gates_passed"
            )
        else:
            acceptance_status.append(
                "LOW_CONFIDENCE_PROPOSAL"
            )
            acceptance_reason.append(
                str(row.get("mapping_basis")).casefold()
                + "_below_threshold"
            )
        continue

    if decision == "KEEP_SOURCE":
        current_concept = normalised_concept(
            row.get("standard_concept")
        )

        if current_concept is None:
            acceptance_status.append("MODEL_UNRESOLVED")
            acceptance_reason.append(
                "keep_source_without_existing_concept"
            )
        elif not evidence_ok:
            acceptance_status.append("MODEL_UNRESOLVED")
            acceptance_reason.append(
                "keep_source_has_insufficient_evidence"
            )
        elif not statement_ok:
            acceptance_status.append("MODEL_UNRESOLVED")
            acceptance_reason.append(
                "keep_source_statement_incompatible"
            )
        elif not bool(
            row.get(
                "fragment_quality_gate_passed",
                False,
            )
        ):
            acceptance_status.append("MODEL_UNRESOLVED")
            acceptance_reason.append(
                "keep_source_failed_fragment_quality_gate:"
                + str(
                    row.get(
                        "fragment_quality_reason"
                    )
                )
            )
        elif not bool(
            row.get(
                "stock_flow_compatibility_gate_passed",
                True,
            )
        ):
            acceptance_status.append("MODEL_UNRESOLVED")
            acceptance_reason.append(
                "keep_source_stock_flow_compatibility_failed:"
                + str(
                    row.get(
                        "stock_flow_compatibility_reason"
                    )
                )
            )
        elif confidence_ok:
            acceptance_status.append("AUTO_ACCEPTED")
            acceptance_reason.append(
                "existing_mapping_confirmed"
            )
        else:
            acceptance_status.append(
                "LOW_CONFIDENCE_PROPOSAL"
            )
            acceptance_reason.append(
                "keep_source_below_threshold"
            )
        continue

    if decision == "PROPOSE_CORRECTION":
        if not proposal_ok:
            acceptance_status.append("MODEL_UNRESOLVED")
            acceptance_reason.append(
                "correction_decision_contains_no_corrected_field"
            )
        elif not evidence_ok:
            acceptance_status.append("MODEL_UNRESOLVED")
            acceptance_reason.append(
                "correction_has_insufficient_evidence"
            )
        elif confidence_ok:
            acceptance_status.append("AUTO_ACCEPTED")
            acceptance_reason.append(
                "evidence_supported_correction_all_gates_passed"
            )
        else:
            acceptance_status.append(
                "LOW_CONFIDENCE_PROPOSAL"
            )
            acceptance_reason.append(
                "evidence_supported_correction_below_threshold"
            )
        continue

    acceptance_status.append("MODEL_UNRESOLVED")
    acceptance_reason.append(
        "unsupported_or_missing_ai_decision"
    )


decisions["acceptance_status"] = acceptance_status
decisions["acceptance_reason"] = acceptance_reason

decisions["publication_status"] = (
    decisions["acceptance_status"]
    .map(
        {
            "AUTO_ACCEPTED": "ACCEPTED",
            "MODEL_REJECTED": "REJECTED",
            "LOW_CONFIDENCE_PROPOSAL": "UNRESOLVED",
            "MODEL_UNRESOLVED": "UNRESOLVED",
            "PROVIDER_FAILED": "UNRESOLVED",
            "PRE_AI_EXCLUDED": "UNRESOLVED",
        }
    )
    .fillna("UNRESOLVED")
)

decisions["automatic_acceptance_eligible"] = (
    decisions["acceptance_status"]
    .eq("AUTO_ACCEPTED")
)

ai_enrichment_decisions_df = (
    decisions.reset_index(drop=True)
)

print(
    "AI decisions:",
    len(ai_enrichment_decisions_df),
)

if not ai_enrichment_decisions_df.empty:
    display(
        ai_enrichment_decisions_df[
            [
                "exception_type",
                "issue_signature",
                "represented_raw_rows",
                "source_account_label",
                "standard_concept",
                "ai_decision",
                "proposed_standard_concept",
                "mapping_basis",
                "ai_confidence",
                "decision_auto_threshold",
                "statement_compatibility_gate_passed",
                "acceptance_status",
                "publication_status",
                "acceptance_reason",
            ]
        ]
    )


AI decisions: 58962


,exception_type,issue_signature,represented_raw_rows,source_account_label,standard_concept,ai_decision,proposed_standard_concept,mapping_basis,ai_confidence,decision_auto_threshold,statement_compatibility_gate_passed,acceptance_status,publication_status,acceptance_reason
0,ACCOUNT_MAPPING,000081b1fb2a256fa633ff9c1caa3e57e36a0d83d29ead...,1,global output of salt-lake lithium grew from,<NA>,REJECT,NaN,MODEL_INFERRED_MAPPING,0.94,0.9,True,PRE_AI_EXCLUDED,UNRESOLVED,pre_ai_fragment_excluded:TRAILING_FRAGMENT
1,ACCOUNT_MAPPING,0000c03a8ef343b459776eba32f60581178dd9dbdb8942...,1,高新技术企业按,<NA>,REJECT,NaN,MODEL_INFERRED_MAPPING,0.78,0.9,True,MODEL_UNRESOLVED,UNRESOLVED,model_rejection_not_deterministically_supported
2,ACCOUNT_MAPPING,000169f3865b5778dcf19895df208379719fbcc8f0610d...,1,average grade of 1.98% nickel) were sold into ...,<NA>,REJECT,NaN,MODEL_INFERRED_MAPPING,0.82,0.9,True,PRE_AI_EXCLUDED,UNRESOLVED,pre_ai_fragment_excluded:TRAILING_FRAGMENT
3,ACCOUNT_MAPPING,00032021373842cf1c318508c2e6494ec98cfce0e464e3...,1,王锟 董事 副总裁 离任,<NA>,REJECT,NaN,MODEL_INFERRED_MAPPING,0.91,0.9,True,MODEL_UNRESOLVED,UNRESOLVED,model_rejection_not_deterministically_supported
4,ACCOUNT_MAPPING,0004499c5b6ef5fcdc0cedcfc531ee94e12c8a1af55949...,1,chen wei december,<NA>,REJECT,NaN,MODEL_INFERRED_MAPPING,0.90,0.9,True,MODEL_UNRESOLVED,UNRESOLVED,model_rejection_not_deterministically_supported
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58957,ACCOUNT_MAPPING,fffd62b8a85de20895cbc3e052403e7f36e11199957420...,1,館剪,<NA>,UNRESOLVED,NaN,MODEL_INFERRED_MAPPING,0.30,0.9,True,MODEL_UNRESOLVED,UNRESOLVED,model_could_not_resolve_mapping_or_fact
58958,ACCOUNT_MAPPING,fffed72126247057577341a0db3a9d66939ee875b444f9...,1,定要求 具体详见公司于,<NA>,UNRESOLVED,NaN,MODEL_INFERRED_MAPPING,0.30,0.9,True,MODEL_UNRESOLVED,UNRESOLVED,model_could_not_resolve_mapping_or_fact
58959,ACCOUNT_MAPPING,ffff0371f31e5387b6ebcb2d90dfd74b54916475ac1e90...,1,长安福特汽车有限公司,<NA>,UNRESOLVED,NaN,MODEL_INFERRED_MAPPING,0.66,0.9,True,MODEL_UNRESOLVED,UNRESOLVED,model_could_not_resolve_mapping_or_fact
58960,ACCOUNT_MAPPING,ffffc6f1b91f1fa81b78dacd772bcb2e52e6ce6df46492...,1,Cash cost ($US / per tonne of Nickel),<NA>,MAP,unmapped_concepts_in_inventory,MODEL_INFERRED_MAPPING,0.60,0.9,False,MODEL_UNRESOLVED,UNRESOLVED,statement_concept_compatibility_failed


In [ ]:
# ============================================================
# 18. PUBLICATION, CLUSTER PROPAGATION, LINEAGE AND REGISTRIES
# ============================================================

if ai_enrichment_decisions_df.empty:
    accepted_ai_enrichments_df = pd.DataFrame()
    rejected_ai_enrichments_df = pd.DataFrame()
    unresolved_ai_enrichments_df = pd.DataFrame()
    low_confidence_proposals_df = pd.DataFrame()
    model_unresolved_ai_enrichments_df = pd.DataFrame()
    provider_failed_ai_enrichments_df = pd.DataFrame()
else:
    accepted_ai_enrichments_df = (
        ai_enrichment_decisions_df.loc[
            ai_enrichment_decisions_df[
                "publication_status"
            ].eq("ACCEPTED")
        ]
        .reset_index(drop=True)
    )
    rejected_ai_enrichments_df = (
        ai_enrichment_decisions_df.loc[
            ai_enrichment_decisions_df[
                "publication_status"
            ].eq("REJECTED")
        ]
        .reset_index(drop=True)
    )
    unresolved_ai_enrichments_df = (
        ai_enrichment_decisions_df.loc[
            ai_enrichment_decisions_df[
                "publication_status"
            ].eq("UNRESOLVED")
        ]
        .reset_index(drop=True)
    )
    low_confidence_proposals_df = (
        ai_enrichment_decisions_df.loc[
            ai_enrichment_decisions_df[
                "acceptance_status"
            ].eq("LOW_CONFIDENCE_PROPOSAL")
        ]
        .reset_index(drop=True)
    )
    model_unresolved_ai_enrichments_df = (
        ai_enrichment_decisions_df.loc[
            ai_enrichment_decisions_df[
                "acceptance_status"
            ].eq("MODEL_UNRESOLVED")
        ]
        .reset_index(drop=True)
    )
    provider_failed_ai_enrichments_df = (
        ai_enrichment_decisions_df.loc[
            ai_enrichment_decisions_df[
                "acceptance_status"
            ].eq("PROVIDER_FAILED")
        ]
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# Direct accepted prototype decisions
# ------------------------------------------------------------

propagation_columns = [
    "issue_signature",
    "review_record_id",
    "model_provider",
    "model_deployment",
    "model_name",
    "prompt_hash",
    "cache_hit",
    "processed_at_utc",
    "ai_decision",
    "ai_confidence",
    "ai_evidence_text",
    "ai_reasoning_summary",
    "proposed_standard_concept",
    "proposed_value",
    "proposed_currency",
    "proposed_unit_scale",
    "proposed_sign_multiplier",
    "proposed_statement_type",
    "proposed_reporting_scope",
    "mapping_basis",
    "acceptance_status",
    "publication_status",
    "acceptance_reason",
]

accepted_signature_decisions_df = (
    accepted_ai_enrichments_df[
        [
            column
            for column in propagation_columns
            if column in accepted_ai_enrichments_df.columns
        ]
    ]
    .rename(
        columns={
            "review_record_id":
                "representative_review_record_id"
        }
    )
    .drop_duplicates("issue_signature", keep="last")
    if not accepted_ai_enrichments_df.empty
    else pd.DataFrame(columns=propagation_columns)
)


# ------------------------------------------------------------
# Conservative cluster consensus
# ------------------------------------------------------------

prototype_context_columns = [
    column
    for column in [
        "issue_signature",
        "deterministic_cluster_id",
        "cluster_size",
        "required_cluster_prototypes",
        "prototype_rank_within_cluster",
        "exception_type",
        "statement_type",
        "reporting_scope",
        "standard_concept",
    ]
    if column in selected_ai_exceptions_df.columns
]

prototype_decisions_df = (
    selected_ai_exceptions_df[
        prototype_context_columns
    ]
    .merge(
        ai_enrichment_decisions_df[
            [
                column
                for column in [
                    "issue_signature",
                    "publication_status",
                    "acceptance_status",
                    "proposed_standard_concept",
                    "ai_confidence",
                    "prompt_hash",
                    "model_provider",
                    "model_deployment",
                    "model_name",
                ]
                if column
                in ai_enrichment_decisions_df.columns
            ]
        ],
        on="issue_signature",
        how="left",
        validate="one_to_one",
    )
)

cluster_consensus_rows = []

for cluster_id, group in prototype_decisions_df.groupby(
    "deterministic_cluster_id",
    dropna=False,
):
    required = int(
        safe_numeric(
            group[
                "required_cluster_prototypes"
            ]
        ).fillna(1).max()
    )

    accepted_group = group.loc[
        group["publication_status"].eq("ACCEPTED")
    ]

    accepted_concepts = (
        accepted_group[
            "proposed_standard_concept"
        ]
        .dropna()
        .astype(str)
    )

    concept_counts = accepted_concepts.value_counts()

    if len(concept_counts):
        winning_concept = concept_counts.index[0]
        winning_count = int(concept_counts.iloc[0])
    else:
        winning_concept = None
        winning_count = 0

    accepted_count = len(accepted_group)

    consensus_ratio = (
        winning_count / accepted_count
        if accepted_count
        else 0.0
    )

    consensus_passed = (
        ENABLE_SIMILARITY_PROPAGATION
        and accepted_count >= required
        and consensus_ratio
        >= CLUSTER_CONSENSUS_REQUIRED
        and winning_concept is not None
    )

    prototype_signature = (
        safe_text(
            accepted_group.loc[
                accepted_group[
                    "proposed_standard_concept"
                ].eq(winning_concept),
                "issue_signature",
            ].iloc[0]
        )
        if consensus_passed
        else None
    )

    prototype_confidence = (
        float(
            safe_numeric(
                accepted_group.loc[
                    accepted_group[
                        "proposed_standard_concept"
                    ].eq(winning_concept),
                    "ai_confidence",
                ]
            )
            .dropna()
            .min()
        )
        if consensus_passed
        else None
    )

    cluster_consensus_rows.append(
        {
            "cluster_id": cluster_id,
            "required_prototypes": required,
            "prototype_count": len(group),
            "accepted_prototype_count": accepted_count,
            "winning_standard_concept":
                winning_concept,
            "consensus_ratio": consensus_ratio,
            "consensus_passed": consensus_passed,
            "prototype_issue_signature":
                prototype_signature,
            "minimum_prototype_confidence":
                prototype_confidence,
        }
    )

ai_cluster_propagation_decisions_df = pd.DataFrame(
    cluster_consensus_rows
)


# ------------------------------------------------------------
# Propagate cluster decisions only to structurally identical members
# not directly reviewed this run.
# ------------------------------------------------------------

directly_reviewed_signatures = set(
    selected_ai_exceptions_df[
        "issue_signature"
    ].astype(str)
)

# ------------------------------------------------------------
# Prepare cluster-propagation decisions defensively
# ------------------------------------------------------------

cluster_decision_schema = {
    "cluster_id": "string",
    "required_prototypes": "Int64",
    "prototype_count": "Int64",
    "accepted_prototype_count": "Int64",
    "winning_standard_concept": "string",
    "consensus_ratio": "float64",
    "consensus_passed": "boolean",
    "prototype_issue_signature": "string",
    "minimum_prototype_confidence": "float64",
}


if (
    "ai_cluster_propagation_decisions_df"
    not in globals()
    or ai_cluster_propagation_decisions_df is None
):
    ai_cluster_propagation_decisions_df = pd.DataFrame({
        column: pd.Series(dtype=dtype)
        for column, dtype in cluster_decision_schema.items()
    })

else:
    ai_cluster_propagation_decisions_df = (
        ai_cluster_propagation_decisions_df.copy()
    )

    if (
        "cluster_id"
        not in ai_cluster_propagation_decisions_df.columns
        and "deterministic_cluster_id"
        in ai_cluster_propagation_decisions_df.columns
    ):
        ai_cluster_propagation_decisions_df = (
            ai_cluster_propagation_decisions_df.rename(
                columns={
                    "deterministic_cluster_id":
                        "cluster_id",
                }
            )
        )

    for (
        column,
        dtype,
    ) in cluster_decision_schema.items():
        if (
            column
            not in ai_cluster_propagation_decisions_df.columns
        ):
            ai_cluster_propagation_decisions_df[
                column
            ] = pd.Series(
                pd.NA,
                index=(
                    ai_cluster_propagation_decisions_df
                    .index
                ),
                dtype=dtype,
            )


ai_cluster_propagation_decisions_df[
    "consensus_passed"
] = (
    ai_cluster_propagation_decisions_df[
        "consensus_passed"
    ]
    .fillna(False)
    .astype(bool)
)

for column in [
    "consensus_ratio",
    "minimum_prototype_confidence",
]:
    ai_cluster_propagation_decisions_df[
        column
    ] = pd.to_numeric(
        ai_cluster_propagation_decisions_df[
            column
        ],
        errors="coerce",
    )


# Ensure the left-side queue also carries the merge key.
if (
    "deterministic_cluster_id"
    not in remaining_ai_work_queue_df.columns
):
    remaining_ai_work_queue_df = (
        remaining_ai_work_queue_df.copy()
    )

    remaining_ai_work_queue_df[
        "deterministic_cluster_id"
    ] = pd.NA


# ------------------------------------------------------------
# Merge propagation decisions onto remaining cluster members
# ------------------------------------------------------------

cluster_member_outcomes = (
    remaining_ai_work_queue_df.merge(
        ai_cluster_propagation_decisions_df,
        left_on="deterministic_cluster_id",
        right_on="cluster_id",
        how="left",
        suffixes=(
            "",
            "_propagation",
        ),
        validate="m:1",
    )
)

# Missing consensus means no propagation.
cluster_member_outcomes[
    "consensus_passed"
] = (
    cluster_member_outcomes[
        "consensus_passed"
    ]
    .fillna(False)
    .astype(bool)
)

cluster_member_outcomes[
    "winning_standard_concept"
] = (
    cluster_member_outcomes[
        "winning_standard_concept"
    ]
    .astype("string")
)

cluster_member_outcomes[
    "minimum_prototype_confidence"
] = pd.to_numeric(
    cluster_member_outcomes[
        "minimum_prototype_confidence"
    ],
    errors="coerce",
)

cluster_member_outcomes[
    "directly_reviewed"
] = (
    cluster_member_outcomes[
        "issue_signature"
    ].astype(str).isin(
        directly_reviewed_signatures
    )
)

cluster_member_outcomes[
    "cluster_similarity_score"
] = np.where(
    cluster_member_outcomes[
        "normalised_cluster_label"
    ].notna(),
    1.0,
    0.0,
)

cluster_member_outcomes[
    "cluster_propagation_eligible"
] = (
    ENABLE_SIMILARITY_PROPAGATION
    & cluster_member_outcomes[
        "consensus_passed"
    ].fillna(False)
    & ~cluster_member_outcomes[
        "directly_reviewed"
    ]
    & cluster_member_outcomes[
        "cluster_similarity_score"
    ].ge(SEMANTIC_PROPAGATION_THRESHOLD)
)

cluster_member_outcomes[
    "cluster_propagated_standard_concept"
] = (
    cluster_member_outcomes[
        "winning_standard_concept"
    ]
    .where(
        cluster_member_outcomes[
            "cluster_propagation_eligible"
        ]
    )
)

cluster_member_outcomes[
    "cluster_completion_method"
] = np.where(
    cluster_member_outcomes[
        "cluster_propagation_eligible"
    ],
    "DETERMINISTIC_CLUSTER_PROPAGATION",
    pd.NA,
)

ai_cluster_member_outcomes_df = (
    cluster_member_outcomes.copy()
)


# ------------------------------------------------------------
# Combined direct and propagated signature decisions
# ------------------------------------------------------------

direct_signature_decisions = (
    accepted_signature_decisions_df.copy()
)

direct_signature_decisions[
    "completion_method"
] = "DIRECT_AZURE"

direct_signature_decisions[
    "prototype_issue_signature"
] = direct_signature_decisions[
    "issue_signature"
]

direct_signature_decisions[
    "cluster_id"
] = pd.NA

direct_signature_decisions[
    "cluster_similarity_score"
] = 1.0

propagated_signature_rows = []

for _, row in ai_cluster_member_outcomes_df.loc[
    ai_cluster_member_outcomes_df[
        "cluster_propagation_eligible"
    ].fillna(False)
].iterrows():
    propagated_signature_rows.append(
        {
            "issue_signature":
                row["issue_signature"],
            "representative_review_record_id":
                row.get("review_record_id"),
            "model_provider":
                "Cluster propagation",
            "model_deployment":
                pd.NA,
            "model_name":
                pd.NA,
            "prompt_hash":
                pd.NA,
            "cache_hit":
                False,
            "processed_at_utc":
                utc_now(),
            "ai_decision":
                "CLUSTER_PROPAGATED",
            "ai_confidence":
                row.get(
                    "minimum_prototype_confidence"
                ),
            "ai_evidence_text":
                pd.NA,
            "ai_reasoning_summary":
                "Accepted prototype consensus propagated to a structurally identical cluster member.",
            "proposed_standard_concept":
                row.get(
                    "cluster_propagated_standard_concept"
                ),
            "proposed_value":
                pd.NA,
            "proposed_currency":
                pd.NA,
            "proposed_unit_scale":
                pd.NA,
            "proposed_sign_multiplier":
                pd.NA,
            "proposed_statement_type":
                row.get("statement_type"),
            "proposed_reporting_scope":
                row.get("reporting_scope"),
            "mapping_basis":
                "DETERMINISTIC_CLUSTER_PROPAGATION",
            "acceptance_status":
                "AUTO_ACCEPTED",
            "publication_status":
                "ACCEPTED",
            "acceptance_reason":
                "accepted_cluster_consensus",
            "completion_method":
                "DETERMINISTIC_CLUSTER_PROPAGATION",
            "prototype_issue_signature":
                row.get(
                    "prototype_issue_signature"
                ),
            "cluster_id":
                row.get("cluster_id"),
            "cluster_similarity_score":
                row.get(
                    "cluster_similarity_score"
                ),
        }
    )

propagated_signature_decisions_df = pd.DataFrame(
    propagated_signature_rows
)

all_accepted_signature_decisions_df = (
    pd.concat(
        [
            direct_signature_decisions,
            propagated_signature_decisions_df,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates("issue_signature", keep="first")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Raw lineage output: original raw queue
# ------------------------------------------------------------

raw_for_propagation = ai_exception_queue_raw_df.copy()

raw_for_propagation[
    "issue_signature"
] = raw_for_propagation.apply(
    make_issue_signature,
    axis=1,
)

ai_exception_raw_lineage_outcomes_df = (
    raw_for_propagation.merge(
        all_accepted_signature_decisions_df,
        on="issue_signature",
        how="left",
    )
)

def add_signature_decision_flags(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    """Separate applied, direct and propagated accepted decisions."""
    result = frame.copy()

    publication_status = result.get(
        "publication_status",
        pd.Series(
            pd.NA,
            index=result.index,
            dtype="string",
        ),
    ).astype("string")

    completion_method = result.get(
        "completion_method",
        pd.Series(
            pd.NA,
            index=result.index,
            dtype="string",
        ),
    ).astype("string")

    result["signature_decision_applied"] = (
        publication_status.eq("ACCEPTED")
    )

    result["signature_decision_direct"] = (
        result["signature_decision_applied"]
        & completion_method.eq("DIRECT_AZURE")
    )

    result["signature_decision_propagated"] = (
        result["signature_decision_applied"]
        & completion_method.eq(
            "DETERMINISTIC_CLUSTER_PROPAGATION"
        )
    )

    return result


ai_exception_raw_lineage_outcomes_df = (
    add_signature_decision_flags(
        ai_exception_raw_lineage_outcomes_df
    )
)


# ------------------------------------------------------------
# Deduplicated production output: Block 10 input
# ------------------------------------------------------------

deduplicated_for_propagation = (
    ai_exception_queue_deduplicated_df.copy()
)

deduplicated_for_propagation[
    "issue_signature"
] = deduplicated_for_propagation.apply(
    make_issue_signature,
    axis=1,
)

ai_exception_deduplicated_outcomes_df = (
    deduplicated_for_propagation.merge(
        all_accepted_signature_decisions_df,
        on="issue_signature",
        how="left",
    )
)

ai_exception_deduplicated_outcomes_df = (
    add_signature_decision_flags(
        ai_exception_deduplicated_outcomes_df
    )
)

decision_application_timestamp = utc_now().isoformat()

for frame in [
    ai_exception_raw_lineage_outcomes_df,
    ai_exception_deduplicated_outcomes_df,
]:
    frame["original_standard_concept"] = (
        frame["standard_concept"]
    )

    frame["original_reported_value"] = (
        frame["reported_value_numeric"]
    )

    frame["effective_standard_concept"] = (
        frame["proposed_standard_concept"]
        .where(
            frame["signature_decision_applied"]
            & frame["proposed_standard_concept"].notna(),
            frame["standard_concept"],
        )
    )

    frame["effective_reported_value"] = (
        frame["proposed_value"]
        .where(
            frame["signature_decision_applied"]
            & frame["proposed_value"].notna(),
            frame["reported_value_numeric"],
        )
    )

    applied_mask = (
        frame["signature_decision_applied"]
        .fillna(False)
        .astype(bool)
    )

    propagated_mask = (
        frame["signature_decision_propagated"]
        .fillna(False)
        .astype(bool)
    )

    frame["decision_applied_at_utc"] = pd.Series(
        pd.NA,
        index=frame.index,
        dtype="string",
    )

    frame.loc[
        applied_mask,
        "decision_applied_at_utc",
    ] = decision_application_timestamp

    frame["propagated_at_utc"] = pd.Series(
        pd.NA,
        index=frame.index,
        dtype="string",
    )

    frame.loc[
        propagated_mask,
        "propagated_at_utc",
    ] = decision_application_timestamp

ai_exception_row_level_outcomes_df = (
    ai_exception_deduplicated_outcomes_df
)


# ------------------------------------------------------------
# Publication contract: fact updates, new facts and registry-only
# ------------------------------------------------------------

accepted_outcome_mask = (
    ai_exception_deduplicated_outcomes_df[
        "signature_decision_applied"
    ]
    .fillna(False)
    .astype(bool)
)

fact_publishable_mask = (
    ai_exception_deduplicated_outcomes_df.get(
        "fact_publishable",
        pd.Series(
            False,
            index=ai_exception_deduplicated_outcomes_df.index,
        ),
    )
    .fillna(False)
    .astype(bool)
)

granularity = (
    ai_exception_deduplicated_outcomes_df.get(
        "source_record_granularity",
        pd.Series(
            "UNKNOWN",
            index=ai_exception_deduplicated_outcomes_df.index,
            dtype="string",
        ),
    )
    .astype("string")
)

row_level_mask = granularity.isin(
    [
        "PRODUCTION_FACT",
        "ROW_LEVEL_REVIEW",
    ]
)

has_existing_fact_lineage = (
    ai_exception_deduplicated_outcomes_df.get(
        "production_source_region",
        pd.Series(pd.NA, index=ai_exception_deduplicated_outcomes_df.index),
    ).notna()
    & ai_exception_deduplicated_outcomes_df.get(
        "production_source_table",
        pd.Series(pd.NA, index=ai_exception_deduplicated_outcomes_df.index),
    ).notna()
    & pd.to_numeric(
        ai_exception_deduplicated_outcomes_df.get(
            "production_source_row_number",
            pd.Series(np.nan, index=ai_exception_deduplicated_outcomes_df.index),
        ),
        errors="coerce",
    ).notna()
)

accepted_fact_mask = (
    accepted_outcome_mask
    & fact_publishable_mask
    & row_level_mask
)

ai_accepted_fact_enrichments_df = (
    ai_exception_deduplicated_outcomes_df.loc[
        accepted_fact_mask
    ]
    .copy()
    .reset_index(drop=True)
)

ai_accepted_existing_fact_updates_df = (
    ai_exception_deduplicated_outcomes_df.loc[
        accepted_fact_mask
        & has_existing_fact_lineage
    ]
    .copy()
    .reset_index(drop=True)
)

ai_accepted_new_fact_candidates_df = (
    ai_exception_deduplicated_outcomes_df.loc[
        accepted_fact_mask
        & ~has_existing_fact_lineage
    ]
    .copy()
    .reset_index(drop=True)
)

ai_accepted_registry_only_df = (
    ai_exception_deduplicated_outcomes_df.loc[
        accepted_outcome_mask
        & ~accepted_fact_mask
    ]
    .copy()
    .reset_index(drop=True)
)

ai_exception_audit_outcomes_df = (
    ai_exception_deduplicated_outcomes_df.copy()
)

# A compact, explicit downstream contract for Block 10.
block9_publication_contract_df = pd.DataFrame(
    {
        "output_class": [
            "accepted_existing_fact_updates",
            "accepted_new_fact_candidates",
            "accepted_registry_only",
            "all_accepted_outcomes",
        ],
        "rows": [
            len(ai_accepted_existing_fact_updates_df),
            len(ai_accepted_new_fact_candidates_df),
            len(ai_accepted_registry_only_df),
            int(accepted_outcome_mask.sum()),
        ],
    }
)

classified_accepted_rows = (
    len(ai_accepted_existing_fact_updates_df)
    + len(ai_accepted_new_fact_candidates_df)
    + len(ai_accepted_registry_only_df)
)

if classified_accepted_rows != int(accepted_outcome_mask.sum()):
    raise RuntimeError(
        "Accepted Block 9 outcomes were not classified exactly once "
        "across the publication contract."
    )


# ------------------------------------------------------------
# Outcome summary
# ------------------------------------------------------------

ai_outcome_summary_df = (
    ai_enrichment_decisions_df.groupby(
        [
            "exception_type",
            "ai_decision",
            "acceptance_status",
            "publication_status",
        ],
        dropna=False,
    )
    .agg(
        signature_count=(
            "issue_signature",
            "size",
        ),
        represented_raw_rows=(
            "represented_raw_rows",
            "sum",
        ),
    )
    .reset_index()
    if not ai_enrichment_decisions_df.empty
    else pd.DataFrame()
)


# ------------------------------------------------------------
# Completion registry
# ------------------------------------------------------------

current_completion_rows = []

audit_status_lookup = {}

if not ai_model_audit_log_df.empty:
    for (
        issue_signature,
        prompt_hash,
    ), group in ai_model_audit_log_df.groupby(
        ["issue_signature", "prompt_hash"],
        dropna=False,
    ):
        statuses = set(
            group["status"].dropna().astype(str)
        )

        if "SUCCEEDED" in statuses:
            completion_status = "SUCCEEDED"
        elif "CACHE_HIT" in statuses:
            completion_status = "CACHE_HIT"
        else:
            completion_status = "FAILED"

        audit_status_lookup[
            (str(issue_signature), str(prompt_hash))
        ] = completion_status

for _, row in ai_enrichment_decisions_df.iterrows():
    exception_type = (
        safe_text(row.get("exception_type"))
        or "UNKNOWN"
    )
    prompt_hash = (
        safe_text(row.get("prompt_hash"))
        or ""
    )
    completion_status = audit_status_lookup.get(
        (
            str(row.get("issue_signature")),
            prompt_hash,
        ),
        "SUCCEEDED",
    )

    current_completion_rows.append(
        {
            "issue_signature":
                row.get("issue_signature"),
            "exception_type":
                exception_type,
            "processing_key":
                current_processing_key(
                    exception_type
                ),
            "processing_generation":
                AI_PROCESSING_GENERATION,
            "provider":
                row.get("model_provider"),
            "deployment":
                row.get("model_deployment"),
            "model":
                row.get("model_name"),
            "schema_name":
                RESPONSE_SCHEMA_BY_TASK[
                    exception_type
                ].__name__,
            "prompt_hash":
                prompt_hash,
            "completed_at_utc":
                row.get("processed_at_utc"),
            "ai_decision":
                row.get("ai_decision"),
            "ai_confidence":
                row.get("ai_confidence"),
            "acceptance_status":
                row.get("acceptance_status"),
            "publication_status":
                row.get("publication_status"),
            "completion_status":
                completion_status,
            "completion_method":
                "DIRECT_AZURE",
            "prototype_issue_signature":
                row.get("issue_signature"),
            "cluster_id": (
                safe_text(
                    selected_ai_exceptions_df.loc[
                        selected_ai_exceptions_df[
                            "issue_signature"
                        ].astype(str).eq(
                            str(
                                row.get(
                                    "issue_signature"
                                )
                            )
                        ),
                        "deterministic_cluster_id",
                    ].iloc[0]
                )
                if (
                    "deterministic_cluster_id"
                    in selected_ai_exceptions_df.columns
                    and (
                        selected_ai_exceptions_df[
                            "issue_signature"
                        ].astype(str).eq(
                            str(
                                row.get(
                                    "issue_signature"
                                )
                            )
                        )
                    ).any()
                )
                else None
            ),
            "cluster_similarity_score":
                1.0,
        }
    )

for _, row in propagated_signature_decisions_df.iterrows():
    signature = safe_text(
        row.get("issue_signature")
    )
    source_row = (
        remaining_ai_work_queue_df.loc[
            remaining_ai_work_queue_df[
                "issue_signature"
            ].astype(str).eq(str(signature))
        ]
    )

    exception_type = (
        safe_text(
            source_row[
                "exception_type"
            ].iloc[0]
        )
        if len(source_row)
        else "ACCOUNT_MAPPING"
    )

    current_completion_rows.append(
        {
            "issue_signature":
                signature,
            "exception_type":
                exception_type,
            "processing_key":
                current_processing_key(
                    exception_type
                ),
            "processing_generation":
                AI_PROCESSING_GENERATION,
            "provider":
                "Cluster propagation",
            "deployment":
                pd.NA,
            "model":
                pd.NA,
            "schema_name":
                RESPONSE_SCHEMA_BY_TASK[
                    exception_type
                ].__name__,
            "prompt_hash":
                pd.NA,
            "completed_at_utc":
                utc_now(),
            "ai_decision":
                "CLUSTER_PROPAGATED",
            "ai_confidence":
                row.get("ai_confidence"),
            "acceptance_status":
                "AUTO_ACCEPTED",
            "publication_status":
                "ACCEPTED",
            "completion_status":
                "PROPAGATED",
            "completion_method":
                row.get("completion_method"),
            "prototype_issue_signature":
                row.get(
                    "prototype_issue_signature"
                ),
            "cluster_id":
                row.get("cluster_id"),
            "cluster_similarity_score":
                row.get(
                    "cluster_similarity_score"
                ),
        }
    )

completed_ai_signatures_current_df = pd.DataFrame(
    current_completion_rows,
    columns=COMPLETION_REGISTRY_COLUMNS,
)

completed_ai_signatures_df = (
    pd.concat(
        [
            completed_ai_signatures_prior_df[
                COMPLETION_REGISTRY_COLUMNS
            ],
            completed_ai_signatures_current_df[
                COMPLETION_REGISTRY_COLUMNS
            ],
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        [
            "issue_signature",
            "processing_key",
            "completion_method",
        ],
        keep="last",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Update accepted synonym registry from represented lineage
# ------------------------------------------------------------

accepted_signature_map_df = (
    accepted_ai_enrichments_df[
        [
            column
            for column in [
                "issue_signature",
                "proposed_standard_concept",
                "standard_concept",
                "statement_type",
                "reporting_scope",
                "processed_at_utc",
            ]
            if column
            in accepted_ai_enrichments_df.columns
        ]
    ]
    .copy()
)

if not accepted_signature_map_df.empty:
    accepted_signature_map_df[
        "resolved_standard_concept"
    ] = accepted_signature_map_df[
        "proposed_standard_concept"
    ].where(
        accepted_signature_map_df[
            "proposed_standard_concept"
        ].notna(),
        accepted_signature_map_df[
            "standard_concept"
        ],
    ).map(normalise_identifier)

    registry_lineage_base_df = (
        deduplicated_for_propagation.copy()
    )

    if (
        "issue_signature"
        not in registry_lineage_base_df.columns
    ):
        registry_lineage_base_df[
            "issue_signature"
        ] = registry_lineage_base_df.apply(
            make_issue_signature,
            axis=1,
        )

    lineage_for_registry = (
        registry_lineage_base_df.merge(
            accepted_signature_map_df[
                [
                    "issue_signature",
                    "resolved_standard_concept",
                    "processed_at_utc",
                ]
            ],
            on="issue_signature",
            how="inner",
            validate="many_to_one",
        )
    )

    lineage_for_registry[
        "normalised_source_account_label"
    ] = lineage_for_registry[
        "source_account_label"
    ].map(canonicalise_source_label)

    accepted_synonym_rows = []

    for keys, group in (
        lineage_for_registry.dropna(
            subset=[
                "normalised_source_account_label",
                "resolved_standard_concept",
            ]
        )
        .groupby(
            [
                "normalised_source_account_label",
                "resolved_standard_concept",
                "statement_type",
                "reporting_scope",
            ],
            dropna=False,
        )
    ):
        (
            normalised_label,
            standard_concept,
            statement_type,
            reporting_scope,
        ) = keys

        accepted_synonym_rows.append(
            {
                "normalised_source_account_label":
                    normalised_label,
                "standard_concept":
                    standard_concept,
                "statement_type":
                    statement_type,
                "reporting_scope":
                    reporting_scope,
                "accepted_observations":
                    len(group),
                "unique_issuers":
                    group.get(
                        "global_issuer_id",
                        pd.Series(dtype="string"),
                    ).nunique(dropna=True),
                "unique_filings":
                    group.get(
                        "filing_id",
                        pd.Series(dtype="string"),
                    ).nunique(dropna=True),
                "first_accepted_at_utc":
                    group.get(
                        "processed_at_utc",
                        pd.Series(dtype="string"),
                    ).min(),
                "last_accepted_at_utc":
                    group.get(
                        "processed_at_utc",
                        pd.Series(dtype="string"),
                    ).max(),
                "registry_generation":
                    AI_PROCESSING_GENERATION,
            }
        )
else:
    accepted_synonym_rows = []

accepted_synonym_registry_current_df = pd.DataFrame(
    accepted_synonym_rows,
    columns=ACCEPTED_SYNONYM_REGISTRY_COLUMNS,
)

registry_combined = pd.concat(
    [
        accepted_synonym_registry_prior_df[
            ACCEPTED_SYNONYM_REGISTRY_COLUMNS
        ],
        accepted_synonym_registry_current_df[
            ACCEPTED_SYNONYM_REGISTRY_COLUMNS
        ],
    ],
    ignore_index=True,
)

if registry_combined.empty:
    accepted_synonym_registry_df = pd.DataFrame(
        columns=ACCEPTED_SYNONYM_REGISTRY_COLUMNS
    )
else:
    for numeric_column in [
        "accepted_observations",
        "unique_issuers",
        "unique_filings",
    ]:
        registry_combined[
            numeric_column
        ] = safe_numeric(
            registry_combined[
                numeric_column
            ]
        ).fillna(0)

    accepted_synonym_registry_df = (
        registry_combined.groupby(
            [
                "normalised_source_account_label",
                "standard_concept",
                "statement_type",
                "reporting_scope",
            ],
            dropna=False,
            as_index=False,
        )
        .agg(
            accepted_observations=(
                "accepted_observations",
                "sum",
            ),
            unique_issuers=(
                "unique_issuers",
                "max",
            ),
            unique_filings=(
                "unique_filings",
                "max",
            ),
            first_accepted_at_utc=(
                "first_accepted_at_utc",
                "min",
            ),
            last_accepted_at_utc=(
                "last_accepted_at_utc",
                "max",
            ),
            registry_generation=(
                "registry_generation",
                "last",
            ),
        )
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# Batch registry
# ------------------------------------------------------------

batch_id = stable_hash(
    "||".join(
        [
            AI_PROCESSING_GENERATION,
            utc_now().isoformat(),
            str(len(selected_ai_exceptions_df)),
        ]
    )
)

estimated_cost_usd = float(
    pd.to_numeric(
        ai_model_audit_log_df.get(
            "estimated_cost_usd",
            pd.Series(dtype=float),
        ),
        errors="coerce",
    )
    .fillna(0)
    .sum()
)

batch_row = {
    "batch_id": batch_id,
    "processing_generation":
        AI_PROCESSING_GENERATION,
    "generated_at_utc":
        utc_now().isoformat(),
    "selected_azure_prototypes":
        len(selected_ai_exceptions_df),
    "direct_accepted_signatures":
        len(accepted_ai_enrichments_df),
    "cluster_propagated_signatures":
        len(propagated_signature_decisions_df),
    "rejected_signatures":
        len(rejected_ai_enrichments_df),
    "unresolved_signatures":
        len(unresolved_ai_enrichments_df),
    "estimated_cost_usd":
        estimated_cost_usd,
    "pipeline_stopped_early":
        pipeline_stopped_early,
    "pipeline_stop_reason":
        pipeline_stop_reason,
}

if BATCH_REGISTRY_PATH.exists():
    try:
        ai_batch_registry_prior_df = (
            pd.read_parquet(BATCH_REGISTRY_PATH)
        )
    except Exception:
        ai_batch_registry_prior_df = pd.DataFrame()
else:
    ai_batch_registry_prior_df = pd.DataFrame()

ai_batch_registry_df = (
    pd.concat(
        [
            ai_batch_registry_prior_df,
            pd.DataFrame([batch_row]),
        ],
        ignore_index=True,
    )
    .drop_duplicates("batch_id", keep="last")
    .reset_index(drop=True)
)

display(ai_outcome_summary_df)

print(
    "Directly accepted signatures:",
    len(accepted_ai_enrichments_df),
)
print(
    "Cluster-propagated signatures:",
    len(propagated_signature_decisions_df),
)
print(
    "Rejected signatures:",
    len(rejected_ai_enrichments_df),
)
print(
    "Unresolved direct signatures:",
    len(unresolved_ai_enrichments_df),
)
print(
    "Directly enriched deduplicated rows:",
    int(
        ai_exception_deduplicated_outcomes_df.get(
            "signature_decision_direct",
            pd.Series(
                False,
                index=ai_exception_deduplicated_outcomes_df.index,
            ),
        )
        .fillna(False)
        .sum()
    ),
)

print(
    "Propagated deduplicated rows:",
    int(
        ai_exception_deduplicated_outcomes_df.get(
            "signature_decision_propagated",
            pd.Series(
                False,
                index=ai_exception_deduplicated_outcomes_df.index,
            ),
        )
        .fillna(False)
        .sum()
    ),
)
print(
    "Raw lineage rows:",
    len(ai_exception_raw_lineage_outcomes_df),
)
print(
    "Deduplicated production rows:",
    len(ai_exception_deduplicated_outcomes_df),
)


In [ ]:
# ============================================================
# 19. FINAL QA DASHBOARD AND DEVELOPMENT DIAGNOSTICS
# ============================================================

def audit_sum(column: str) -> float:
    if (
        ai_model_audit_log_df.empty
        or column not in ai_model_audit_log_df
    ):
        return 0.0

    return float(
        pd.to_numeric(
            ai_model_audit_log_df[column],
            errors="coerce",
        )
        .fillna(0)
        .sum()
    )


def audit_mean(column: str) -> float:
    if (
        ai_model_audit_log_df.empty
        or column not in ai_model_audit_log_df
    ):
        return 0.0

    values = pd.to_numeric(
        ai_model_audit_log_df.loc[
            ~ai_model_audit_log_df[
                "cache_hit"
            ].fillna(False),
            column,
        ],
        errors="coerce",
    ).dropna()

    return (
        float(values.mean())
        if len(values)
        else 0.0
    )


status_counts = (
    ai_enrichment_decisions_df[
        "acceptance_status"
    ].value_counts()
    if not ai_enrichment_decisions_df.empty
    else pd.Series(dtype=int)
)

publication_counts = (
    ai_enrichment_decisions_df[
        "publication_status"
    ].value_counts()
    if not ai_enrichment_decisions_df.empty
    else pd.Series(dtype=int)
)

azure_requests = (
    int(
        (
            ai_model_audit_log_df
            .get(
                "provider",
                pd.Series(dtype=str),
            )
            .eq("Azure OpenAI")
            &
            ~ai_model_audit_log_df
            .get(
                "cache_hit",
                pd.Series(dtype=bool),
            )
            .fillna(False)
        )
        .sum()
    )
    if not ai_model_audit_log_df.empty
    else 0
)

cache_hits = (
    int(
        ai_model_audit_log_df
        .get(
            "cache_hit",
            pd.Series(dtype=bool),
        )
        .fillna(False)
        .sum()
    )
    if not ai_model_audit_log_df.empty
    else 0
)


successful_request_audit_df = (
    ai_model_audit_log_df.loc[
        ai_model_audit_log_df[
            "status"
        ].isin(
            [
                "SUCCEEDED",
                "CACHE_HIT",
            ]
        )
    ]
    .copy()
    if not ai_model_audit_log_df.empty
    else pd.DataFrame()
)

newly_billed_audit_df = (
    successful_request_audit_df.loc[
        ~successful_request_audit_df[
            "cache_hit"
        ].fillna(False)
    ]
    .copy()
    if not successful_request_audit_df.empty
    else pd.DataFrame()
)

def numeric_mean(
    frame: pd.DataFrame,
    column: str,
) -> float:
    if frame.empty or column not in frame.columns:
        return 0.0

    values = pd.to_numeric(
        frame[column],
        errors="coerce",
    ).dropna()

    return (
        float(values.mean())
        if len(values)
        else 0.0
    )


average_input_tokens = numeric_mean(
    newly_billed_audit_df,
    "prompt_token_count",
)
average_output_tokens = numeric_mean(
    newly_billed_audit_df,
    "candidate_token_count",
)
average_total_tokens = numeric_mean(
    newly_billed_audit_df,
    "total_token_count",
)
average_cost_per_signature_usd = numeric_mean(
    newly_billed_audit_df,
    "estimated_cost_usd",
)

resolved_this_run_signatures = (
    set(
        ai_enrichment_decisions_df.loc[
            ai_enrichment_decisions_df[
                "publication_status"
            ].eq("ACCEPTED"),
            "issue_signature",
        ].astype(str)
    )
    | set(
        propagated_signature_decisions_df.get(
            "issue_signature",
            pd.Series(dtype="string"),
        ).dropna().astype(str)
    )
)

remaining_signature_count = max(
    0,
    len(remaining_ai_work_queue_df)
    - len(resolved_this_run_signatures),
)

projected_remaining_cost_usd = (
    average_cost_per_signature_usd
    * remaining_signature_count
)

average_latency_seconds = numeric_mean(
    newly_billed_audit_df,
    "latency_seconds",
)

projected_remaining_runtime_hours = (
    average_latency_seconds
    * remaining_signature_count
    / 3600
)

# ------------------------------------------------------------
# Cumulative completion metrics
# ------------------------------------------------------------

eligible_mask_for_progress = (
    ai_exception_signature_queue_df.get(
        "ai_eligible",
        pd.Series(
            False,
            index=ai_exception_signature_queue_df.index,
        ),
    )
    .fillna(False)
    .astype(bool)
)

eligible_signature_set = set(
    ai_exception_signature_queue_df.loc[
        eligible_mask_for_progress,
        "issue_signature",
    ]
    .dropna()
    .astype("string")
)

completed_signature_set = set()

if (
    isinstance(
        completed_ai_signatures_df,
        pd.DataFrame,
    )
    and not completed_ai_signatures_df.empty
    and "issue_signature"
    in completed_ai_signatures_df.columns
):
    completed_signature_set = set(
        completed_ai_signatures_df[
            "issue_signature"
        ]
        .dropna()
        .astype("string")
    )

current_processed_signature_set = set(
    ai_enrichment_decisions_df.get(
        "issue_signature",
        pd.Series(dtype="string"),
    )
    .dropna()
    .astype("string")
)

completed_signature_set = (
    completed_signature_set
    | current_processed_signature_set
)

total_ai_eligible_signatures = len(
    eligible_signature_set
)

completed_ai_eligible_signatures = len(
    eligible_signature_set
    & completed_signature_set
)

remaining_ai_eligible_signatures = max(
    total_ai_eligible_signatures
    - completed_ai_eligible_signatures,
    0,
)

ai_completion_percentage = (
    100.0
    * completed_ai_eligible_signatures
    / total_ai_eligible_signatures
    if total_ai_eligible_signatures
    else 0.0
)

remaining_ai_percentage = max(
    100.0 - ai_completion_percentage,
    0.0,
)

qa_dashboard_df = pd.DataFrame(
    [
        {
            "metric": "total_exceptions",
            "value": len(
                ai_exception_queue_raw_df
            ),
        },
        {
            "metric": "rows_after_exact_deduplication",
            "value": len(
                ai_exception_queue_deduplicated_df
            ),
        },
        {
            "metric": "representative_signatures",
            "value": len(
                ai_exception_signature_queue_df
            ),
        },
        {
            "metric": "selected_ai_signatures",
            "value": len(
                selected_ai_exceptions_df
            ),
        },
        {
            "metric": "azure_requests",
            "value": azure_requests,
        },
        {
            "metric": "cache_hits",
            "value": cache_hits,
        },
        {
            "metric": "cached_excluded_before_selection",
            "value": len(
                cached_but_unregistered_df
            ),
        },
        {
            "metric": "fresh_azure_requests_selected",
            "value": len(
                selected_ai_exceptions_df
            ),
        },
        {
            "metric": "fresh_batch_cache_hit_rate",
            "value": (
                cache_hits
                / len(selected_ai_exceptions_df)
                if len(
                    selected_ai_exceptions_df
                )
                else 0.0
            ),
        },
        {
            "metric": "accepted",
            "value": int(
                publication_counts.get(
                    "ACCEPTED",
                    0,
                )
            ),
        },
        {
            "metric": "rejected",
            "value": int(
                publication_counts.get(
                    "REJECTED",
                    0,
                )
            ),
        },
        {
            "metric": "unresolved",
            "value": int(
                publication_counts.get(
                    "UNRESOLVED",
                    0,
                )
            ),
        },
        {
            "metric": "low_confidence_proposals",
            "value": int(
                status_counts.get(
                    "LOW_CONFIDENCE_PROPOSAL",
                    0,
                )
            ),
        },
        {
            "metric": "model_unresolved",
            "value": int(
                status_counts.get(
                    "MODEL_UNRESOLVED",
                    0,
                )
            ),
        },
        {
            "metric": "model_rejected",
            "value": int(
                status_counts.get(
                    "MODEL_REJECTED",
                    0,
                )
            ),
        },
        {
            "metric": "provider_failed",
            "value": int(
                status_counts.get(
                    "PROVIDER_FAILED",
                    0,
                )
            ),
        },
        {
            "metric": "directly_enriched_deduplicated_rows",
            "value": int(
                ai_exception_deduplicated_outcomes_df.get(
                    "signature_decision_direct",
                    pd.Series(
                        False,
                        index=ai_exception_deduplicated_outcomes_df.index,
                    ),
                )
                .fillna(False)
                .sum()
            ),
        },
        {
            "metric": "propagated_deduplicated_rows",
            "value": int(
                ai_exception_deduplicated_outcomes_df.get(
                    "signature_decision_propagated",
                    pd.Series(
                        False,
                        index=ai_exception_deduplicated_outcomes_df.index,
                    ),
                )
                .fillna(False)
                .sum()
            ),
        },
        {
            "metric": "direct_accepted_signatures",
            "value": len(
                accepted_ai_enrichments_df
            ),
        },
        {
            "metric": "high_certainty_phrase_acceptances",
            "value": int(
                (
                    ai_enrichment_decisions_df.get(
                        "mapping_basis",
                        pd.Series(dtype="string"),
                    )
                    .eq("HIGH_CERTAINTY_PHRASE_MAPPING")
                    & ai_enrichment_decisions_df.get(
                        "publication_status",
                        pd.Series(dtype="string"),
                    ).eq("ACCEPTED")
                ).sum()
            ),
        },
        {
            "metric": "historical_synonym_acceptances",
            "value": int(
                (
                    ai_enrichment_decisions_df.get(
                        "mapping_basis",
                        pd.Series(dtype="string"),
                    )
                    .eq(
                        "HISTORICAL_SYNONYM_MAPPING"
                    )
                    & ai_enrichment_decisions_df.get(
                        "publication_status",
                        pd.Series(dtype="string"),
                    ).eq("ACCEPTED")
                ).sum()
            ),
        },
        {
            "metric": "stock_flow_compatibility_failures",
            "value": int(
                ai_enrichment_decisions_df.get(
                    "stock_flow_compatibility_gate_passed",
                    pd.Series(dtype=bool),
                )
                .fillna(True)
                .eq(False)
                .sum()
            ),
        },
        {
            "metric": "fragment_quality_failures",
            "value": int(
                ai_enrichment_decisions_df.get(
                    "fragment_quality_gate_passed",
                    pd.Series(dtype=bool),
                )
                .fillna(False)
                .eq(False)
                .sum()
            ),
        },
        {
            "metric": "synonym_registry_rows",
            "value": len(
                accepted_synonym_registry_df
            ),
        },
        {
            "metric": "ai_eligible_signatures_total",
            "value": total_ai_eligible_signatures,
        },
        {
            "metric": "ai_eligible_signatures_completed",
            "value": completed_ai_eligible_signatures,
        },
        {
            "metric": "ai_eligible_signatures_remaining",
            "value": remaining_ai_eligible_signatures,
        },
        {
            "metric": "ai_completion_percentage",
            "value": ai_completion_percentage,
        },
        {
            "metric": "cluster_propagated_signatures",
            "value": len(
                propagated_signature_decisions_df
            ),
        },
        {
            "metric": "remaining_ai_clusters",
            "value": int(
                remaining_ai_work_queue_df[
                    "deterministic_cluster_id"
                ].nunique()
            ),
        },
        {
            "metric": "estimated_azure_cost_usd",
            "value": audit_sum(
                "estimated_cost_usd"
            ),
        },
        {
            "metric": "average_latency_seconds",
            "value": audit_mean(
                "latency_seconds"
            ),
        },
        {
            "metric": "average_input_tokens",
            "value": average_input_tokens,
        },
        {
            "metric": "average_output_tokens",
            "value": average_output_tokens,
        },
        {
            "metric": "average_total_tokens",
            "value": average_total_tokens,
        },
        {
            "metric": "average_cost_per_signature_usd",
            "value": average_cost_per_signature_usd,
        },
        {
            "metric": "remaining_ai_signatures_before_cluster_propagation",
            "value": remaining_signature_count,
        },
        {
            "metric": "projected_remaining_cost_usd",
            "value": projected_remaining_cost_usd,
        },
        {
            "metric": "projected_remaining_runtime_hours",
            "value": projected_remaining_runtime_hours,
        },
    ]
)

publication_dashboard_rows = pd.DataFrame(
    [
        {
            "metric": "accepted_existing_fact_updates",
            "value": len(ai_accepted_existing_fact_updates_df),
        },
        {
            "metric": "accepted_new_fact_candidates",
            "value": len(ai_accepted_new_fact_candidates_df),
        },
        {
            "metric": "accepted_registry_only",
            "value": len(ai_accepted_registry_only_df),
        },
        {
            "metric": "accepted_outcomes_classified",
            "value": (
                len(ai_accepted_existing_fact_updates_df)
                + len(ai_accepted_new_fact_candidates_df)
                + len(ai_accepted_registry_only_df)
            ),
        },
    ]
)

qa_dashboard_df = pd.concat(
    [
        qa_dashboard_df,
        publication_dashboard_rows,
    ],
    ignore_index=True,
)

pipeline_diagnostics_df = qa_dashboard_df.copy()

if not ai_model_audit_log_df.empty:
    ai_error_summary_df = (
        ai_model_audit_log_df
        .groupby(
            [
                "provider",
                "deployment",
                "model",
                "exception_type",
                "status",
                "error_category",
            ],
            dropna=False,
        )
        .size()
        .reset_index(
            name="attempt_count"
        )
    )
else:
    ai_error_summary_df = pd.DataFrame()


def failed_gate_labels(row: pd.Series) -> str:
    """Return failed gates that are relevant to automatic publication."""
    gate_map = {
        "confidence_gate_passed":
            "confidence",
        "evidence_gate_passed":
            "evidence",
        "non_null_proposal_gate_passed":
            "proposal-present",
        "statement_compatibility_gate_passed":
            "statement-compatibility",
        "canonical_dictionary_gate_passed":
            "canonical-dictionary",
    }

    failed = []

    for column, label in gate_map.items():
        if column not in row.index:
            continue

        value = row[column]

        if pd.isna(value) or not bool(value):
            failed.append(label)

    return (
        ", ".join(failed)
        if failed
        else "none"
    )


def derive_diagnostic_case_type(
    row: pd.Series,
) -> str:
    status = safe_text(
        row.get("acceptance_status")
    )

    basis = safe_text(
        row.get("mapping_basis")
    )

    decision = safe_text(
        row.get("normalised_ai_decision")
    )

    if status == "PRE_AI_EXCLUDED":
        return "PRE_AI_FRAGMENT_EXCLUDED"

    if status == "AUTO_ACCEPTED":
        return basis or "AUTO_ACCEPTED"

    if status == "LOW_CONFIDENCE_PROPOSAL":
        return (
            (basis or "POSITIVE_PROPOSAL")
            + "_BELOW_THRESHOLD"
        )

    if status == "MODEL_REJECTED":
        return "DETERMINISTICALLY_SUPPORTED_REJECTION"

    if status == "PROVIDER_FAILED":
        return "PROVIDER_FAILURE"

    if status == "MODEL_UNRESOLVED":
        if decision in {
            "REJECT",
            "REJECT_FACT",
        }:
            return "UNSUPPORTED_MODEL_REJECTION"

        if decision == "UNRESOLVED":
            return "MODEL_DECLARED_UNRESOLVED"

        return "FAILED_DETERMINISTIC_GATES"

    return "OTHER"


case_diagnostics_df = (
    ai_enrichment_decisions_df.copy()
)

if not case_diagnostics_df.empty:
    case_diagnostics_df["failed_gates"] = (
        case_diagnostics_df.apply(
            failed_gate_labels,
            axis=1,
        )
    )

    case_diagnostics_df[
        "diagnostic_case_type"
    ] = (
        case_diagnostics_df.apply(
            derive_diagnostic_case_type,
            axis=1,
        )
    )

    case_type_summary_df = (
        case_diagnostics_df
        .groupby(
            [
                "exception_type",
                "publication_status",
                "acceptance_status",
                "diagnostic_case_type",
            ],
            dropna=False,
        )
        .agg(
            representative_cases=(
                "issue_signature",
                "size",
            ),
            represented_raw_rows=(
                "represented_raw_rows",
                "sum",
            ),
            average_confidence=(
                "ai_confidence",
                "mean",
            ),
            minimum_confidence=(
                "ai_confidence",
                "min",
            ),
            maximum_confidence=(
                "ai_confidence",
                "max",
            ),
        )
        .reset_index()
        .sort_values(
            [
                "representative_cases",
                "represented_raw_rows",
            ],
            ascending=False,
        )
    )

    decision_reason_summary_df = (
        case_diagnostics_df
        .groupby(
            [
                "publication_status",
                "acceptance_status",
                "acceptance_reason",
            ],
            dropna=False,
        )
        .agg(
            representative_cases=(
                "issue_signature",
                "size",
            ),
            represented_raw_rows=(
                "represented_raw_rows",
                "sum",
            ),
            average_confidence=(
                "ai_confidence",
                "mean",
            ),
        )
        .reset_index()
        .sort_values(
            [
                "representative_cases",
                "represented_raw_rows",
            ],
            ascending=False,
        )
    )
else:
    case_type_summary_df = pd.DataFrame()
    decision_reason_summary_df = pd.DataFrame()




# ------------------------------------------------------------
# Persistent AI completion progress
# ------------------------------------------------------------

successful_completion_statuses = {
    "SUCCEEDED",
    "CACHE_HIT",
    "PROPAGATED",
}

current_generation_completion_df = (
    completed_ai_signatures_df.loc[
        completed_ai_signatures_df[
            "processing_generation"
        ].astype(str).eq(
            AI_PROCESSING_GENERATION
        )
        & completed_ai_signatures_df[
            "completion_status"
        ].astype(str).isin(
            successful_completion_statuses
        )
    ]
    .copy()
)

completed_ai_eligible_signatures = int(
    current_generation_completion_df[
        "issue_signature"
    ].dropna().astype(str).nunique()
)

total_ai_eligible_signatures = int(
    eligible_signatures_df[
        "issue_signature"
    ].dropna().astype(str).nunique()
)

remaining_ai_eligible_signatures = max(
    0,
    total_ai_eligible_signatures
    - completed_ai_eligible_signatures,
)

ai_completion_percentage = (
    100.0
    * completed_ai_eligible_signatures
    / total_ai_eligible_signatures
    if total_ai_eligible_signatures
    else 100.0
)

ai_completion_progress_df = pd.DataFrame(
    [
        {
            "processing_generation":
                AI_PROCESSING_GENERATION,
            "ai_eligible_signatures":
                total_ai_eligible_signatures,
            "completed_signatures":
                completed_ai_eligible_signatures,
            "remaining_signatures":
                remaining_ai_eligible_signatures,
            "completion_percentage":
                ai_completion_percentage,
        }
    ]
)


if (
    "pre_ai_fragment_reason"
    not in ai_exception_signature_queue_df.columns
):
    ai_exception_signature_queue_df[
        "pre_ai_fragment_reason"
    ] = ai_exception_signature_queue_df[
        "source_account_label"
    ].map(pre_ai_fragment_reason)

if (
    "pre_ai_fragment_excluded"
    not in ai_exception_signature_queue_df.columns
):
    ai_exception_signature_queue_df[
        "pre_ai_fragment_excluded"
    ] = ai_exception_signature_queue_df[
        "pre_ai_fragment_reason"
    ].notna()

pre_ai_fragment_rows_df = (
    ai_exception_signature_queue_df.loc[
        ai_exception_signature_queue_df[
            "pre_ai_fragment_excluded"
        ]
        .fillna(False)
        .astype(bool)
    ]
    .copy()
)

if pre_ai_fragment_rows_df.empty:
    pre_ai_fragment_summary_df = pd.DataFrame(
        columns=[
            "pre_ai_fragment_reason",
            "representative_signatures",
            "represented_raw_rows",
        ]
    )
else:
    if (
        "represented_raw_rows"
        not in pre_ai_fragment_rows_df.columns
    ):
        pre_ai_fragment_rows_df[
            "represented_raw_rows"
        ] = 1

    pre_ai_fragment_summary_df = (
        pre_ai_fragment_rows_df
        .groupby(
            "pre_ai_fragment_reason",
            dropna=False,
        )
        .agg(
            representative_signatures=(
                "issue_signature",
                "size",
            ),
            represented_raw_rows=(
                "represented_raw_rows",
                "sum",
            ),
        )
        .reset_index()
        .sort_values(
            [
                "representative_signatures",
                "represented_raw_rows",
            ],
            ascending=False,
        )
    )


acceptance_policy_summary_df = pd.DataFrame({
    "mapping_basis": [
        "IDENTITY_MAPPING",
        "EXACT_LABEL_MAPPING",
        "KNOWN_SYNONYM_MAPPING",
        "HISTORICAL_SYNONYM_MAPPING",
        "HIGH_CERTAINTY_PHRASE_MAPPING",
        "KEEP_SOURCE",
        "MODEL_INFERRED_MAPPING",
        "PROPOSE_CORRECTION",
    ],
    "automatic_acceptance_threshold": [
        IDENTITY_MAPPING_THRESHOLD,
        EXACT_LABEL_MAPPING_THRESHOLD,
        KNOWN_SYNONYM_MAPPING_THRESHOLD,
        HISTORICAL_SYNONYM_THRESHOLD,
        HIGH_CERTAINTY_PHRASE_THRESHOLD,
        KEEP_SOURCE_THRESHOLD,
        MODEL_INFERRED_MAPPING_THRESHOLD,
        CORRECTION_THRESHOLD,
    ],
    "all_deterministic_gates_still_required": [
        True,
        True,
        True,
        True,
        True,
        True,
        True,
        True,
    ],
    "acceptance_policy_version": [
        ACCEPTANCE_POLICY_VERSION
    ] * 8,
})

cumulative_republication_coverage_df = pd.DataFrame({
    "metric": [
        "historical_cache_rows_loaded",
        "combined_unique_proposals_rebuilt",
        "cumulative_decisions_re_evaluated",
        "current_execution_selected_signatures",
        "new_azure_requests_from_republication",
        "acceptance_policy_version",
    ],
    "value_text": [
        str(len(historical_ai_cache_df)),
        str(len(ai_enrichment_proposals_df)),
        str(len(ai_enrichment_decisions_df)),
        str(len(selected_ai_exceptions_df)),
        "0",
        str(REPUBLICATION_POLICY_VERSION),
    ],
    "value_numeric": [
        float(len(historical_ai_cache_df)),
        float(len(ai_enrichment_proposals_df)),
        float(len(ai_enrichment_decisions_df)),
        float(len(selected_ai_exceptions_df)),
        0.0,
        np.nan,
    ],
})

print("=" * 80)
print("BLOCK 9 HISTORICAL CACHE REPUBLICATION")
print("=" * 80)
display(
    historical_republication_summary_df
)
display(
    cumulative_republication_coverage_df
)

print("=" * 80)
print("BLOCK 9 QA DASHBOARD")
print("=" * 80)
display(qa_dashboard_df)

print("\nBLOCK 9 PUBLICATION CONTRACT")
display(block9_publication_contract_df)

print("\nSIGNATURE COLLAPSE")
display(collapse_summary_df)

print("\nDUPLICATE SOURCE CONTROL")
display(
    duplicate_source_neutralisation_summary_df
)

print("\nPRE-AZURE FRAGMENT EXCLUSIONS")
display(pre_ai_fragment_summary_df)

print("\nACCEPTANCE POLICY")
display(acceptance_policy_summary_df)

print("\nAI PROVIDER AUDIT")
display(ai_error_summary_df)

print("\nAUTOMATED CASE TYPES")
display(case_type_summary_df)

print("\nDECISION REASONS")
display(decision_reason_summary_df)

print("\nDEVELOPMENT DIAGNOSTIC SAMPLE")
diagnostic_columns = [
    column
    for column in [
        "review_record_id",
        "issue_signature",
        "exception_type",
        "region",
        "upstream_block",
        "source_table",
        "queue_source_table",
        "source_record_granularity",
        "production_source_region",
        "production_source_table",
        "production_source_row_number",
        "fact_publishable",
        "represented_raw_rows",
        "global_issuer_id",
        "filing_id",
        "source_account_label",
        "standard_concept",
        "statement_type",
        "ai_decision",
        "normalised_ai_decision",
        "proposed_standard_concept",
        "mapping_basis",
        "ai_confidence",
        "decision_auto_threshold",
        "ai_evidence_text",
        "ai_reasoning_summary",
        "acceptance_status",
        "publication_status",
        "acceptance_reason",
        "diagnostic_case_type",
        "failed_gates",
    ]
    if column in case_diagnostics_df.columns
]

display(
    case_diagnostics_df[
        diagnostic_columns
    ].head(50)
)


print("\n" + "=" * 80)
print("BLOCK 9 PERSISTENT AI COMPLETION PROGRESS")
print("=" * 80)
display(ai_completion_progress_df)

print(
    "Progress: "
    f"{completed_ai_eligible_signatures:,}"
    " / "
    f"{total_ai_eligible_signatures:,}"
    " AI-eligible signatures completed "
    f"({ai_completion_percentage:.2f}%)."
)

print(
    "Remaining AI-eligible signatures:",
    f"{remaining_ai_eligible_signatures:,}",
)


In [ ]:
# ============================================================
# 20. SAVE OUTPUTS AND BUILD PERSISTENCE REPORT
# ============================================================

OUTPUT_TABLES = {
    "upstream_table_inventory":
        upstream_table_inventory_df,
    "duplicate_source_neutralisation_summary":
        duplicate_source_neutralisation_summary_df,
    "collapse_summary":
        collapse_summary_df,
    "ai_exception_signature_queue":
        ai_exception_signature_queue_df,
    "selected_ai_exceptions":
        selected_ai_exceptions_df,
    "cached_but_unregistered":
        cached_but_unregistered_df,
    "remaining_ai_work_queue":
        remaining_ai_work_queue_df,
    "ai_task_funnel":
        ai_task_funnel_df,
    "ai_similarity_clusters":
        ai_similarity_clusters_df,
    "ai_cluster_prototypes":
        ai_cluster_prototypes_df,
    "ai_cluster_propagation_decisions":
        ai_cluster_propagation_decisions_df,
    "ai_cluster_member_outcomes":
        ai_cluster_member_outcomes_df,
    "propagated_signature_decisions":
        propagated_signature_decisions_df,
    "all_accepted_signature_decisions":
        all_accepted_signature_decisions_df,
    "ai_enrichment_proposals":
        ai_enrichment_proposals_df,
    "historical_ai_cache":
        historical_ai_cache_df,
    "historical_ai_enrichment_proposals":
        historical_ai_enrichment_proposals_df,
    "historical_republication_summary":
        historical_republication_summary_df,
    "cumulative_republication_coverage":
        cumulative_republication_coverage_df,
    "current_ai_enrichment_proposals":
        current_ai_enrichment_proposals_df,
    "ai_model_audit_log":
        ai_model_audit_log_df,
    "ai_enrichment_decisions":
        ai_enrichment_decisions_df,
    "accepted_ai_enrichments":
        accepted_ai_enrichments_df,
    "rejected_ai_enrichments":
        rejected_ai_enrichments_df,
    "unresolved_ai_enrichments":
        unresolved_ai_enrichments_df,
    "low_confidence_proposals":
        low_confidence_proposals_df,
    "model_unresolved_ai_enrichments":
        model_unresolved_ai_enrichments_df,
    "provider_failed_ai_enrichments":
        provider_failed_ai_enrichments_df,
    "accepted_signature_decisions":
        accepted_signature_decisions_df,
    "ai_exception_raw_lineage_outcomes":
        ai_exception_raw_lineage_outcomes_df,
    "ai_exception_deduplicated_outcomes":
        ai_exception_deduplicated_outcomes_df,
    "ai_exception_row_level_outcomes":
        ai_exception_row_level_outcomes_df,
    "ai_accepted_fact_enrichments":
        ai_accepted_fact_enrichments_df,
    "ai_accepted_existing_fact_updates":
        ai_accepted_existing_fact_updates_df,
    "ai_accepted_new_fact_candidates":
        ai_accepted_new_fact_candidates_df,
    "ai_accepted_registry_only":
        ai_accepted_registry_only_df,
    "ai_exception_audit_outcomes":
        ai_exception_audit_outcomes_df,
    "block9_publication_contract":
        block9_publication_contract_df,
    "accepted_synonym_registry":
        accepted_synonym_registry_df,
    "ai_completion_progress":
        ai_completion_progress_df,
    "completed_ai_signatures":
        completed_ai_signatures_df,
    "ai_batch_registry":
        ai_batch_registry_df,
    "ai_outcome_summary":
        ai_outcome_summary_df,
    "pipeline_diagnostics":
        pipeline_diagnostics_df,
    "qa_dashboard":
        qa_dashboard_df,
    "provider_validation":
        provider_validation_df,
    "ai_error_summary":
        ai_error_summary_df,
    "case_diagnostics":
        case_diagnostics_df,
    "case_type_summary":
        case_type_summary_df,
    "decision_reason_summary":
        decision_reason_summary_df,
}

CSV_OUTPUT_NAMES = {
    "upstream_table_inventory",
    "selected_ai_exceptions",
    "ai_task_funnel",
    "accepted_ai_enrichments",
    "ai_accepted_existing_fact_updates",
    "ai_accepted_new_fact_candidates",
    "ai_accepted_registry_only",
    "block9_publication_contract",
    "rejected_ai_enrichments",
    "unresolved_ai_enrichments",
    "low_confidence_proposals",
    "pipeline_diagnostics",
    "qa_dashboard",
    "provider_validation",
    "case_type_summary",
    "decision_reason_summary",
}


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def parquet_safe_frame(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    """Return a deterministic PyArrow-compatible copy."""
    result = frame.copy()

    for column in result.columns:
        series = result[column]

        if series.dtype != "object":
            continue

        non_null_types = {
            type(value)
            for value in series.dropna()
        }

        if len(non_null_types) > 1:
            result[column] = series.astype("string")

    return result


persistence_rows = []

for table_name, frame in OUTPUT_TABLES.items():
    if not isinstance(frame, pd.DataFrame):
        persistence_rows.append(
            {
                "table_name": table_name,
                "file_type": None,
                "expected_rows": None,
                "persisted_rows": None,
                "expected_columns": None,
                "persisted_columns": None,
                "file_size_bytes": None,
                "sha256": None,
                "status": "SKIPPED_NOT_DATAFRAME",
                "saved_file": None,
            }
        )
        continue

    expected_rows = len(frame)
    expected_columns = len(frame.columns)

    if SAVE_PARQUET:
        parquet_path = (
            BLOCK_9_OUTPUT_DIR
            / f"{table_name}.parquet"
        )

        try:
            parquet_frame = parquet_safe_frame(
                frame
            )

            parquet_frame.to_parquet(
                parquet_path,
                index=False,
            )

            persisted_frame = pd.read_parquet(
                parquet_path
            )

            persisted_rows = len(
                persisted_frame
            )

            persisted_columns = len(
                persisted_frame.columns
            )

            if (
                persisted_rows == expected_rows
                and persisted_columns == expected_columns
            ):
                status = "PASSED"
            elif persisted_rows != expected_rows:
                status = "ROW_COUNT_MISMATCH"
            else:
                status = "COLUMN_COUNT_MISMATCH"

            file_size_bytes = (
                parquet_path.stat().st_size
            )

            file_hash = sha256_file(
                parquet_path
            )

        except Exception as exc:
            persisted_rows = None
            persisted_columns = None
            file_size_bytes = None
            file_hash = None
            status = (
                "FAILED: "
                + type(exc).__name__
                + ": "
                + str(exc)[:180]
            )

        persistence_rows.append(
            {
                "table_name": table_name,
                "file_type": "parquet",
                "expected_rows": expected_rows,
                "persisted_rows": persisted_rows,
                "expected_columns": expected_columns,
                "persisted_columns": persisted_columns,
                "file_size_bytes": file_size_bytes,
                "sha256": file_hash,
                "status": status,
                "saved_file": str(parquet_path),
            }
        )

    save_csv = (
        SAVE_CSV_SUMMARIES
        and (
            table_name.endswith("_summary")
            or table_name in CSV_OUTPUT_NAMES
        )
    )

    if save_csv:
        csv_path = (
            BLOCK_9_OUTPUT_DIR
            / f"{table_name}.csv"
        )

        try:
            frame.to_csv(
                csv_path,
                index=False,
            )

            try:
                persisted_frame = pd.read_csv(
                    csv_path,
                    low_memory=False,
                )
            except pd.errors.EmptyDataError:
                persisted_frame = pd.DataFrame(
                    columns=frame.columns
                )

            persisted_rows = len(
                persisted_frame
            )

            persisted_columns = len(
                persisted_frame.columns
            )

            if (
                persisted_rows == expected_rows
                and persisted_columns == expected_columns
            ):
                status = "PASSED"
            elif persisted_rows != expected_rows:
                status = "ROW_COUNT_MISMATCH"
            else:
                status = "COLUMN_COUNT_MISMATCH"

            file_size_bytes = (
                csv_path.stat().st_size
            )

            file_hash = sha256_file(
                csv_path
            )

        except Exception as exc:
            persisted_rows = None
            persisted_columns = None
            file_size_bytes = None
            file_hash = None
            status = (
                "FAILED: "
                + type(exc).__name__
                + ": "
                + str(exc)[:180]
            )

        persistence_rows.append(
            {
                "table_name": table_name,
                "file_type": "csv",
                "expected_rows": expected_rows,
                "persisted_rows": persisted_rows,
                "expected_columns": expected_columns,
                "persisted_columns": persisted_columns,
                "file_size_bytes": file_size_bytes,
                "sha256": file_hash,
                "status": status,
                "saved_file": str(csv_path),
            }
        )


block_9_persistence_report_df = (
    pd.DataFrame(persistence_rows)
)

manifest_csv_path = (
    BLOCK_9_OUTPUT_DIR
    / "block_9_output_manifest.csv"
)

block_9_persistence_report_df.to_csv(
    manifest_csv_path,
    index=False,
)

manifest_json_path = (
    BLOCK_9_OUTPUT_DIR
    / "block_9_manifest.json"
)

manifest_payload = {
    "block_number": 9,
    "block_name": (
        "AI Enrichment and Quality Control"
    ),
    "generated_at_utc": (
        utc_now().isoformat()
    ),
    "deterministic_dry_run": (
        DETERMINISTIC_DRY_RUN
    ),
    "ai_provider": (
        "Azure OpenAI"
        if AI_ENABLED
        else None
    ),
    "final_production_mode":
        FINAL_PRODUCTION_MODE,
    "processing_generation":
        AI_PROCESSING_GENERATION,
    "block_10_primary_input":
        "ai_exception_deduplicated_outcomes_df",
    "max_ai_records_per_task": (
        MAX_AI_RECORDS_PER_TASK
    ),
    "processing_generation": (
        AI_PROCESSING_GENERATION
    ),
    "skip_completed_signatures": (
        SKIP_COMPLETED_SIGNATURES
    ),
    "outputs": (
        block_9_persistence_report_df
        .where(
            block_9_persistence_report_df.notna(),
            None,
        )
        .to_dict(orient="records")
    ),
}

manifest_json_path.write_text(
    json.dumps(
        manifest_payload,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

block_9_manifest_df = (
    block_9_persistence_report_df.copy()
)

print(
    "Persisted outputs:",
    len(block_9_persistence_report_df),
)

display(
    block_9_persistence_report_df[
        [
            "table_name",
            "file_type",
            "expected_rows",
            "persisted_rows",
            "expected_columns",
            "persisted_columns",
            "file_size_bytes",
            "status",
            "saved_file",
        ]
    ]
)

## Final production execution sequence

1. Run from Step 1 in a new runtime.
2. Step 16 reconstructs historical cache responses locally without Azure charges.
3. Step 17 re-evaluates the full reconstructed proposal population.
4. Step 18 distinguishes direct acceptance from propagation and applies accepted decisions only to effective fields while retaining original fields.
5. Step 19 reports direct and propagated row counts separately and classifies pre-AI exclusions explicitly.
6. Step 20 normalises mixed object columns before Parquet persistence and validates CSV files with `low_memory=False`.
7. Block 10 should use `effective_standard_concept` and `effective_reported_value` where `signature_decision_applied` is true.
